Cell 1 — Cài đặt thư viện

**Research-complete version:** preserves the AR-DRFM core and adds row-level preprocessing audit, dataset EDA, chronological train/validation/test snapshots, validation-only K_MAX sensitivity, cluster-imbalance analysis, classical clustering benchmark (K-Means/GMM/Agglomerative), frozen-centroid temporal holdout, runtime/memory reporting and supplementary reproducibility exports.

Important: the new cells are code to be executed on the real UCI Online Retail II dataset. No empirical result is pre-filled. Synthetic Page-Hinkley streams remain detector stress-tests only and must not be reported as real-dataset results.


**Research Quality Upgrade 6.0:** the adaptive window objective now separates behavioural clustering quality from activity coverage and temporal stability. Policy weights and hysteresis are selected using validation snapshots only with a one-standard-error rule, then the complete chronological policy is rerun with the locked configuration. Cluster profiles and within-snapshot Kruskal-Wallis effect diagnostics are exported separately. Exact-duplicate impact is audited explicitly. All previous cached outputs are cleared because the selection objective changed.


**Execution guidance:** set `AR_DRFM_OUTPUT_DIR` before running Cell 2 to choose a persistent output folder. Execute all cells in order from a clean runtime. Test snapshots are not used to select K_MAX, adaptive-policy weights or hysteresis. The final temporal test is an out-of-sample clustering-policy evaluation; it is not a supervised classification test because Online Retail II has no ground-truth customer-segment labels.


In [ ]:
!pip -q install pandas numpy scikit-learn scipy matplotlib seaborn openpyxl psutil

Cell 2 — Import thư viện và cấu hình

In [ ]:
import os
import time
import math
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

from scipy.optimize import linear_sum_assignment
from threadpoolctl import threadpool_limits, threadpool_info

warnings.filterwarnings("ignore")

# Reproducibility
RUN_STARTED_AT_UTC = datetime.now(timezone.utc).isoformat()
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Limit BLAS/OpenMP worker threads to reduce run-to-run numerical variation.
THREADPOOL_CONTROLLER = threadpool_limits(limits=1)
DETERMINISTIC_THREAD_LIMIT = 1

# ==========================================
# DATA CONFIGURATION
# ==========================================

# Nếu chạy Colab, sẽ có tùy chọn upload file.
DATA_PATH = "/content/online_retail_II.xlsx"

# ==========================================
# RFM CONFIGURATION
# ==========================================

# Các cửa sổ Rolling RFM ứng viên
WINDOWS = [28, 56, 91]

# Khoảng cách giữa các snapshot
SNAPSHOT_STRIDE = 28

# ==========================================
# CLUSTERING CONFIGURATION
# ==========================================

K_MIN = 2
K_MAX = 8
KMEANS_N_INIT = 10  # fixed across adaptive, baseline and frozen-centroid K-Means

MIN_CLUSTER_FRACTION = 0.01

MAX_SILHOUETTE_SAMPLE = 5000

# ==========================================
# ADAPTIVE WINDOW CONFIGURATION
# ==========================================

# Three-objective adaptive policy. Quality measures behavioural separation
# on the same common-active cohort; stability measures temporal consistency;
# coverage measures the share of the 91-day reference population that remains
# eligible for behavioural clustering in the candidate window.
QUALITY_WEIGHT = 0.65
STABILITY_WEIGHT = 0.20
COVERAGE_WEIGHT = 0.15
if not np.isclose(QUALITY_WEIGHT + STABILITY_WEIGHT + COVERAGE_WEIGHT, 1.0):
    raise ValueError("Quality, stability and coverage weights must sum to 1.")

# Chỉ đổi window nếu điểm mới tốt hơn đủ ngưỡng
# Switch only when the challenger is meaningfully better and remains so
# for multiple consecutive snapshots. Sensitivity is reported in Cell 10.
HYSTERESIS_MARGIN = 0.01
HYSTERESIS_CONFIRMATIONS = 2

# ARI is evaluated only on shared customers; stability credit is discounted
# by the actual retained fraction of the smaller reference cohort.
# A fixed reference population is used for fair window comparisons:
# customers with >=1 valid purchase in the longest candidate window.
REFERENCE_COHORT_WINDOW = max(WINDOWS)

# Primary behavioural clustering cohort: customers active in the shortest
# candidate window. This set is contained in every longer candidate window.
# Inactive customers are reported as dormant states, not forced into K-Means
# using synthetic (Frequency=0, Monetary=0) feature vectors.
COMMON_ACTIVE_WINDOW_DAYS = min(WINDOWS)
ACTIVITY_STATE_WINDOW_DAYS = min(WINDOWS)  # fixed horizon for lifecycle transitions
CLUSTER_INACTIVE_CUSTOMERS = False

# Fixed, bounded utility weights for comparing window representations.
WINDOW_SILHOUETTE_WEIGHT = 0.65
WINDOW_DBI_WEIGHT = 0.35
if not np.isclose(WINDOW_SILHOUETTE_WEIGHT + WINDOW_DBI_WEIGHT, 1.0):
    raise ValueError("Window quality weights must sum to 1.")

# Temporal ARI is discounted by the actual fraction of the smaller
# reference cohort retained across consecutive snapshots.
STABILITY_FULL_OVERLAP_FRACTION = 1.0
STABILITY_MIN_OVERLAP_FRACTION = 0.20

# ==========================================
# OUTPUT
# ==========================================

OUTPUT_DIR = Path(os.environ.get("AR_DRFM_OUTPUT_DIR", "/content/adaptive_rolling_dynamic_rfm_results"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration initialized.")


Cell 3 — Upload và đọc Online Retail II

In [ ]:
import zipfile
from pathlib import Path
import pandas as pd

def upload_dataset_if_needed():
    """Use DATA_PATH when it exists; otherwise offer a Colab upload."""
    configured_path = Path(DATA_PATH)
    if configured_path.exists():
        print(f"Using dataset at DATA_PATH: {configured_path}")
        return configured_path

    try:
        from google.colab import files
        print("DATA_PATH does not exist. Please upload Online Retail II ZIP/XLSX/CSV.")
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file uploaded.")
        filename = next(iter(uploaded.keys()))
        return Path("/content") / filename
    except ImportError as exc:
        raise FileNotFoundError(
            f"Dataset not found at {configured_path}. Set DATA_PATH to a valid "
            "XLSX/CSV/ZIP file or run this notebook in Colab and upload it."
        ) from exc


def extract_zip_file(zip_path):

    extract_dir = Path("/content/online_retail_ii_extracted")

    extract_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    print("Extracting ZIP file...")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:

        zip_ref.extractall(extract_dir)

    print("Extraction completed.")

    return extract_dir


def load_retail_dataset(path):

    path = Path(path)

    # ======================================
    # CASE 1: ZIP FILE
    # ======================================

    if path.suffix.lower() == ".zip":

        extract_dir = extract_zip_file(path)

        files_found = []

        for extension in ["*.xlsx", "*.xls", "*.csv"]:

            files_found.extend(
                extract_dir.rglob(extension)
            )

        if not files_found:

            raise FileNotFoundError(
                "No XLSX, XLS or CSV file found inside ZIP."
            )

        print("\nFiles found:")

        for file in files_found:

            print("-", file.name)

        excel_files = [
            f for f in files_found
            if f.suffix.lower() in [".xlsx", ".xls"]
        ]

        csv_files = [
            f for f in files_found
            if f.suffix.lower() == ".csv"
        ]

        if excel_files:

            path = excel_files[0]

        elif csv_files:

            path = csv_files[0]

        print("\nSelected file:", path.name)

    # ======================================
    # CASE 2: EXCEL
    # ======================================

    if path.suffix.lower() in [".xlsx", ".xls"]:

        excel = pd.ExcelFile(path)

        frames = []

        for sheet in excel.sheet_names:

            print("Reading sheet:", sheet)

            temp = pd.read_excel(
                path,
                sheet_name=sheet
            )

            frames.append(temp)

        data = pd.concat(
            frames,
            ignore_index=True
        )

    # ======================================
    # CASE 3: CSV
    # ======================================

    elif path.suffix.lower() == ".csv":

        data = pd.read_csv(
            path,
            encoding="latin1",
            low_memory=False
        )

    else:

        raise ValueError(
            f"Unsupported file format: {path.suffix}"
        )

    return data


# ======================================
# LOAD DATASET
# ======================================

dataset_path = upload_dataset_if_needed()

raw_df = load_retail_dataset(dataset_path)


print("\n" + "=" * 50)
print("ONLINE RETAIL II DATASET")
print("=" * 50)

print("Rows:", f"{len(raw_df):,}")

print("Columns:", len(raw_df.columns))

print("Column names:", raw_df.columns.tolist())

display(raw_df.head())

print("\nDataset information:")

raw_df.info()

Cell 4 — Data preprocessing

In [ ]:
def normalize_columns(data):
    df = data.copy()
    column_mapping = {}
    for col in df.columns:
        normalized = str(col).strip().lower()
        if normalized in ["invoice", "invoiceno"]:
            column_mapping[col] = "Invoice"
        elif normalized == "quantity":
            column_mapping[col] = "Quantity"
        elif normalized in ["invoicedate", "invoice date"]:
            column_mapping[col] = "InvoiceDate"
        elif normalized in ["price", "unitprice", "unit price"]:
            column_mapping[col] = "Price"
        elif normalized in ["customer id", "customerid"]:
            column_mapping[col] = "CustomerID"
        elif normalized == "stockcode":
            column_mapping[col] = "StockCode"
        elif normalized == "description":
            column_mapping[col] = "Description"
        elif normalized == "country":
            column_mapping[col] = "Country"
    return df.rename(columns=column_mapping)


def preprocess_retail_data(raw):
    """Clean purchase transactions and create transparent row-level audit tables.

    Sequential audit counts are mutually exclusive: each rule is applied only
    to rows remaining after the previous rule. Raw missingness and type-coercion
    failures are also reported separately, so total removed rows are not
    incorrectly described as the missing-value rate.
    """
    global raw_missingness_df, coercion_audit_df, preprocessing_audit_df, duplicate_sensitivity_df

    df = normalize_columns(raw)
    required_columns = ["Invoice", "Quantity", "InvoiceDate", "Price", "CustomerID"]
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    rows_before = int(len(df))

    # Missingness is measured before conversion or filtering.
    raw_missing_rows = []
    for col in required_columns:
        missing_mask = df[col].isna()
        if col == "Invoice":
            missing_mask = missing_mask | df[col].astype("string").str.strip().eq("").fillna(False)
        raw_missing_rows.append({
            "column": col,
            "missing_count": int(missing_mask.sum()),
            "missing_rate": float(missing_mask.mean()) if rows_before else 0.0
        })
    raw_missingness_df = pd.DataFrame(raw_missing_rows).sort_values("missing_rate", ascending=False)

    # Keep string missing values as <NA>; never convert NaN to the literal "nan".
    df["Invoice"] = df["Invoice"].astype("string").str.strip()
    df.loc[df["Invoice"].eq("").fillna(False), "Invoice"] = pd.NA

    coercion_rows = []
    conversion_specs = {
        "InvoiceDate": lambda x: pd.to_datetime(x, errors="coerce"),
        "Quantity": lambda x: pd.to_numeric(x, errors="coerce"),
        "Price": lambda x: pd.to_numeric(x, errors="coerce"),
        "CustomerID": lambda x: pd.to_numeric(x, errors="coerce")
    }
    for col, converter in conversion_specs.items():
        before_nonmissing = int(df[col].notna().sum())
        converted = converter(df[col])
        invalid_count = int((df[col].notna() & converted.isna()).sum())
        coercion_rows.append({
            "column": col,
            "nonmissing_before_conversion": before_nonmissing,
            "unparseable_count": invalid_count,
            "unparseable_rate_among_nonmissing": (
                invalid_count / before_nonmissing if before_nonmissing else 0.0
            )
        })
        df[col] = converted
    coercion_audit_df = pd.DataFrame(coercion_rows)

    audit_rows = []
    def apply_filter(frame, keep_mask, stage, rationale):
        before = int(len(frame))
        filtered = frame.loc[keep_mask].copy()
        after = int(len(filtered))
        removed = before - after
        audit_rows.append({
            "stage": stage,
            "rationale": rationale,
            "rows_before": before,
            "rows_removed": removed,
            "rows_after": after,
            "removed_pct_of_stage_input": (removed / before * 100.0) if before else 0.0,
            "cumulative_removed_pct_of_raw": ((rows_before - after) / rows_before * 100.0) if rows_before else 0.0
        })
        return filtered

    required_valid = df[required_columns].notna().all(axis=1)
    df = apply_filter(
        df,
        required_valid,
        "remove_missing_required_fields",
        "Invoice, InvoiceDate, Quantity, Price and CustomerID are needed for customer-level RFM."
    )

    cancelled_mask = df["Invoice"].str.upper().str.startswith("C", na=False)
    df = apply_filter(
        df,
        ~cancelled_mask,
        "remove_cancelled_invoices",
        "Invoice identifiers beginning with C represent cancelled invoices in Online Retail II."
    )

    valid_quantity = df["Quantity"] > 0
    df = apply_filter(
        df,
        valid_quantity,
        "remove_nonpositive_quantity",
        "The main purchase-only experiment excludes returns and zero-quantity records; returns should be studied separately."
    )

    valid_price = df["Price"] > 0
    df = apply_filter(
        df,
        valid_price,
        "remove_nonpositive_price",
        "Nonpositive unit prices cannot form valid positive purchase monetary value in the main experiment."
    )

    integral_customer_id = np.isclose(df["CustomerID"] % 1, 0)
    df = apply_filter(
        df,
        integral_customer_id,
        "remove_noninteger_customer_id",
        "CustomerID is an identifier and must be integer-valued."
    )
    df["CustomerID"] = df["CustomerID"].astype("int64")

    # Quantify the impact of exact-row deduplication before applying it.
    # Identical rows may be duplicated exports or repeated same-value invoice
    # lines; the audit exposes the impact instead of hiding the assumption.
    before_dedup = int(len(df))
    duplicate_mask = df.duplicated(keep="first")
    duplicate_rows = int(duplicate_mask.sum())
    duplicate_sensitivity_df = pd.DataFrame([
        {
            "scenario": "retain_exact_duplicates",
            "rows": int(len(df)),
            "unique_customers": int(df["CustomerID"].nunique()),
            "unique_invoices": int(df["Invoice"].nunique()),
            "gross_amount": float((df["Quantity"] * df["Price"]).sum()),
            "exact_duplicate_extra_rows": duplicate_rows
        },
        {
            "scenario": "remove_exact_duplicates_primary",
            "rows": int((~duplicate_mask).sum()),
            "unique_customers": int(df.loc[~duplicate_mask, "CustomerID"].nunique()),
            "unique_invoices": int(df.loc[~duplicate_mask, "Invoice"].nunique()),
            "gross_amount": float((df.loc[~duplicate_mask, "Quantity"] * df.loc[~duplicate_mask, "Price"]).sum()),
            "exact_duplicate_extra_rows": 0
        }
    ])
    df = df.loc[~duplicate_mask].copy()
    after_dedup = int(len(df))
    removed = before_dedup - after_dedup
    audit_rows.append({
        "stage": "remove_exact_duplicates",
        "rationale": "Remove exact duplicate rows only; no fuzzy or customer-level deduplication is performed.",
        "rows_before": before_dedup,
        "rows_removed": removed,
        "rows_after": after_dedup,
        "removed_pct_of_stage_input": (removed / before_dedup * 100.0) if before_dedup else 0.0,
        "cumulative_removed_pct_of_raw": ((rows_before - after_dedup) / rows_before * 100.0) if rows_before else 0.0
    })

    df["Amount"] = df["Quantity"] * df["Price"]
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["Amount"])
    df = df.sort_values("InvoiceDate").reset_index(drop=True)

    preprocessing_audit_df = pd.DataFrame(audit_rows)
    print("Rows before preprocessing:", f"{rows_before:,}")
    print("Rows after preprocessing:", f"{len(df):,}")
    print("Total rows removed:", f"{rows_before - len(df):,}")
    print("Overall removal rate (not missing rate):", f"{(rows_before-len(df))/rows_before:.2%}" if rows_before else "N/A")
    print("Unique customers:", f"{df['CustomerID'].nunique():,}")
    print("Unique invoices:", f"{df['Invoice'].nunique():,}")
    print("Date range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())
    print("Total revenue:", f"{df['Amount'].sum():,.2f}")
    print("\nRaw missingness by required field:")
    display(raw_missingness_df.round(5))
    print("\nExact-duplicate sensitivity (row and gross-value impact; no model result inferred):")
    display(duplicate_sensitivity_df.round(3))
    print("\nSequential preprocessing audit:")
    display(preprocessing_audit_df.round(3))
    print("\nType coercion audit:")
    display(coercion_audit_df.round(5))
    return df


df = preprocess_retail_data(raw_df)
display(df.head())

Cell 4.1 — Exploratory Data Analysis (EDA), phân phối và chất lượng dữ liệu

Các thống kê/biểu đồ dưới đây được tính trực tiếp từ Online Retail II sau bước làm sạch. Phần này phân biệt missing values, độ lệch hành vi khách hàng và cluster imbalance; không gọi dữ liệu giao dịch là class-imbalanced vì dataset không có nhãn lớp khách hàng gốc.

In [ ]:
# ============================================================
# DATASET EDA: missingness, transaction distributions, customer
# behavior, time trend, country and revenue concentration
# ============================================================
EDA_DIR = Path(OUTPUT_DIR) / "eda"
EDA_DIR.mkdir(parents=True, exist_ok=True)

# 1) Missingness and row-removal audit
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
if not raw_missingness_df.empty:
    plot_missing = raw_missingness_df.sort_values("missing_rate", ascending=True)
    axes[0].barh(plot_missing["column"], 100 * plot_missing["missing_rate"])
    axes[0].set_xlabel("Missing rate (%)")
    axes[0].set_title("Raw missingness by required field")
if not preprocessing_audit_df.empty:
    axes[1].barh(preprocessing_audit_df["stage"], preprocessing_audit_df["rows_removed"])
    axes[1].set_xlabel("Rows removed at sequential stage")
    axes[1].set_title("Sequential preprocessing audit")
fig.tight_layout()
fig.savefig(EDA_DIR / "eda_01_missingness_and_cleaning.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close(fig)

# 2) Transaction-level distributions (log-scale view is supplementary, raw values retained)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0,0].hist(df["Quantity"].clip(upper=df["Quantity"].quantile(0.99)), bins=60)
axes[0,0].set_title("Quantity distribution (winsorized view for display only)")
axes[0,1].hist(df["Price"].clip(upper=df["Price"].quantile(0.99)), bins=60)
axes[0,1].set_title("Unit price distribution (display capped at P99)")
axes[1,0].hist(np.log1p(df["Amount"].clip(lower=0)), bins=60)
axes[1,0].set_title("Log1p transaction amount distribution")
axes[1,1].hist(np.log1p(df.groupby("CustomerID")["Invoice"].nunique()), bins=50)
axes[1,1].set_title("Log1p customer invoice frequency")
fig.tight_layout()
fig.savefig(EDA_DIR / "eda_02_transaction_and_customer_distributions.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close(fig)

# 3) Customer-level RFM-like descriptive profile at dataset end
eda_snapshot_end = pd.Timestamp(df["InvoiceDate"].max()).normalize() + pd.Timedelta(days=1)
eda_customer = df.groupby("CustomerID").agg(
    LastPurchase=("InvoiceDate", "max"),
    Frequency=("Invoice", "nunique"),
    Monetary=("Amount", "sum"),
    TransactionRows=("Invoice", "size")
)
eda_customer["Recency"] = (eda_snapshot_end - eda_customer["LastPurchase"]).dt.total_seconds() / 86400
eda_customer = eda_customer[["Recency", "Frequency", "Monetary", "TransactionRows"]]

# 4) Monthly behavior
monthly_eda = df.set_index("InvoiceDate").resample("MS").agg(
    transaction_rows=("Invoice", "size"),
    invoices=("Invoice", "nunique"),
    revenue=("Amount", "sum"),
    active_customers=("CustomerID", "nunique")
).reset_index()
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(monthly_eda["InvoiceDate"], monthly_eda["revenue"], marker="o", label="Revenue")
ax1.set_ylabel("Revenue")
ax1.set_xlabel("Month")
ax2 = ax1.twinx()
ax2.plot(monthly_eda["InvoiceDate"], monthly_eda["active_customers"], linestyle="--", label="Active customers")
ax2.set_ylabel("Active customers")
ax1.set_title("Monthly revenue and active customers")
fig.tight_layout()
fig.savefig(EDA_DIR / "eda_03_monthly_revenue_customers.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close(fig)

# 5) Pareto concentration (descriptive, not an assumption of imbalance)
customer_revenue = eda_customer["Monetary"].sort_values(ascending=False)
top20_n = max(1, int(np.ceil(0.20 * len(customer_revenue))))
top20_revenue_share = float(customer_revenue.head(top20_n).sum() / customer_revenue.sum()) if customer_revenue.sum() else np.nan
cumulative_share = customer_revenue.cumsum() / customer_revenue.sum()
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.arange(1, len(cumulative_share)+1) / len(cumulative_share) * 100, cumulative_share.values * 100)
ax.axvline(20, color="grey", linestyle="--", linewidth=1)
ax.axhline(top20_revenue_share * 100, color="grey", linestyle=":", linewidth=1)
ax.set_xlabel("Cumulative share of customers ranked by revenue (%)")
ax.set_ylabel("Cumulative revenue share (%)")
ax.set_title("Customer revenue concentration (Pareto curve)")
fig.tight_layout()
fig.savefig(EDA_DIR / "eda_04_customer_revenue_pareto.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close(fig)

# 6) Country distribution
if "Country" in df.columns:
    country_summary = df.groupby("Country", dropna=False).agg(
        transactions=("Invoice", "nunique"),
        customers=("CustomerID", "nunique"),
        revenue=("Amount", "sum")
    ).sort_values("revenue", ascending=False).head(15).reset_index()
    fig, ax = plt.subplots(figsize=(12, 6))
    country_plot = country_summary.sort_values("revenue", ascending=True)
    ax.barh(country_plot["Country"], country_plot["revenue"])
    ax.set_title("Top 15 countries by revenue")
    fig.tight_layout()
    fig.savefig(EDA_DIR / "eda_05_top_countries_revenue.png", dpi=220, bbox_inches="tight")
    plt.show(); plt.close(fig)
else:
    country_summary = pd.DataFrame()

# 7) RFM descriptive correlation and skewness
rfm_eda_summary = eda_customer[["Recency", "Frequency", "Monetary"]].describe(percentiles=[.01,.25,.5,.75,.95,.99]).T
rfm_eda_summary["skewness"] = eda_customer[["Recency", "Frequency", "Monetary"]].skew()
rfm_eda_summary["zero_fraction"] = (eda_customer[["Recency", "Frequency", "Monetary"]] == 0).mean()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(eda_customer[["Recency", "Frequency", "Monetary"]].corr(), annot=True, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Customer-level RFM correlation")
fig.tight_layout()
fig.savefig(EDA_DIR / "eda_06_rfm_correlation.png", dpi=220, bbox_inches="tight")
plt.show(); plt.close(fig)

eda_distribution_summary = pd.DataFrame({
    "metric": ["raw_rows", "clean_rows", "unique_customers", "unique_invoices", "top20_customer_revenue_share", "customer_frequency_skewness", "customer_monetary_skewness"],
    "value": [len(raw_df), len(df), df["CustomerID"].nunique(), df["Invoice"].nunique(), top20_revenue_share,
              eda_customer["Frequency"].skew(), eda_customer["Monetary"].skew()]
})
print("EDA descriptive summary:")
display(eda_distribution_summary)
print("Customer-level RFM distribution summary:")
display(rfm_eda_summary.round(4))
print("Monthly activity preview:")
display(monthly_eda.head())
print("Important interpretation: no original customer-class labels exist; class imbalance is not defined before clustering. The outputs describe behavioral skew and, later, cluster-size imbalance.")

Cell 5 — Xây dựng snapshot dates

In [ ]:
def generate_snapshot_dates(
    data,
    windows,
    stride_days=28
):

    max_window = max(windows)

    first_date = (
        data["InvoiceDate"].min()
        + pd.Timedelta(days=max_window)
    )

    last_date = data["InvoiceDate"].max()

    dates = pd.date_range(
        start=first_date.normalize(),
        end=last_date.normalize(),
        freq=f"{stride_days}D"
    )

    # Keep the configured snapshot cadence exactly. Do not append
    # last_date when it would create a shorter-than-stride final interval.
    dates = dates.unique()

    return list(dates)


snapshot_dates = generate_snapshot_dates(
    df,
    WINDOWS,
    SNAPSHOT_STRIDE
)

print("Number of snapshots:", len(snapshot_dates))

print("First snapshot:", snapshot_dates[0])

print("Last snapshot:", snapshot_dates[-1])

display(
    pd.DataFrame({
        "SnapshotDate": snapshot_dates
    })
)

# Chronological model-development / validation / final-test partitions.
# For 24 snapshots this yields 12 / 6 / 6. This is temporal evaluation,
# not a supervised class split (the dataset has no customer-segment ground truth).
def make_chronological_snapshot_split(dates, train_fraction=1/2, validation_fraction=1/4):
    dates = list(pd.to_datetime(dates))
    n = len(dates)
    if n < 9:
        raise ValueError(f"At least 9 snapshots are required for 3 chronological partitions; got {n}.")
    n_train = int(np.floor(n * train_fraction))
    n_validation = int(np.floor(n * validation_fraction))
    n_test = n - n_train - n_validation
    if min(n_train, n_validation, n_test) < 1:
        raise ValueError("Chronological split produced an empty partition.")
    return dates[:n_train], dates[n_train:n_train+n_validation], dates[n_train+n_validation:]

train_snapshot_dates, validation_snapshot_dates, test_snapshot_dates = make_chronological_snapshot_split(snapshot_dates)
TEMPORAL_SPLIT_POLICY = "chronological snapshots: first 1/2 train, next 1/4 validation, final 1/4 test"
TEMPORAL_SPLIT_TABLE = pd.DataFrame([
    {"partition": "train", "n_snapshots": len(train_snapshot_dates), "start": train_snapshot_dates[0], "end": train_snapshot_dates[-1]},
    {"partition": "validation", "n_snapshots": len(validation_snapshot_dates), "start": validation_snapshot_dates[0], "end": validation_snapshot_dates[-1]},
    {"partition": "test", "n_snapshots": len(test_snapshot_dates), "start": test_snapshot_dates[0], "end": test_snapshot_dates[-1]}
])
print("Chronological temporal split (no random row-level split):")
display(TEMPORAL_SPLIT_TABLE)


Cell 6 — Tính Lifetime RFM và Rolling RFM

In [ ]:
def calculate_rfm(
    data,
    snapshot_date,
    window_days=None
):

    snapshot_end = (
        pd.Timestamp(snapshot_date).normalize()
        + pd.Timedelta(days=1)
    )

    if window_days is None:

        start_date = data["InvoiceDate"].min()

    else:

        start_date = (
            snapshot_end
            - pd.Timedelta(days=window_days)
        )

    subset = data[
        (data["InvoiceDate"] >= start_date) &
        (data["InvoiceDate"] < snapshot_end)
    ]

    if subset.empty:

        return pd.DataFrame(
            columns=[
                "Recency",
                "Frequency",
                "Monetary"
            ]
        )

    rfm = subset.groupby(
        "CustomerID"
    ).agg(

        LastPurchase=("InvoiceDate", "max"),

        Frequency=("Invoice", "nunique"),

        Monetary=("Amount", "sum")

    )

    rfm["Recency"] = (
        snapshot_end - rfm["LastPurchase"]
    ).dt.total_seconds() / 86400

    rfm = rfm[
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]

    rfm = rfm.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    return rfm


def align_rfm_to_reference_cohort(
    rfm,
    reference_customer_ids,
    window_days,
    reference_rfm
):
    """Legacy helper for explicitly two-part lifecycle analyses.

    IMPORTANT: this helper inserts F=M=0 for inactive-in-window customers.
    The primary AR-DRFM window-quality comparison no longer uses this helper,
    because a large zero-inflated dormant group can dominate Silhouette. The
    primary comparison fits/evaluates behaviour clusters on a common active
    cohort and reports dormant customers separately.
    """
    required = ["Recency", "Frequency", "Monetary"]
    if not isinstance(rfm, pd.DataFrame) or not isinstance(reference_rfm, pd.DataFrame):
        raise TypeError("rfm and reference_rfm must be pandas DataFrames.")
    if not set(required).issubset(rfm.columns) or not set(required).issubset(reference_rfm.columns):
        raise ValueError(f"Both RFM tables must contain columns {required}.")
    reference_ids = pd.Index(reference_customer_ids)
    if not reference_ids.is_unique:
        raise ValueError("Reference customer IDs must be unique.")
    if not reference_ids.isin(reference_rfm.index).all():
        raise ValueError("Reference RFM does not cover all reference customer IDs.")

    aligned = rfm.reindex(reference_ids)[required].copy()
    reference_recency = reference_rfm.reindex(reference_ids)["Recency"]
    inactive = aligned["Frequency"].isna()

    if window_days is None:
        if inactive.any():
            raise ValueError("Lifetime RFM unexpectedly misses reference-cohort customers.")
    else:
        window_days = int(window_days)
        if window_days <= 0:
            raise ValueError("window_days must be positive.")
        aligned.loc[inactive, ["Frequency", "Monetary"]] = 0.0

    # Recency is kept consistent across window candidates using the observed
    # last purchase within the reference horizon.
    aligned["Recency"] = reference_recency.astype(float)
    aligned = aligned.apply(pd.to_numeric, errors="coerce")
    if aligned.isna().any().any():
        raise ValueError("Aligned RFM still contains missing values.")
    if not np.isfinite(aligned.to_numpy(dtype=float)).all():
        raise ValueError("Aligned RFM contains non-finite values.")
    if (aligned[["Frequency", "Monetary"]] < 0).any().any():
        raise ValueError("Frequency and Monetary must be non-negative.")
    aligned.index.name = rfm.index.name or "CustomerID"
    aligned.attrs["reference_cohort_size"] = int(len(reference_ids))
    aligned.attrs["inactive_in_window"] = int(inactive.sum())
    aligned.attrs["window_days"] = None if window_days is None else int(window_days)
    aligned.attrs["recency_reference_window_days"] = int(REFERENCE_COHORT_WINDOW)
    return aligned


# Test Rolling RFM
test_snapshot = snapshot_dates[0]

rfm_28 = calculate_rfm(
    df,
    test_snapshot,
    window_days=28
)

rfm_56 = calculate_rfm(
    df,
    test_snapshot,
    window_days=56
)

rfm_91 = calculate_rfm(
    df,
    test_snapshot,
    window_days=91
)

# Lifetime RFM
rfm_lifetime = calculate_rfm(
    df,
    test_snapshot,
    window_days=None
)

print("28-day customers:", len(rfm_28))
print("56-day customers:", len(rfm_56))
print("91-day customers:", len(rfm_91))
print("Lifetime customers:", len(rfm_lifetime))

display(rfm_28.head())


Cell 7 — Chuẩn hóa dữ liệu RFM

In [ ]:
def transform_rfm(rfm):

    features = rfm[
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ].copy()

    # Apply the same monotonic log1p transform to all non-negative R/F/M
    # features. Inactive-in-window customers retain observed Recency from the
    # fixed reference horizon and receive Frequency=Monetary=0.
    for col in ["Recency", "Frequency", "Monetary"]:
        features[col] = np.log1p(
            pd.to_numeric(features[col], errors="coerce").clip(lower=0)
        )

    if features.isna().any().any() or not np.isfinite(features.to_numpy()).all():
        raise ValueError("RFM transformation produced missing/non-finite values.")

    # Clip extreme values using this snapshot only
    for col in features.columns:

        lower = features[col].quantile(0.01)

        upper = features[col].quantile(0.99)

        features[col] = features[col].clip(
            lower,
            upper
        )

    scaler = StandardScaler()

    X = scaler.fit_transform(features)

    return X, scaler, features


X_test, scaler_test, transformed_test = transform_rfm(
    rfm_56
)

print("Original RFM shape:", rfm_56.shape)

print("Transformed shape:", X_test.shape)

display(
    pd.DataFrame(
        X_test,
        columns=[
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ).head()
)

Cell 7.1 — Validation-only sensitivity cho K_MAX

Thử nghiệm K_MAX ∈ {8, 13, 25} chỉ sử dụng các snapshot Validation. Chọn giới hạn tìm kiếm bằng quy tắc one-standard-error (chọn không gian nhỏ nhất có điểm validation nằm trong một SE của cấu hình tốt nhất). Test snapshots không được dùng để chọn K_MAX. Đây là phân tích độ nhạy, không mặc định rằng K lớn hơn là tốt hơn.

In [ ]:
# ============================================================
# VALIDATION-ONLY K_MAX SENSITIVITY ON A COMMON ACTIVE POPULATION
# ============================================================
# For each validation snapshot, form the intersection of customers active in
# every candidate window. Since windows are nested, this is normally the
# shortest-window active population. This avoids selecting K_MAX primarily
# from an artificial active/inactive split created by zero-filled RFM rows.
KMAX_SENSITIVITY_CANDIDATES = [8, 13, 25]
KMAX_SENSITIVITY_N_INIT = 10
KMAX_SENSITIVITY_SIL_SAMPLE = min(3000, MAX_SILHOUETTE_SAMPLE)
KMAX_DEFAULT_BEFORE_VALIDATION = int(K_MAX)

kmax_candidate_rows = []
for snap_idx, snap in enumerate(validation_snapshot_dates):
    native_by_window = {
        int(w): calculate_rfm(df, snap, window_days=int(w))
        for w in WINDOWS
    }
    active_sets = [set(frame.index.tolist()) for frame in native_by_window.values() if not frame.empty]
    if len(active_sets) != len(WINDOWS) or not active_sets:
        print(f"Skipping K_MAX sensitivity at {snap.date()}: one or more windows have no active customers.")
        continue
    common_ids = sorted(set.intersection(*active_sets))
    if len(common_ids) < max(KMAX_SENSITIVITY_CANDIDATES) + 2:
        print(f"Skipping K_MAX sensitivity at {snap.date()}: common active cohort too small ({len(common_ids)}).")
        continue

    for window_days, native_rfm in native_by_window.items():
        rfm_common = native_rfm.reindex(common_ids).copy()
        if rfm_common.isna().any().any() or (rfm_common[["Frequency", "Monetary"]] <= 0).any().any():
            raise RuntimeError("K_MAX sensitivity cohort is not consistently active across all windows.")
        X_val, _, _ = transform_rfm(rfm_common)
        n_val = len(X_val)
        max_k = min(max(KMAX_SENSITIVITY_CANDIDATES), n_val - 1)
        for k in range(K_MIN, max_k + 1):
            start_k = time.perf_counter()
            km = KMeans(
                n_clusters=k,
                random_state=RANDOM_STATE,
                n_init=KMAX_SENSITIVITY_N_INIT,
                algorithm="lloyd"
            )
            labels = km.fit_predict(X_val)
            unique_labels = np.unique(labels)
            if len(unique_labels) < 2 or len(unique_labels) >= n_val:
                continue
            counts = np.bincount(labels, minlength=k)
            min_fraction = float(counts.min() / n_val)
            min_required = max(2, int(np.ceil(MIN_CLUSTER_FRACTION * n_val)))
            valid_size = bool(counts.min() >= min_required)
            sil = float(silhouette_score(
                X_val, labels,
                sample_size=min(KMAX_SENSITIVITY_SIL_SAMPLE, n_val),
                random_state=RANDOM_STATE
            ))
            dbi = float(davies_bouldin_score(X_val, labels))
            chi = float(calinski_harabasz_score(X_val, labels))
            score = (
                0.65 * float(np.clip((sil + 1.0) / 2.0, 0.0, 1.0))
                + 0.35 * float(1.0 / (1.0 + max(dbi, 0.0)))
            )
            kmax_candidate_rows.append({
                "snapshot_date": pd.Timestamp(snap),
                "window_days": int(window_days),
                "k": int(k),
                "n_common_active_customers": int(n_val),
                "silhouette": sil,
                "dbi": dbi,
                "chi_diagnostic": chi,
                "composite_score": score,
                "minimum_cluster_size": int(counts.min()),
                "minimum_cluster_fraction": min_fraction,
                "size_constraint_satisfied": valid_size,
                "fit_seconds": time.perf_counter() - start_k
            })

kmax_candidate_metrics = pd.DataFrame(kmax_candidate_rows)
if kmax_candidate_metrics.empty:
    raise RuntimeError("K_MAX sensitivity produced no validation results.")

# Aggregate the best score across windows within each validation snapshot,
# so the one-SE rule uses temporal snapshots (not window rows) as blocks.
kmax_policy_rows = []
for cap in KMAX_SENSITIVITY_CANDIDATES:
    selected_pairs = []
    per_snapshot_rows = []
    for snap, snap_group in kmax_candidate_metrics.groupby("snapshot_date"):
        window_scores = []
        snapshot_selected_pairs = []
        for window_days in WINDOWS:
            window_group = snap_group[snap_group["window_days"] == int(window_days)]
            eligible = window_group[
                (window_group["k"] <= cap)
                & window_group["size_constraint_satisfied"]
            ]
            if eligible.empty:
                break
            best = eligible.loc[eligible["composite_score"].idxmax()]
            snapshot_selected_pairs.append(best)
            window_scores.append(float(best["composite_score"]))
        # Include a snapshot only if every candidate window had a valid choice.
        # This prevents partial snapshots from biasing K_MAX aggregate statistics.
        if len(window_scores) == len(WINDOWS):
            selected_pairs.extend(snapshot_selected_pairs)
            per_snapshot_rows.append({
                "snapshot_date": pd.Timestamp(snap),
                "mean_score_across_windows": float(np.mean(window_scores))
            })
    if per_snapshot_rows:
        vals = np.asarray([r["mean_score_across_windows"] for r in per_snapshot_rows], dtype=float)
        selected_df = pd.DataFrame(selected_pairs)
        kmax_policy_rows.append({
            "k_max": int(cap),
            "validation_snapshots_evaluated": int(len(per_snapshot_rows)),
            "validation_window_snapshot_pairs": int(len(selected_pairs)),
            "mean_best_score": float(np.mean(vals)),
            "std_best_score": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
            "se_best_score": float(np.std(vals, ddof=1) / np.sqrt(len(vals))) if len(vals) > 1 else 0.0,
            "mean_selected_k": float(selected_df["k"].mean()),
            "selected_k_values": ",".join(map(str, selected_df["k"].astype(int).tolist())),
            "mean_min_cluster_fraction": float(selected_df["minimum_cluster_fraction"].mean()),
            "all_selected_partitions_satisfy_size_constraint": bool(selected_df["size_constraint_satisfied"].all())
        })

kmax_sensitivity_summary = pd.DataFrame(kmax_policy_rows).sort_values("k_max").reset_index(drop=True)
if kmax_sensitivity_summary.empty:
    raise RuntimeError("No K_MAX candidate produced a complete validation result.")

best_row = kmax_sensitivity_summary.loc[kmax_sensitivity_summary["mean_best_score"].idxmax()]
one_se_threshold = float(best_row["mean_best_score"] - best_row["se_best_score"])
within_one_se = kmax_sensitivity_summary[
    kmax_sensitivity_summary["mean_best_score"] >= one_se_threshold
]
K_MAX_RECOMMENDED = int(within_one_se["k_max"].min())
K_MAX = K_MAX_RECOMMENDED

print("Original configured K_MAX:", KMAX_DEFAULT_BEFORE_VALIDATION)
print("Validation-selected K_MAX (one-SE rule):", K_MAX_RECOMMENDED)
print("Selection population: common customers active in all candidate windows")
print("One-SE score threshold:", round(one_se_threshold, 5))
display(kmax_sensitivity_summary.round(4))
fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=kmax_sensitivity_summary, x="k_max", y="mean_best_score", marker="o", ax=ax)
ax.axhline(one_se_threshold, color="grey", linestyle="--", label="One-SE threshold")
ax.axvline(K_MAX_RECOMMENDED, color="green", linestyle=":", label=f"Selected K_MAX={K_MAX_RECOMMENDED}")
ax.set_title("K_MAX sensitivity on validation common-active cohorts")
ax.set_xlabel("Candidate K_MAX")
ax.set_ylabel("Mean best validation composite score")
ax.legend()
plt.tight_layout()
plt.show()


Cell 8 — Adaptive K-Means

In [ ]:
def normalize_score(series, higher_is_better=True):

    min_value = series.min()
    max_value = series.max()

    if np.isclose(min_value, max_value):

        return pd.Series(
            0.5,
            index=series.index
        )

    normalized = (
        (series - min_value) /
        (max_value - min_value)
    )

    if not higher_is_better:

        normalized = 1 - normalized

    return normalized


def evaluate_k_candidates(
    X,
    k_min=K_MIN,
    k_max=K_MAX
):
    """Select K with bounded Silhouette/DBI utilities.

    Calinski-Harabasz remains a diagnostic only. It is not included in the
    selection score because its scale can vary strongly with sample size and
    model complexity. Candidate K values are evaluated on the same X matrix.
    """
    X = np.asarray(X, dtype=float)
    n_samples = len(X)
    if X.ndim != 2 or n_samples < max(10, int(k_min) + 1):
        return None, pd.DataFrame()
    if not np.isfinite(X).all():
        raise ValueError("X contains non-finite values.")
    if int(k_min) < 2 or int(k_max) < int(k_min):
        raise ValueError("Invalid K range.")

    candidates = {}
    rejected_small_clusters = {}
    min_cluster_size = max(2, int(np.ceil(MIN_CLUSTER_FRACTION * n_samples)))
    sample_size = min(int(MAX_SILHOUETTE_SAMPLE), n_samples)

    for k in range(int(k_min), min(int(k_max) + 1, n_samples)):
        model = KMeans(
            n_clusters=k,
            random_state=RANDOM_STATE,
            n_init=KMEANS_N_INIT,
            algorithm="lloyd"
        )
        labels = model.fit_predict(X)
        unique_labels = np.unique(labels)
        if len(unique_labels) < 2 or len(unique_labels) >= n_samples:
            continue

        cluster_counts = np.bincount(labels, minlength=k)
        try:
            silhouette = float(silhouette_score(
                X, labels, sample_size=sample_size, random_state=RANDOM_STATE
            ))
            dbi = float(davies_bouldin_score(X, labels))
            chi = float(calinski_harabasz_score(X, labels))
        except Exception:
            continue

        item = {
            "model": model,
            "labels": labels,
            "silhouette": silhouette,
            "dbi": dbi,
            "chi": chi,
            "inertia": float(model.inertia_),
            "cluster_sizes": cluster_counts,
            "minimum_cluster_size": int(cluster_counts.min()),
            "minimum_cluster_fraction": float(cluster_counts.min() / n_samples),
            "cluster_size_constraint_satisfied": bool(cluster_counts.min() >= min_cluster_size)
        }
        if item["cluster_size_constraint_satisfied"]:
            candidates[k] = item
        else:
            rejected_small_clusters[k] = item

    fallback_used = False
    if not candidates and rejected_small_clusters:
        # Do not silently discard every K when the size constraint is too strict.
        # Use the least-violating valid partition and expose the fallback status.
        fallback_used = True
        candidates = rejected_small_clusters

    if not candidates:
        return None, pd.DataFrame()

    rows = []
    for k, item in candidates.items():
        silhouette_utility = float(np.clip((item["silhouette"] + 1.0) / 2.0, 0.0, 1.0))
        dbi_utility = float(1.0 / (1.0 + max(item["dbi"], 0.0)))
        composite = 0.65 * silhouette_utility + 0.35 * dbi_utility
        rows.append({
            "k": int(k),
            "silhouette": item["silhouette"],
            "dbi": item["dbi"],
            "chi": item["chi"],
            "inertia": item["inertia"],
            "silhouette_utility": silhouette_utility,
            "dbi_utility": dbi_utility,
            "composite_score": composite,
            "minimum_cluster_size": item["minimum_cluster_size"],
            "minimum_cluster_fraction": item["minimum_cluster_fraction"],
            "cluster_size_constraint_satisfied": item["cluster_size_constraint_satisfied"],
            "fallback_used": fallback_used
        })

    metrics = pd.DataFrame(rows).set_index("k").sort_index()
    best_k = int(metrics["composite_score"].idxmax())
    best_result = {
        "best_k": best_k,
        **candidates[best_k],
        "selection_status": "fallback_cluster_size_constraint" if fallback_used else "cluster_size_constraint_satisfied"
    }
    metrics["selected"] = metrics.index == best_k
    return best_result, metrics


# Test Adaptive K-Means
best_result, k_metrics = evaluate_k_candidates(
    X_test
)

if best_result is not None:

    print("Selected K:", best_result["best_k"])

    print(
        "Silhouette:",
        round(best_result["silhouette"], 4)
    )

    print(
        "DBI:",
        round(best_result["dbi"], 4)
    )

    print(
        "CHI:",
        round(best_result["chi"], 4)
    )

    display(k_metrics)

else:

    print("Unable to find valid K.")

Cell 9 — Adaptive Window Evaluation trên common cohort

Các window được fit trên cohort riêng, nhưng chất lượng của mô hình được so sánh trên cùng giao của CustomerID giữa các window. CHI chỉ giữ để báo cáo, không dùng để xếp hạng window.


**Research correction:** Quality is evaluated only on the intersection of customers active in every candidate window. This removes the artificial zero-inflated active/dormant split from the primary Silhouette/DBI comparison. Final clustering and evolution use each selected window's native-active population; dormant customers and activity transitions are reported separately.


In [ ]:
def evaluate_windows_at_snapshot(
    data,
    snapshot_date,
    min_common_customers=10
):
    """Compare windows on the same customers who are active in every window.

    The common-active cohort is the intersection of candidate-window customer
    IDs (normally equal to customers active in the shortest nested window).
    All candidate RFM tables are reindexed to these exact IDs; no synthetic
    F=M=0 rows enter the primary Silhouette/DBI score. This prevents the score
    from being dominated by a simple active-vs-dormant split.

    After choosing K on the common-active cohort, a separate K-Means model is
    fitted on every customer natively active in that candidate window. Those
    native-active assignments are used for temporal stability and evolution.
    Customers in the 91-day reference population but inactive in the candidate
    window are reported as dormant and are not forced into a behavioural
    cluster. Their activity transitions are evaluated separately in Cell 12.
    """
    windows = sorted({int(w) for w in WINDOWS})
    if not windows:
        raise ValueError("WINDOWS must contain at least one window.")
    reference_window = int(max(windows))
    native_rfm_by_window = {
        window: calculate_rfm(data, snapshot_date, window_days=window)
        for window in windows
    }
    reference_rfm = native_rfm_by_window[reference_window]
    if reference_rfm.empty:
        return pd.DataFrame(), {}

    active_sets = [set(frame.index.tolist()) for frame in native_rfm_by_window.values() if not frame.empty]
    if len(active_sets) != len(windows):
        raise ValueError("At least one candidate window has no active customers.")
    common_active_ids = pd.Index(sorted(set.intersection(*active_sets)), name="CustomerID")
    if len(common_active_ids) < min_common_customers:
        raise ValueError(
            f"Common-active cohort has only {len(common_active_ids)} customers; "
            f"at least {min_common_customers} are required."
        )

    candidate_details = {}
    candidate_rows = []
    n_reference = int(len(reference_rfm))

    for window in windows:
        native_rfm = native_rfm_by_window[window].copy()
        common_rfm = native_rfm.reindex(common_active_ids).copy()
        if common_rfm.isna().any().any():
            raise RuntimeError(f"Window {window}: common-active RFM contains missing rows.")
        if (common_rfm[["Frequency", "Monetary"]] <= 0).any().any():
            raise RuntimeError(
                f"Window {window}: common-active cohort contains non-positive F/M; "
                "check nested-window intersection logic."
            )

        # Quality and K selection use exactly the same customer IDs for all windows.
        quality_X, quality_scaler, quality_transformed = transform_rfm(common_rfm)
        best_result, k_metrics = evaluate_k_candidates(quality_X)
        if best_result is None:
            continue

        silhouette = float(best_result["silhouette"])
        dbi = float(best_result["dbi"])
        chi = float(best_result["chi"])
        silhouette_utility = float(np.clip((silhouette + 1.0) / 2.0, 0.0, 1.0))
        dbi_utility = float(1.0 / (1.0 + max(dbi, 0.0)))
        quality = (
            WINDOW_SILHOUETTE_WEIGHT * silhouette_utility
            + WINDOW_DBI_WEIGHT * dbi_utility
        )

        # Fit the selected K on the full native-active cohort for this window.
        # This is the population used for temporal tracking/evolution.
        selected_k = int(best_result["best_k"])
        native_X, native_scaler, native_transformed = transform_rfm(native_rfm)
        native_model = KMeans(
            n_clusters=selected_k,
            random_state=RANDOM_STATE,
            n_init=KMEANS_N_INIT,
            algorithm="lloyd"
        )
        native_labels = native_model.fit_predict(native_X)
        if len(np.unique(native_labels)) < 2:
            raise RuntimeError(
                f"Window {window}: native-active fit collapsed to fewer than two clusters."
            )

        native_silhouette = float("nan")
        native_dbi = float("nan")
        native_chi = float("nan")
        if len(native_labels) > selected_k:
            native_sample_size = min(MAX_SILHOUETTE_SAMPLE, len(native_labels))
            native_silhouette = float(silhouette_score(
                native_X, native_labels,
                sample_size=native_sample_size,
                random_state=RANDOM_STATE
            ))
            native_dbi = float(davies_bouldin_score(native_X, native_labels))
            native_chi = float(calinski_harabasz_score(native_X, native_labels))

        inactive_count = int(n_reference - len(native_rfm))
        active_coverage = float(len(native_rfm) / n_reference) if n_reference else np.nan
        native_counts = np.bincount(native_labels, minlength=selected_k)

        candidate_details[window] = {
            # Primary record for temporal tracking: native-active customers only.
            "rfm": native_rfm.copy(),
            "X": np.asarray(native_X),
            "scaler": native_scaler,
            "transformed": native_transformed.copy(),
            "labels": np.asarray(native_labels).copy(),
            "model": native_model,
            # Separate common-active representation used for window quality.
            "quality_rfm": common_rfm.copy(),
            "quality_X": np.asarray(quality_X),
            "quality_scaler": quality_scaler,
            "quality_transformed": quality_transformed.copy(),
            "quality_labels": np.asarray(best_result["labels"]).copy(),
            "quality_model": best_result["model"],
            "k_metrics": k_metrics,
            "k": selected_k,
            "silhouette": silhouette,
            "dbi": dbi,
            "chi": chi,
            "native_silhouette": native_silhouette,
            "native_dbi": native_dbi,
            "native_chi": native_chi,
            "native_rfm": native_rfm.copy(),
            "native_X": np.asarray(native_X),
            "native_scaler": native_scaler,
            "native_transformed": native_transformed.copy(),
            "native_labels": np.asarray(native_labels).copy(),
            "native_model": native_model,
            "selection_status": best_result["selection_status"],
            "n_common_evaluation_customers": int(len(common_active_ids)),
            "n_common_active_customers": int(len(common_active_ids)),
            "n_reference_cohort_customers": n_reference,
            "n_native_customers": int(len(native_rfm)),
            "n_inactive_in_window": inactive_count,
            "activity_coverage_fraction": active_coverage,
            "reference_window_days": reference_window,
            "common_active_window_days": int(min(windows)),
            "reference_customer_ids": reference_rfm.index.copy(),
            "common_active_customer_ids": common_active_ids.copy(),
            "native_min_cluster_fraction": float(native_counts.min() / len(native_labels))
        }

        candidate_rows.append({
            "window": int(window),
            "k": selected_k,
            "n_customers": int(len(native_rfm)),
            "n_native_customers": int(len(native_rfm)),
            "n_eval_customers": int(len(common_active_ids)),
            "n_common_active_customers": int(len(common_active_ids)),
            "n_reference_cohort_customers": n_reference,
            "n_inactive_in_window": inactive_count,
            "activity_coverage_fraction": active_coverage,
            "silhouette": silhouette,
            "dbi": dbi,
            "native_silhouette": native_silhouette,
            "native_dbi": native_dbi,
            "chi": chi,
            "quality": float(quality),
            "quality_status": "common_active_cohort",
            "k_selection_status": best_result["selection_status"],
            "minimum_cluster_fraction": float(best_result["minimum_cluster_fraction"]),
            "native_minimum_cluster_fraction": float(native_counts.min() / len(native_labels))
        })

    table = pd.DataFrame(candidate_rows)
    if table.empty or len(candidate_details) != len(windows):
        raise RuntimeError("Not all candidate windows produced a valid active-cohort clustering result.")
    return table, candidate_details


# Test one snapshot.
window_table, window_details = evaluate_windows_at_snapshot(
    df,
    snapshot_dates[0]
)
print("Quality cohort: common customers active in all candidate windows.")
print("Dormant customers are reported separately and are not included in K-Means.")
display(window_table.round(4))


# Cell 10 — Adaptive Rolling Window

Chọn cửa sổ bằng common-cohort Quality và temporal Stability. ARI được chuyển thành stability utility trong [0, 1], với ARI <= 0 không nhận điểm thưởng; stability còn được điều chỉnh theo tỷ lệ overlap của cohort. Window challenger phải thắng incumbent đủ hysteresis margin trong số snapshot liên tiếp đã cấu hình mới được chuyển. Cell này cũng tạo bảng sensitivity cho các chính sách hysteresis (replay trên điểm đã tính, không fit lại K-Means).

In [ ]:
# ============================================================
# CELL 10 - ADAPTIVE ROLLING WINDOW
# Common-cohort quality + overlap-aware temporal stability
# + persistence-confirmed hysteresis + policy sensitivity analysis
# ============================================================

import time
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score


def calculate_temporal_stability(
    previous_record,
    current_details,
    min_common_customers=10,
    min_overlap_fraction=STABILITY_MIN_OVERLAP_FRACTION,
    full_overlap_fraction=STABILITY_FULL_OVERLAP_FRACTION,
    verbose=True
):
    """Calculate label-invariant ARI stability on shared customers.

    ARI itself is preserved as a diagnostic. Only positive ARI receives a
    stability reward; ARI <= 0 maps to zero, not 0.5. The reward is then
    discounted when the common cohort covers only a small fraction of the
    smaller snapshot cohort.
    """
    empty_result = {
        "ari": np.nan,
        "ari_utility": np.nan,
        "overlap_reliability": np.nan,
        "stability": np.nan,
        "n_common": 0,
        "common_fraction_smaller_cohort": np.nan,
        "status": "unavailable"
    }

    if previous_record is None:
        return {**empty_result, "status": "first_snapshot"}

    previous_rfm = previous_record["rfm"]
    current_rfm = current_details["rfm"]

    previous_series = pd.Series(
        np.asarray(previous_record["labels"]),
        index=previous_rfm.index
    )
    current_series = pd.Series(
        np.asarray(current_details["labels"]),
        index=current_rfm.index
    )

    common_customers = previous_series.index.intersection(current_series.index)
    n_common = int(len(common_customers))
    smaller_cohort = max(1, min(len(previous_series), len(current_series)))
    common_fraction = float(n_common / smaller_cohort)

    if n_common < min_common_customers:
        return {
            **empty_result,
            "n_common": n_common,
            "common_fraction_smaller_cohort": common_fraction,
            "status": "insufficient_customer_count"
        }
    if not 0 <= min_overlap_fraction <= 1:
        raise ValueError("min_overlap_fraction must be in [0, 1].")
    if common_fraction < min_overlap_fraction:
        return {
            **empty_result,
            "n_common": n_common,
            "common_fraction_smaller_cohort": common_fraction,
            "status": "insufficient_overlap_fraction"
        }

    if not 0 < full_overlap_fraction <= 1:
        raise ValueError("full_overlap_fraction must be in (0, 1].")

    ari = float(adjusted_rand_score(
        previous_series.loc[common_customers],
        current_series.loc[common_customers]
    ))

    # ARI = 0 is chance-level agreement, so it must not receive 0.5 credit.
    ari_utility = float(np.clip(ari, 0.0, 1.0))

    # Reliability rises linearly with cohort overlap, reaching 1 at the
    # configured full-overlap fraction. Raw ARI remains separately available.
    overlap_reliability = float(np.clip(
        common_fraction / full_overlap_fraction,
        0.0,
        1.0
    ))
    stability = ari_utility * overlap_reliability

    return {
        "ari": ari,
        "ari_utility": ari_utility,
        "overlap_reliability": overlap_reliability,
        "stability": float(stability),
        "n_common": n_common,
        "common_fraction_smaller_cohort": common_fraction,
        "status": "valid"
    }


def _simulate_hysteresis_policy(
    history,
    margin,
    confirmations,
    initial_window=None
):
    """Replay a hysteresis policy on cached per-snapshot candidate scores.

    This is a policy sensitivity audit only: it does not refit any model.
    """
    if margin < 0:
        raise ValueError("margin must be non-negative.")
    if confirmations < 1:
        raise ValueError("confirmations must be >= 1.")

    previous_window = initial_window
    pending_window = None
    pending_count = 0
    selected_windows = []
    selected_scores = []
    oracle_scores = []

    for record in history:
        candidate_table = record.get("window_table", pd.DataFrame()).copy()
        if candidate_table.empty:
            continue
        candidate_table["window"] = candidate_table["window"].astype(int)
        valid = candidate_table[
            candidate_table["adaptive_score"].notna()
        ].copy()
        if valid.empty:
            continue

        best_row = valid.loc[valid["adaptive_score"].idxmax()]
        best_window = int(best_row["window"])
        best_score = float(best_row["adaptive_score"])
        oracle_scores.append(best_score)

        if previous_window is None:
            selected_window = best_window
            pending_window, pending_count = None, 0
        else:
            incumbent_rows = valid[valid["window"] == previous_window]
            if incumbent_rows.empty:
                selected_window = best_window
                pending_window, pending_count = None, 0
            else:
                incumbent_score = float(incumbent_rows.iloc[0]["adaptive_score"])
                improvement = best_score - incumbent_score

                if best_window == previous_window or improvement < margin:
                    selected_window = previous_window
                    pending_window, pending_count = None, 0
                else:
                    if pending_window == best_window:
                        pending_count += 1
                    else:
                        pending_window, pending_count = best_window, 1

                    if pending_count >= confirmations:
                        selected_window = best_window
                        pending_window, pending_count = None, 0
                    else:
                        selected_window = previous_window

        chosen_rows = valid[valid["window"] == selected_window]
        chosen_score = float(chosen_rows.iloc[0]["adaptive_score"])
        selected_windows.append(selected_window)
        selected_scores.append(chosen_score)
        previous_window = selected_window

    if not selected_windows:
        return {
            "margin": margin,
            "confirmations": confirmations,
            "snapshots": 0,
            "switches": 0,
            "mean_selected_score": np.nan,
            "mean_regret_vs_snapshot_oracle": np.nan,
            "window_28_count": 0,
            "window_56_count": 0,
            "window_91_count": 0,
            "selected_sequence": ""
        }

    counts = pd.Series(selected_windows).value_counts()
    regrets = np.asarray(oracle_scores) - np.asarray(selected_scores)
    switches = int(np.sum(np.asarray(selected_windows[1:]) != np.asarray(selected_windows[:-1])))
    return {
        "margin": float(margin),
        "confirmations": int(confirmations),
        "snapshots": int(len(selected_windows)),
        "switches": switches,
        "mean_selected_score": float(np.mean(selected_scores)),
        "mean_regret_vs_snapshot_oracle": float(np.mean(regrets)),
        "window_28_count": int(counts.get(28, 0)),
        "window_56_count": int(counts.get(56, 0)),
        "window_91_count": int(counts.get(91, 0)),
        "selected_sequence": "-".join(map(str, selected_windows))
    }


def _simulate_adaptive_parameter_policy(
    history,
    quality_weight,
    stability_weight,
    coverage_weight,
    margin,
    confirmations,
    score_start_index=0
):
    """Replay policy over train+validation while scoring only validation.

    The policy is evaluated only on validation snapshots. No clustering model
    is refitted. Missing stability at the first validation transition is
    removed from the denominator rather than imputed as zero.
    """
    quality_weight = float(quality_weight)
    stability_weight = float(stability_weight)
    coverage_weight = float(coverage_weight)
    margin = float(margin)
    confirmations = int(confirmations)
    if min(quality_weight, stability_weight, coverage_weight) < 0:
        raise ValueError("Policy weights must be non-negative.")
    if not np.isclose(quality_weight + stability_weight + coverage_weight, 1.0):
        raise ValueError("Quality, stability and coverage weights must sum to 1.")
    if quality_weight <= 0 or margin < 0 or confirmations < 1:
        raise ValueError("Quality weight must be positive; margin >= 0; confirmations >= 1.")

    previous_window = None
    pending_window = None
    pending_count = 0
    selected_windows, selected_scores, oracle_scores = [], [], []
    per_snapshot_scores = []
    validation_switches = 0

    for history_index, record in enumerate(history):
        table = record.get("window_table", pd.DataFrame()).copy()
        required = {"window", "quality", "stability_smoothed", "activity_coverage_fraction"}
        if table.empty or not required.issubset(table.columns):
            continue
        quality = pd.to_numeric(table["quality"], errors="coerce")
        stability = pd.to_numeric(table["stability_smoothed"], errors="coerce")
        coverage = pd.to_numeric(table["activity_coverage_fraction"], errors="coerce")
        has_stability = stability.notna()
        has_coverage = coverage.notna()
        denominator = (
            quality_weight
            + stability_weight * has_stability.astype(float)
            + coverage_weight * has_coverage.astype(float)
        )
        numerator = (
            quality_weight * quality
            + stability_weight * stability.fillna(0.0)
            + coverage_weight * coverage.fillna(0.0)
        )
        table["replayed_score"] = numerator / denominator.replace(0, np.nan)
        valid = table[table["quality"].notna() & table["replayed_score"].notna()].copy()
        if valid.empty:
            continue
        best_row = valid.loc[valid["replayed_score"].idxmax()]
        best_window = int(best_row["window"])
        selected_window = best_window
        previous_window_before_decision = previous_window

        if previous_window is not None:
            incumbent = valid[valid["window"].astype(int) == previous_window]
            if incumbent.empty:
                pending_window, pending_count = None, 0
            else:
                incumbent_score = float(incumbent.iloc[0]["replayed_score"])
                improvement = float(best_row["replayed_score"]) - incumbent_score
                if best_window == previous_window or improvement < margin:
                    selected_window = previous_window
                    pending_window, pending_count = None, 0
                else:
                    if pending_window == best_window:
                        pending_count += 1
                    else:
                        pending_window, pending_count = best_window, 1
                    if pending_count >= confirmations:
                        selected_window = best_window
                        pending_window, pending_count = None, 0
                    else:
                        selected_window = previous_window

        selected_row = valid[valid["window"].astype(int) == selected_window]
        if selected_row.empty:
            selected_window = best_window
            selected_row = valid[valid["window"].astype(int) == best_window]
        chosen_score = float(selected_row.iloc[0]["replayed_score"])
        if history_index >= int(score_start_index):
            oracle_scores.append(float(best_row["replayed_score"]))
            selected_scores.append(chosen_score)
            per_snapshot_scores.append(chosen_score)
            selected_windows.append(selected_window)
            if previous_window_before_decision is not None and previous_window_before_decision != selected_window:
                validation_switches += 1
        previous_window = selected_window

    if not selected_windows:
        return {
            "quality_weight": quality_weight,
            "stability_weight": stability_weight,
            "coverage_weight": coverage_weight,
            "margin": margin,
            "confirmations": confirmations,
            "snapshots": 0,
            "switches": 0,
            "mean_selected_score": np.nan,
            "sd_selected_score": np.nan,
            "se_selected_score": np.nan,
            "mean_regret_vs_snapshot_oracle": np.nan,
            "window_28_count": 0,
            "window_56_count": 0,
            "window_91_count": 0,
            "selected_sequence": ""
        }

    counts = pd.Series(selected_windows).value_counts()
    regrets = np.asarray(oracle_scores) - np.asarray(selected_scores)
    switches = int(validation_switches)
    sd_score = float(np.std(per_snapshot_scores, ddof=1)) if len(per_snapshot_scores) > 1 else 0.0
    return {
        "quality_weight": quality_weight,
        "stability_weight": stability_weight,
        "coverage_weight": coverage_weight,
        "margin": margin,
        "confirmations": confirmations,
        "snapshots": int(len(selected_windows)),
        "switches": switches,
        "mean_selected_score": float(np.mean(selected_scores)),
        "sd_selected_score": sd_score,
        "se_selected_score": float(sd_score / np.sqrt(len(selected_scores))),
        "mean_regret_vs_snapshot_oracle": float(np.mean(regrets)),
        "window_28_count": int(counts.get(28, 0)),
        "window_56_count": int(counts.get(56, 0)),
        "window_91_count": int(counts.get(91, 0)),
        "selected_sequence": "-".join(map(str, selected_windows))
    }


def run_adaptive_rolling_rfm(
    data,
    dates,
    quality_weight=QUALITY_WEIGHT,
    stability_weight=STABILITY_WEIGHT,
    coverage_weight=COVERAGE_WEIGHT,
    hysteresis_margin=HYSTERESIS_MARGIN,
    hysteresis_confirmations=HYSTERESIS_CONFIRMATIONS,
    min_common_customers=10,
    stability_ewma_alpha=0.70,
    full_overlap_fraction=STABILITY_FULL_OVERLAP_FRACTION,
    verbose=True
):
    """Choose a window with common-cohort quality and temporal continuity.

    A challenger must exceed the incumbent by `hysteresis_margin` for
    `hysteresis_confirmations` consecutive snapshots before switching. This
    prevents both rapid oscillation and indefinite lock-in caused by an
    oversized fixed hysteresis margin.
    """
    if min(quality_weight, stability_weight, coverage_weight) < 0:
        raise ValueError("All adaptive policy weights must be non-negative.")
    if not np.isclose(quality_weight + stability_weight + coverage_weight, 1.0):
        raise ValueError("quality_weight + stability_weight + coverage_weight must equal 1.")
    if quality_weight <= 0:
        raise ValueError("quality_weight must be > 0.")
    if hysteresis_margin < 0:
        raise ValueError("hysteresis_margin must be non-negative.")
    if hysteresis_confirmations < 1:
        raise ValueError("hysteresis_confirmations must be >= 1.")
    if min_common_customers < 2:
        raise ValueError("min_common_customers must be at least 2.")
    if not 0 < stability_ewma_alpha <= 1:
        raise ValueError("stability_ewma_alpha must be in (0, 1].")
    if not 0 < full_overlap_fraction <= 1:
        raise ValueError("full_overlap_fraction must be in (0, 1].")
    if dates is None or len(dates) == 0:
        raise ValueError("No snapshot dates were provided.")

    history = []
    previous_window = None
    previous_window_records = {}
    pending_window = None
    pending_switch_count = 0

    for index, snapshot in enumerate(dates):
        start_time = time.time()
        snapshot = pd.Timestamp(snapshot)
        print("\n" + "-" * 65)
        print(f"Processing snapshot {index + 1}/{len(dates)}: {snapshot.date()}")

        table, details = evaluate_windows_at_snapshot(
            data,
            snapshot,
            min_common_customers=min_common_customers
        )
        if table.empty or not details:
            if verbose:
                print("WARNING: No valid candidate windows; temporal continuity reset.")
            previous_window_records = {}
            previous_window = None
            pending_window, pending_switch_count = None, 0
            continue

        table = table.copy()
        table["window"] = table["window"].astype(int)

        ari_values = []
        ari_utilities = []
        stability_current = []
        stability_smoothed = []
        stability_reliabilities = []
        common_counts = []
        common_fractions = []
        stability_status = []

        for _, row in table.iterrows():
            window = int(row["window"])
            current_details = details[window]
            previous_record = previous_window_records.get(window)
            stability_result = calculate_temporal_stability(
                previous_record,
                current_details,
                min_common_customers=min_common_customers,
                min_overlap_fraction=STABILITY_MIN_OVERLAP_FRACTION,
                full_overlap_fraction=full_overlap_fraction
            )

            current_score = stability_result["stability"]
            previous_smoothed = (
                previous_record.get("stability_smoothed", np.nan)
                if previous_record is not None else np.nan
            )

            if pd.notna(current_score):
                if pd.notna(previous_smoothed):
                    smooth_score = (
                        stability_ewma_alpha * current_score
                        + (1.0 - stability_ewma_alpha) * previous_smoothed
                    )
                else:
                    smooth_score = current_score
            else:
                smooth_score = np.nan

            ari_values.append(stability_result["ari"])
            ari_utilities.append(stability_result["ari_utility"])
            stability_current.append(current_score)
            stability_smoothed.append(smooth_score)
            stability_reliabilities.append(stability_result["overlap_reliability"])
            common_counts.append(stability_result["n_common"])
            common_fractions.append(stability_result["common_fraction_smaller_cohort"])
            stability_status.append(stability_result["status"])

        table["temporal_ari"] = ari_values
        table["ari_positive_utility"] = ari_utilities
        table["stability_reliability"] = stability_reliabilities
        table["stability"] = stability_current
        table["stability_smoothed"] = stability_smoothed
        table["common_customers"] = common_counts
        table["common_fraction_smaller_cohort"] = common_fractions
        table["stability_status"] = stability_status

        # Three explicit objectives: behavioural clustering quality, temporal
        # stability among recurring active customers, and reference-population
        # coverage. Unavailable stability is omitted and remaining weights are
        # renormalized; it is never imputed as a false zero.
        has_stability = table["stability_smoothed"].notna()
        has_coverage = table["activity_coverage_fraction"].notna()
        effective_denominator = (
            quality_weight
            + stability_weight * has_stability.astype(float)
            + coverage_weight * has_coverage.astype(float)
        )
        weighted_numerator = (
            quality_weight * table["quality"]
            + stability_weight * table["stability_smoothed"].fillna(0.0)
            + coverage_weight * table["activity_coverage_fraction"].fillna(0.0)
        )
        table["coverage_component"] = table["activity_coverage_fraction"]
        table["adaptive_score"] = weighted_numerator / effective_denominator
        table["effective_quality_weight"] = quality_weight / effective_denominator
        table["effective_stability_weight"] = (
            stability_weight * has_stability.astype(float) / effective_denominator
        )
        table["effective_coverage_weight"] = (
            coverage_weight * has_coverage.astype(float) / effective_denominator
        )
        table["adaptive_score"] = table["adaptive_score"].replace(
            [np.inf, -np.inf], np.nan
        )

        valid_table = table[
            table["adaptive_score"].notna() & table["quality"].notna()
        ].copy()
        if valid_table.empty:
            if verbose:
                print("WARNING: No valid adaptive scores; temporal continuity reset.")
            previous_window_records = {}
            previous_window = None
            pending_window, pending_switch_count = None, 0
            continue

        best_row = valid_table.loc[valid_table["adaptive_score"].idxmax()]
        best_window = int(best_row["window"])
        best_score = float(best_row["adaptive_score"])
        selected_window = best_window
        hysteresis_applied = False
        previous_score = np.nan
        score_improvement = np.nan
        switch_reason = "initial_selection"
        pending_candidate_record = None
        pending_count_record = 0

        if previous_window is not None:
            previous_candidate = valid_table[valid_table["window"] == previous_window]
            if previous_candidate.empty:
                # The incumbent is no longer a valid candidate: switch rather
                # than keeping a window that cannot be evaluated this period.
                selected_window = best_window
                pending_window, pending_switch_count = None, 0
                switch_reason = "incumbent_unavailable"
            else:
                previous_score = float(previous_candidate.iloc[0]["adaptive_score"])
                score_improvement = best_score - previous_score

                if best_window == previous_window:
                    selected_window = previous_window
                    pending_window, pending_switch_count = None, 0
                    switch_reason = "incumbent_is_best"
                elif score_improvement < hysteresis_margin:
                    selected_window = previous_window
                    pending_window, pending_switch_count = None, 0
                    hysteresis_applied = True
                    switch_reason = "challenger_within_margin"
                else:
                    if pending_window == best_window:
                        pending_switch_count += 1
                    else:
                        pending_window = best_window
                        pending_switch_count = 1

                    if pending_switch_count >= hysteresis_confirmations:
                        selected_window = best_window
                        pending_window, pending_switch_count = None, 0
                        switch_reason = "challenger_confirmed"
                    else:
                        selected_window = previous_window
                        hysteresis_applied = True
                        switch_reason = "awaiting_confirmation"

        pending_candidate_record = pending_window
        pending_count_record = pending_switch_count

        if selected_window not in details:
            raise RuntimeError(f"Selected window {selected_window} does not exist.")

        selected = details[selected_window]
        selected_row = valid_table[valid_table["window"] == selected_window].iloc[0]

        # Candidate quality is scored on a common-active cohort. Final selected
        # labels and temporal tracking use the full native-active population for
        # the selected window. Reuse the native model already fitted in Cell 9.
        native_rfm = selected["native_rfm"].copy()
        native_X = np.asarray(selected["native_X"])
        native_scaler = selected["native_scaler"]
        native_transformed = selected["native_transformed"].copy()
        native_model = selected["native_model"]
        native_labels = np.asarray(selected["native_labels"]).copy()
        native_silhouette = float(selected["native_silhouette"])
        native_dbi = float(selected["native_dbi"])
        native_chi = float(selected["native_chi"])
        selected_k = int(selected["k"])

        elapsed = time.time() - start_time

        record = {
            "snapshot_date": snapshot,
            "window": selected_window,
            "k": int(selected["k"]),
            "n_customers": int(len(native_rfm)),
            "n_native_customers": int(len(native_rfm)),
            "n_reference_cohort_customers": int(selected["n_reference_cohort_customers"]),
            "n_inactive_in_window": int(selected["n_inactive_in_window"]),
            "activity_coverage_fraction": float(selected["activity_coverage_fraction"]),
            "reference_window_days": int(selected["reference_window_days"]),
            "common_active_window_days": int(selected["common_active_window_days"]),
            "n_eval_customers": int(selected["n_common_evaluation_customers"]),
            "n_common_active_customers": int(selected["n_common_active_customers"]),
            "silhouette": float(selected_row["silhouette"]),
            "dbi": float(selected_row["dbi"]),
            "native_silhouette": native_silhouette,
            "native_dbi": native_dbi,
            "native_chi": native_chi,
            "chi": float(selected["chi"]),
            "quality": float(selected_row["quality"]),
            "temporal_ari": float(selected_row["temporal_ari"]) if pd.notna(selected_row["temporal_ari"]) else np.nan,
            "ari_positive_utility": float(selected_row["ari_positive_utility"]) if pd.notna(selected_row["ari_positive_utility"]) else np.nan,
            "stability_reliability": float(selected_row["stability_reliability"]) if pd.notna(selected_row["stability_reliability"]) else np.nan,
            "stability": float(selected_row["stability"]) if pd.notna(selected_row["stability"]) else np.nan,
            "stability_smoothed": float(selected_row["stability_smoothed"]) if pd.notna(selected_row["stability_smoothed"]) else np.nan,
            "stability_status": selected_row["stability_status"],
            "common_customers": int(selected_row["common_customers"]),
            "common_fraction_smaller_cohort": float(selected_row["common_fraction_smaller_cohort"]) if pd.notna(selected_row["common_fraction_smaller_cohort"]) else np.nan,
            "adaptive_score": float(selected_row["adaptive_score"]),
            "effective_quality_weight": float(selected_row["effective_quality_weight"]),
            "effective_stability_weight": float(selected_row["effective_stability_weight"]),
            "effective_coverage_weight": float(selected_row["effective_coverage_weight"]),
            "coverage_component": float(selected_row["coverage_component"]),
            "best_candidate_window": best_window,
            "previous_window": previous_window,
            "previous_window_score": previous_score,
            "score_improvement": score_improvement,
            "hysteresis_applied": hysteresis_applied,
            "switch_reason": switch_reason,
            "pending_candidate_window": pending_candidate_record,
            "pending_confirmation_count": int(pending_count_record),
            "window_changed": previous_window is not None and selected_window != previous_window,
            "runtime_seconds": elapsed,
            # Native-active cohort for evolution/migration/frozen assignments.
            "rfm": native_rfm.copy(),
            "X": np.asarray(native_X),
            "scaler": native_scaler,
            "transformed_features": native_transformed.copy(),
            "labels": np.asarray(native_labels).copy(),
            "model": native_model,
            "native_rfm": native_rfm.copy(),
            "native_X": np.asarray(native_X),
            "native_labels": np.asarray(native_labels).copy(),
            "native_model": native_model,
            "native_scaler": native_scaler,
            # Keep common-active quality representation explicitly for audit.
            "quality_rfm": selected["quality_rfm"].copy(),
            "quality_labels": np.asarray(selected["quality_labels"]).copy(),
            "quality_X": np.asarray(selected["quality_X"]),
            "window_table": table.copy(),
            # Cache per-window fitted models so validation-selected policy can
            # be replayed without repeating expensive K-Means fits.
            "candidate_details_by_window": details
        }
        history.append(record)

        # Preserve candidate-specific previous-snapshot assignments and EWMA.
        current_window_records = {}
        table_by_window = table.set_index("window", drop=False)
        for window, candidate in details.items():
            row = table_by_window.loc[int(window)]
            current_window_records[int(window)] = {
                "rfm": candidate["rfm"].copy(),
                "labels": np.asarray(candidate["labels"]).copy(),
                "snapshot_date": snapshot,
                "stability_smoothed": float(row["stability_smoothed"]) if pd.notna(row["stability_smoothed"]) else np.nan
            }

        previous_window_records = current_window_records
        previous_window = selected_window

        if verbose:
            print(
                f"Selected window: {selected_window} days | "
                f"best candidate: {best_window} days | reason: {switch_reason}"
            )
            print(f"Selected K: {selected['k']} | native-active customers: {len(native_rfm):,} | common-active quality cohort: {selected['n_common_active_customers']:,}")
            print(f"Dormant in selected window (within 91-day reference): {selected['n_inactive_in_window']:,} ({1.0-selected['activity_coverage_fraction']:.1%})")
            print(f"Silhouette (common cohort): {record['silhouette']:.4f}")
            print(f"Temporal ARI: {record['temporal_ari']}")
            print(f"Stability reliability: {record['stability_reliability']}")
            print(f"Adaptive score: {record['adaptive_score']:.4f}")
            print(
                f"Hysteresis pending: {pending_count_record}/{hysteresis_confirmations} "
                f"| runtime: {elapsed:.2f}s"
            )

    return history



def replay_adaptive_policy_from_cached_candidates(
    cached_history,
    quality_weight,
    stability_weight,
    coverage_weight,
    hysteresis_margin,
    hysteresis_confirmations
):
    """Replay a locked policy using cached candidate models and per-window metrics.

    The expensive RFM transformations and K-Means fits have already been
    performed once for every snapshot/window. Candidate quality and temporal
    stability are policy-independent, so only the score/hysteresis decision is
    replayed after validation tuning. This avoids a second full clustering run.
    """
    weights = [float(quality_weight), float(stability_weight), float(coverage_weight)]
    if min(weights) < 0 or not np.isclose(sum(weights), 1.0) or weights[0] <= 0:
        raise ValueError("Invalid adaptive policy weights.")
    if hysteresis_margin < 0 or int(hysteresis_confirmations) < 1:
        raise ValueError("Invalid hysteresis settings.")

    history = []
    previous_window = None
    pending_window = None
    pending_count = 0

    for source_record in cached_history:
        if "candidate_details_by_window" not in source_record:
            raise RuntimeError("Candidate cache missing; cannot replay policy without refitting.")
        details = source_record["candidate_details_by_window"]
        table = source_record["window_table"].copy()
        if table.empty or not details:
            raise RuntimeError("Cached candidate table/details are empty.")

        has_stability = table["stability_smoothed"].notna()
        has_coverage = table["activity_coverage_fraction"].notna()
        denominator = (
            quality_weight
            + stability_weight * has_stability.astype(float)
            + coverage_weight * has_coverage.astype(float)
        )
        numerator = (
            quality_weight * table["quality"]
            + stability_weight * table["stability_smoothed"].fillna(0.0)
            + coverage_weight * table["activity_coverage_fraction"].fillna(0.0)
        )
        table["coverage_component"] = table["activity_coverage_fraction"]
        table["adaptive_score"] = numerator / denominator.replace(0, np.nan)
        table["effective_quality_weight"] = quality_weight / denominator
        table["effective_stability_weight"] = (
            stability_weight * has_stability.astype(float) / denominator
        )
        table["effective_coverage_weight"] = (
            coverage_weight * has_coverage.astype(float) / denominator
        )
        table["adaptive_score"] = table["adaptive_score"].replace([np.inf, -np.inf], np.nan)

        valid = table[
            table["adaptive_score"].notna() & table["quality"].notna()
        ].copy()
        if valid.empty:
            raise RuntimeError("No valid candidate after policy replay.")

        best_row = valid.loc[valid["adaptive_score"].idxmax()]
        best_window = int(best_row["window"])
        best_score = float(best_row["adaptive_score"])
        selected_window = best_window
        previous_score = np.nan
        score_improvement = np.nan
        hysteresis_applied = False
        switch_reason = "initial_selection"
        if previous_window is not None:
            incumbent = valid[valid["window"].astype(int) == previous_window]
            if incumbent.empty:
                switch_reason = "incumbent_unavailable"
                pending_window, pending_count = None, 0
            else:
                previous_score = float(incumbent.iloc[0]["adaptive_score"])
                score_improvement = best_score - previous_score
                if best_window == previous_window:
                    selected_window = previous_window
                    pending_window, pending_count = None, 0
                    switch_reason = "incumbent_is_best"
                elif score_improvement < hysteresis_margin:
                    selected_window = previous_window
                    pending_window, pending_count = None, 0
                    hysteresis_applied = True
                    switch_reason = "challenger_within_margin"
                else:
                    if pending_window == best_window:
                        pending_count += 1
                    else:
                        pending_window, pending_count = best_window, 1
                    if pending_count >= int(hysteresis_confirmations):
                        selected_window = best_window
                        pending_window, pending_count = None, 0
                        switch_reason = "challenger_confirmed"
                    else:
                        selected_window = previous_window
                        hysteresis_applied = True
                        switch_reason = "awaiting_confirmation"

        selected_window = int(selected_window)
        if selected_window not in details:
            raise RuntimeError(f"Cached details do not contain selected window {selected_window}.")
        selected = details[selected_window]
        selected_row = valid.loc[valid["window"].astype(int) == selected_window].iloc[0]
        native_rfm = selected["native_rfm"].copy()
        native_X = np.asarray(selected["native_X"])
        native_scaler = selected["native_scaler"]
        native_transformed = selected["native_transformed"].copy()
        native_model = selected["native_model"]
        native_labels = np.asarray(selected["native_labels"]).copy()
        row = {
            **source_record,
            "window": selected_window,
            "k": int(selected["k"]),
            "n_customers": int(len(native_rfm)),
            "n_native_customers": int(len(native_rfm)),
            "n_reference_cohort_customers": int(selected["n_reference_cohort_customers"]),
            "n_inactive_in_window": int(selected["n_inactive_in_window"]),
            "activity_coverage_fraction": float(selected["activity_coverage_fraction"]),
            "reference_window_days": int(selected["reference_window_days"]),
            "common_active_window_days": int(selected["common_active_window_days"]),
            "n_eval_customers": int(selected["n_common_evaluation_customers"]),
            "n_common_active_customers": int(selected["n_common_active_customers"]),
            "silhouette": float(selected_row["silhouette"]),
            "dbi": float(selected_row["dbi"]),
            "native_silhouette": float(selected["native_silhouette"]),
            "native_dbi": float(selected["native_dbi"]),
            "native_chi": float(selected["native_chi"]),
            "chi": float(selected["chi"]),
            "quality": float(selected_row["quality"]),
            "temporal_ari": float(selected_row["temporal_ari"]) if pd.notna(selected_row["temporal_ari"]) else np.nan,
            "ari_positive_utility": float(selected_row["ari_positive_utility"]) if pd.notna(selected_row["ari_positive_utility"]) else np.nan,
            "stability_reliability": float(selected_row["stability_reliability"]) if pd.notna(selected_row["stability_reliability"]) else np.nan,
            "stability": float(selected_row["stability"]) if pd.notna(selected_row["stability"]) else np.nan,
            "stability_smoothed": float(selected_row["stability_smoothed"]) if pd.notna(selected_row["stability_smoothed"]) else np.nan,
            "stability_status": selected_row["stability_status"],
            "common_customers": int(selected_row["common_customers"]),
            "common_fraction_smaller_cohort": float(selected_row["common_fraction_smaller_cohort"]) if pd.notna(selected_row["common_fraction_smaller_cohort"]) else np.nan,
            "adaptive_score": float(selected_row["adaptive_score"]),
            "effective_quality_weight": float(selected_row["effective_quality_weight"]),
            "effective_stability_weight": float(selected_row["effective_stability_weight"]),
            "effective_coverage_weight": float(selected_row["effective_coverage_weight"]),
            "coverage_component": float(selected_row["coverage_component"]),
            "best_candidate_window": best_window,
            "previous_window": previous_window,
            "previous_window_score": previous_score,
            "score_improvement": score_improvement,
            "hysteresis_applied": hysteresis_applied,
            "switch_reason": switch_reason,
            "pending_candidate_window": pending_window,
            "pending_confirmation_count": int(pending_count),
            "window_changed": previous_window is not None and selected_window != previous_window,
            # Keep the one expensive candidate-computation time. Replay time is
            # reported separately and is not confused with clustering latency.
            "candidate_computation_runtime_seconds": float(source_record["runtime_seconds"]),
            "policy_replay_runtime_seconds": 0.0,
            "runtime_seconds": float(source_record["runtime_seconds"]),
            "rfm": native_rfm.copy(),
            "X": native_X,
            "scaler": native_scaler,
            "transformed_features": native_transformed,
            "labels": native_labels,
            "model": native_model,
            "native_rfm": native_rfm.copy(),
            "native_X": native_X,
            "native_labels": native_labels,
            "native_model": native_model,
            "native_scaler": native_scaler,
            "quality_rfm": selected["quality_rfm"].copy(),
            "quality_labels": np.asarray(selected["quality_labels"]).copy(),
            "quality_X": np.asarray(selected["quality_X"]),
            "window_table": table.copy()
        }
        row.pop("candidate_details_by_window", None)
        history.append(row)
        previous_window = selected_window

    return history


# ============================================================
# RUN ADAPTIVE ROLLING RFM
# ============================================================
adaptive_history = run_adaptive_rolling_rfm(
    data=df,
    dates=snapshot_dates,
    quality_weight=QUALITY_WEIGHT,
    stability_weight=STABILITY_WEIGHT,
    coverage_weight=COVERAGE_WEIGHT,
    hysteresis_margin=HYSTERESIS_MARGIN,
    hysteresis_confirmations=HYSTERESIS_CONFIRMATIONS,
    min_common_customers=10,
    stability_ewma_alpha=0.70,
    full_overlap_fraction=STABILITY_FULL_OVERLAP_FRACTION,
    verbose=False
)

print("\n" + "=" * 70)
print("ADAPTIVE ROLLING RFM COMPLETED")
print("=" * 70)
print("Snapshots processed:", len(adaptive_history))

if adaptive_history:
    adaptive_results_preview = pd.DataFrame([
        {
            "snapshot_date": r["snapshot_date"],
            "window": r["window"],
            "best_candidate_window": r["best_candidate_window"],
            "k": r["k"],
            "customers": r["n_customers"],
            "evaluation_customers": r["n_eval_customers"],
            "silhouette": r["silhouette"],
            "temporal_ari": r["temporal_ari"],
            "ari_positive_utility": r["ari_positive_utility"],
            "stability_reliability": r["stability_reliability"],
            "stability_smoothed": r["stability_smoothed"],
            "adaptive_score": r["adaptive_score"],
            "switch_reason": r["switch_reason"],
            "pending_confirmation_count": r["pending_confirmation_count"],
            "runtime_seconds": r["runtime_seconds"]
        }
        for r in adaptive_history
    ])
    display(adaptive_results_preview.round(4))
    print("\nSelected window distribution:")
    display(adaptive_results_preview["window"].value_counts().sort_index().to_frame("snapshot_count"))

    # Select policy weights and hysteresis using VALIDATION snapshots only.
    # The final test partition is never used for hyperparameter selection.
    validation_start = len(train_snapshot_dates)
    validation_stop = validation_start + len(validation_snapshot_dates)
    validation_policy_history = adaptive_history[validation_start:validation_stop]
    development_policy_history = adaptive_history[:validation_stop]
    if len(validation_policy_history) != len(validation_snapshot_dates) or len(development_policy_history) != validation_stop:
        raise RuntimeError("Train/validation adaptive policy history is incomplete.")
    initial_validation_incumbent = (
        int(adaptive_history[validation_start - 1]["window"]) if validation_start > 0 else None
    )

    hysteresis_sensitivity_rows = []
    for margin in [0.0, 0.005, 0.01, 0.02]:
        for confirmations in [1, 2, 3]:
            hysteresis_sensitivity_rows.append(
                _simulate_hysteresis_policy(
                    validation_policy_history,
                    margin=margin,
                    confirmations=confirmations,
                    initial_window=initial_validation_incumbent
                )
            )
    adaptive_hysteresis_sensitivity = pd.DataFrame(hysteresis_sensitivity_rows)
    adaptive_hysteresis_sensitivity["is_default_policy"] = (
        np.isclose(adaptive_hysteresis_sensitivity["margin"], HYSTERESIS_MARGIN)
        & (adaptive_hysteresis_sensitivity["confirmations"] == HYSTERESIS_CONFIRMATIONS)
    )
    print("\nHysteresis sensitivity: state warmed on train, scores measured on validation only:")
    display(adaptive_hysteresis_sensitivity.round(4))

    weight_policies = [
        (0.80, 0.20, 0.00),
        (0.70, 0.20, 0.10),
        (0.65, 0.20, 0.15),
        (0.60, 0.20, 0.20),
        (0.70, 0.15, 0.15),
        (0.75, 0.15, 0.10)
    ]
    adaptive_parameter_sensitivity_rows = []
    for q_weight, s_weight, c_weight in weight_policies:
        for margin in [0.0, 0.005, 0.01]:
            for confirmations in [1, 2, 3]:
                adaptive_parameter_sensitivity_rows.append(
                    _simulate_adaptive_parameter_policy(
                        development_policy_history,
                        quality_weight=q_weight,
                        stability_weight=s_weight,
                        coverage_weight=c_weight,
                        margin=margin,
                        confirmations=confirmations,
                        score_start_index=len(train_snapshot_dates)
                    )
                )
    adaptive_parameter_sensitivity = pd.DataFrame(adaptive_parameter_sensitivity_rows)
    adaptive_parameter_sensitivity["is_default_policy"] = (
        np.isclose(adaptive_parameter_sensitivity["quality_weight"], QUALITY_WEIGHT)
        & np.isclose(adaptive_parameter_sensitivity["stability_weight"], STABILITY_WEIGHT)
        & np.isclose(adaptive_parameter_sensitivity["coverage_weight"], COVERAGE_WEIGHT)
        & np.isclose(adaptive_parameter_sensitivity["margin"], HYSTERESIS_MARGIN)
        & (adaptive_parameter_sensitivity["confirmations"] == HYSTERESIS_CONFIRMATIONS)
    )
    print("\nThree-objective policy sensitivity: state replayed over train+validation, scores measured on validation only (54 policies):")
    display(adaptive_parameter_sensitivity.round(4))

    valid_policy_rows = adaptive_parameter_sensitivity[
        adaptive_parameter_sensitivity["snapshots"] == len(validation_snapshot_dates)
        ].copy()
    if valid_policy_rows.empty:
        raise RuntimeError("No complete policy sensitivity run on validation snapshots.")
    best_policy_row = valid_policy_rows.loc[
        valid_policy_rows["mean_selected_score"].idxmax()
    ]
    one_se_limit = float(
        best_policy_row["mean_selected_score"] - best_policy_row["se_selected_score"]
    )
    eligible_policy_rows = valid_policy_rows[
        valid_policy_rows["mean_selected_score"] >= one_se_limit
    ].copy()
    # Within one standard error, prefer lower oracle regret; if close, prefer
    # broader reference-population coverage and then fewer policy switches.
    selected_policy_row = eligible_policy_rows.sort_values(
        ["mean_regret_vs_snapshot_oracle", "coverage_weight", "switches", "confirmations"],
        ascending=[True, False, True, True]
    ).iloc[0]
    SELECTED_ADAPTIVE_POLICY = {
        "quality_weight": float(selected_policy_row["quality_weight"]),
        "stability_weight": float(selected_policy_row["stability_weight"]),
        "coverage_weight": float(selected_policy_row["coverage_weight"]),
        "hysteresis_margin": float(selected_policy_row["margin"]),
        "hysteresis_confirmations": int(selected_policy_row["confirmations"]),
        "selection_rule": "validation-only one-standard-error; minimum regret, then broader coverage, then fewer switches"
    }
    print("\nValidation-selected adaptive policy (locked before test):")
    print(SELECTED_ADAPTIVE_POLICY)
    print("Validation one-SE threshold:", round(one_se_limit, 6))

    # Apply the validation-selected policy by replaying cached candidate
    # evaluations. This prevents a second expensive round of K-Means fitting.
    QUALITY_WEIGHT = SELECTED_ADAPTIVE_POLICY["quality_weight"]
    STABILITY_WEIGHT = SELECTED_ADAPTIVE_POLICY["stability_weight"]
    COVERAGE_WEIGHT = SELECTED_ADAPTIVE_POLICY["coverage_weight"]
    HYSTERESIS_MARGIN = SELECTED_ADAPTIVE_POLICY["hysteresis_margin"]
    HYSTERESIS_CONFIRMATIONS = SELECTED_ADAPTIVE_POLICY["hysteresis_confirmations"]
    adaptive_history = replay_adaptive_policy_from_cached_candidates(
        cached_history=adaptive_history,
        quality_weight=QUALITY_WEIGHT,
        stability_weight=STABILITY_WEIGHT,
        coverage_weight=COVERAGE_WEIGHT,
        hysteresis_margin=HYSTERESIS_MARGIN,
        hysteresis_confirmations=HYSTERESIS_CONFIRMATIONS
    )
    if len(adaptive_history) != len(snapshot_dates):
        raise RuntimeError("Locked adaptive policy did not produce every snapshot.")
    adaptive_results_preview = pd.DataFrame([
        {
            "snapshot_date": r["snapshot_date"],
            "window": r["window"],
            "best_candidate_window": r["best_candidate_window"],
            "k": r["k"],
            "customers": r["n_customers"],
            "common_active_quality_customers": r["n_eval_customers"],
            "activity_coverage": r["activity_coverage_fraction"],
            "silhouette": r["silhouette"],
            "temporal_ari": r["temporal_ari"],
            "stability_smoothed": r["stability_smoothed"],
            "adaptive_score": r["adaptive_score"],
            "switch_reason": r["switch_reason"],
            "runtime_seconds": r["runtime_seconds"]
        }
        for r in adaptive_history
    ])
    print("\nFinal adaptive results after validation-only policy lock (cached-candidate replay):")
    display(adaptive_results_preview.round(4))
    print("Final selected-window distribution:")
    display(adaptive_results_preview["window"].value_counts().sort_index().to_frame("snapshot_count"))

else:
    raise RuntimeError(
        "Adaptive Rolling RFM did not produce valid snapshots. "
        "Check preprocessing, candidate windows and common-cohort size."
    )


Cell 11 — Segment Evolution Tracking

**Population definition (research correction):** Cell 11 tracks behavioural segment evolution among customers active in the selected window. Customers dormant in that window are not forced into a K-Means cluster; their fixed-28-day active/dormant transitions are reported separately in Cell 12. Therefore Birth/Death refer to segment-structure events among the active population, not customer acquisition/churn by themselves.


In [ ]:

# ============================================================
# CELL 11 - SEGMENT EVOLUTION TRACKING
# Improved Research Implementation
# ============================================================

import numpy as np
import pandas as pd

from collections import defaultdict, deque
from scipy.optimize import linear_sum_assignment


# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

# Conservative defaults reduce weak cross-links between unrelated segments.
# Change these values explicitly here when testing alternative thresholds.
EVOLUTION_MIN_OVERLAP = 10
EVOLUTION_COVERAGE_THRESHOLD = 0.50
EVOLUTION_JACCARD_THRESHOLD = 0.30


# ------------------------------------------------------------
# 2. BUILD CLUSTER MEMBERSHIPS
# ------------------------------------------------------------

def build_cluster_memberships(rfm, labels):
    """
    Build mapping:
        cluster_id -> set(customer_ids)

    K-Means labels are treated as temporary identifiers.
    They are NOT assumed to retain identity across snapshots.
    """

    labels = np.asarray(labels)

    if len(rfm) != len(labels):
        raise ValueError(
            "Number of labels does not match RFM customers."
        )

    if not rfm.index.is_unique:
        raise ValueError(
            "Customer IDs in RFM index must be unique."
        )

    membership = {}

    for cluster_id in np.unique(labels):

        mask = labels == cluster_id

        membership[int(cluster_id)] = set(
            rfm.index[mask]
        )

    return membership


# ------------------------------------------------------------
# 3. CLUSTER OVERLAP METRICS
# ------------------------------------------------------------

def calculate_cluster_overlap(old_members, new_members):

    intersection = len(
        old_members.intersection(new_members)
    )

    union = len(
        old_members.union(new_members)
    )

    old_coverage = (
        intersection / len(old_members)
        if len(old_members) > 0 else 0.0
    )

    new_coverage = (
        intersection / len(new_members)
        if len(new_members) > 0 else 0.0
    )

    jaccard = (
        intersection / union
        if union > 0 else 0.0
    )

    return {
        "intersection": intersection,
        "union": union,
        "jaccard": jaccard,
        "old_coverage": old_coverage,
        "new_coverage": new_coverage
    }


# ------------------------------------------------------------
# 4. BUILD OVERLAP GRAPH
# ------------------------------------------------------------

def build_evolution_graph(
    old_membership,
    new_membership,
    min_overlap,
    coverage_threshold,
    jaccard_threshold
):
    """
    Build a bipartite graph between old and new clusters.

    An edge represents a sufficiently strong customer overlap.
    """

    adjacency_old = defaultdict(set)
    adjacency_new = defaultdict(set)

    overlap_records = []

    for old_id, old_members in old_membership.items():

        for new_id, new_members in new_membership.items():

            metrics = calculate_cluster_overlap(
                old_members,
                new_members
            )

            meaningful_edge = (
                metrics["intersection"] >= min_overlap
                and (
                    metrics["old_coverage"] >= coverage_threshold
                    or metrics["new_coverage"] >= coverage_threshold
                    or metrics["jaccard"] >= jaccard_threshold
                )
            )

            overlap_records.append({

                "old_cluster": old_id,

                "new_cluster": new_id,

                **metrics,

                "meaningful_edge": meaningful_edge

            })

            if meaningful_edge:

                adjacency_old[old_id].add(new_id)

                adjacency_new[new_id].add(old_id)

    overlap_df = pd.DataFrame(overlap_records)

    return adjacency_old, adjacency_new, overlap_df


# ------------------------------------------------------------
# 5. FIND CONNECTED COMPONENTS
# ------------------------------------------------------------

def find_evolution_components(
    old_clusters,
    new_clusters,
    adjacency_old,
    adjacency_new
):
    """
    Find connected components in the bipartite overlap graph.

    Components represent relationships between old and new
    segments, independent of arbitrary K-Means label numbers.
    """

    visited_old = set()
    visited_new = set()

    components = []

    # Start traversal from every old cluster.
    for start_old in old_clusters:

        if start_old in visited_old:
            continue

        component_old = set()
        component_new = set()

        queue = deque([
            ("old", start_old)
        ])

        while queue:

            node_type, node_id = queue.popleft()

            if node_type == "old":

                if node_id in visited_old:
                    continue

                visited_old.add(node_id)

                component_old.add(node_id)

                for new_id in adjacency_old[node_id]:

                    if new_id not in visited_new:

                        queue.append(
                            ("new", new_id)
                        )

            else:

                if node_id in visited_new:
                    continue

                visited_new.add(node_id)

                component_new.add(node_id)

                for old_id in adjacency_new[node_id]:

                    if old_id not in visited_old:

                        queue.append(
                            ("old", old_id)
                        )

        components.append(
            (component_old, component_new)
        )

    # New clusters without old-cluster relationships.
    for new_id in new_clusters:

        if new_id not in visited_new:

            components.append(
                (set(), {new_id})
            )

    return components


# ------------------------------------------------------------
# 6. CLASSIFY EVOLUTION EVENTS
# ------------------------------------------------------------

def classify_evolution_event(
    old_count,
    new_count
):

    if old_count == 0 and new_count > 0:

        return "Birth"

    if old_count > 0 and new_count == 0:

        return "Death"

    if old_count == 1 and new_count == 1:

        return "Continuation"

    if old_count == 1 and new_count > 1:

        return "Split"

    if old_count > 1 and new_count == 1:

        return "Merge"

    return "Reorganization"


# ------------------------------------------------------------
# 7. HUNGARIAN MATCHING FOR CUSTOMER MIGRATION
# ------------------------------------------------------------

def align_clusters_for_migration(
    old_labels,
    new_labels
):
    """
    Match old and new clusters using maximum common-customer
    overlap.

    This is a one-to-one matching.

    Unmatched new clusters are not forced into an old cluster.
    """

    old_ids = np.unique(old_labels)

    new_ids = np.unique(new_labels)

    contingency = np.zeros(
        (len(old_ids), len(new_ids)),
        dtype=int
    )

    for i, old_id in enumerate(old_ids):

        for j, new_id in enumerate(new_ids):

            contingency[i, j] = np.sum(
                (old_labels == old_id)
                &
                (new_labels == new_id)
            )

    row_ind, col_ind = linear_sum_assignment(
        -contingency
    )

    mapping = {}

    matched_overlap = 0

    for row, col in zip(row_ind, col_ind):

        # Do not create a mapping for zero-overlap pairs.
        if contingency[row, col] <= 0:
            continue

        new_id = new_ids[col]

        old_id = old_ids[row]

        mapping[int(new_id)] = int(old_id)

        matched_overlap += int(
            contingency[row, col]
        )

    return {
        "mapping": mapping,
        "contingency": contingency,
        "old_cluster_ids": old_ids,
        "new_cluster_ids": new_ids,
        "matched_overlap": matched_overlap
    }


# ------------------------------------------------------------
# 8. CUSTOMER-LEVEL MIGRATION ANALYSIS
# ------------------------------------------------------------

def analyze_customer_migrations(
    previous_record,
    current_record
):
    """
    Evaluate customer migration on the common customer cohort.

    migrated:
        True  -> Customer moved to a different old-cluster identity
                 after Hungarian alignment.
        False -> Customer remains in its matched segment.
        <NA>  -> New cluster has no valid one-to-one old-cluster match.

    Unmatched clusters are reported separately instead of being
    incorrectly interpreted as ordinary migration.
    """

    old_labels = pd.Series(
        np.asarray(previous_record["labels"]),
        index=previous_record["rfm"].index
    )

    new_labels = pd.Series(
        np.asarray(current_record["labels"]),
        index=current_record["rfm"].index
    )

    common_customers = old_labels.index.intersection(
        new_labels.index
    )

    old_y = old_labels.loc[
        common_customers
    ].to_numpy()

    new_y = new_labels.loc[
        common_customers
    ].to_numpy()

    alignment = align_clusters_for_migration(
        old_y,
        new_y
    )

    mapping = alignment["mapping"]

    records = []

    for customer_id, old_cluster, new_cluster in zip(
        common_customers,
        old_y,
        new_y
    ):

        old_cluster = int(old_cluster)

        new_cluster = int(new_cluster)

        matched_old_cluster = mapping.get(
            new_cluster
        )

        if matched_old_cluster is None:

            migrated = pd.NA

            migration_status = "unmatched_new_cluster"

        else:

            migrated = (
                old_cluster != matched_old_cluster
            )

            migration_status = (
                "migrated"
                if migrated
                else "stable"
            )

        records.append({

            "snapshot_from": previous_record["snapshot_date"],

            "snapshot_to": current_record["snapshot_date"],

            "customer_id": customer_id,

            "old_cluster": old_cluster,

            "new_cluster": new_cluster,

            "matched_old_cluster": matched_old_cluster,

            "migrated": migrated,

            "migration_status": migration_status

        })

    migrations_df = pd.DataFrame(records)

    if not migrations_df.empty:

        migrations_df["migrated"] = (
            migrations_df["migrated"].astype("boolean")
        )

    return migrations_df, alignment


# ------------------------------------------------------------
# 9. ANALYZE TWO CONSECUTIVE SNAPSHOTS
# ------------------------------------------------------------

def analyze_segment_transition(
    previous_record,
    current_record,
    min_overlap=None,
    coverage_threshold=None,
    jaccard_threshold=None
):
    """
    Detect segment evolution between two consecutive snapshots.
    """

    if min_overlap is None:
        min_overlap = EVOLUTION_MIN_OVERLAP

    if coverage_threshold is None:
        coverage_threshold = EVOLUTION_COVERAGE_THRESHOLD

    if jaccard_threshold is None:
        jaccard_threshold = EVOLUTION_JACCARD_THRESHOLD

    old_date = previous_record["snapshot_date"]

    new_date = current_record["snapshot_date"]

    # --------------------------------------------------------
    # Build memberships
    # --------------------------------------------------------

    old_membership = build_cluster_memberships(
        previous_record["rfm"],
        previous_record["labels"]
    )

    new_membership = build_cluster_memberships(
        current_record["rfm"],
        current_record["labels"]
    )

    old_clusters = list(old_membership.keys())

    new_clusters = list(new_membership.keys())

    # --------------------------------------------------------
    # Calculate overlap graph
    # --------------------------------------------------------

    adjacency_old, adjacency_new, overlap_df = (
        build_evolution_graph(
            old_membership,
            new_membership,
            min_overlap,
            coverage_threshold,
            jaccard_threshold
        )
    )

    components = find_evolution_components(
        old_clusters,
        new_clusters,
        adjacency_old,
        adjacency_new
    )

    # --------------------------------------------------------
    # Classify events
    # --------------------------------------------------------

    events = []

    for component_old, component_new in components:

        old_count = len(component_old)

        new_count = len(component_new)

        event_type = classify_evolution_event(
            old_count,
            new_count
        )

        old_customers = set()

        for cluster_id in component_old:

            old_customers.update(
                old_membership[cluster_id]
            )

        new_customers = set()

        for cluster_id in component_new:

            new_customers.update(
                new_membership[cluster_id]
            )

        shared_customers = (
            old_customers.intersection(new_customers)
        )

        union_customers = (
            old_customers.union(new_customers)
        )

        component_jaccard = (
            len(shared_customers) / len(union_customers)
            if union_customers else 0.0
        )

        old_coverage = (
            len(shared_customers) / len(old_customers)
            if old_customers else 0.0
        )

        new_coverage = (
            len(shared_customers) / len(new_customers)
            if new_customers else 0.0
        )

        # This is an overlap-strength measure, NOT a
        # statistically calibrated event probability.
        overlap_strength = (
            0.5 * component_jaccard
            + 0.25 * old_coverage
            + 0.25 * new_coverage
        )

        events.append({

            "snapshot_from": old_date,

            "snapshot_to": new_date,

            "event_type": event_type,

            "old_clusters": sorted(component_old),

            "new_clusters": sorted(component_new),

            "n_old_clusters": old_count,

            "n_new_clusters": new_count,

            "old_customers": len(old_customers),

            "new_customers": len(new_customers),

            "shared_customers": len(shared_customers),

            "component_jaccard": component_jaccard,

            "old_coverage": old_coverage,

            "new_coverage": new_coverage,

            "overlap_strength": overlap_strength

        })

    events_df = pd.DataFrame(events)

    # --------------------------------------------------------
    # Customer migration
    # --------------------------------------------------------

    migrations_df, alignment = analyze_customer_migrations(
        previous_record,
        current_record
    )

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    if not migrations_df.empty:

        evaluable = migrations_df[
            migrations_df["migrated"].notna()
        ]

        migration_rate = (
            evaluable["migrated"].astype(bool).mean()
            if not evaluable.empty else np.nan
        )

        migrated_count = (
            int(evaluable["migrated"].sum())
            if not evaluable.empty else 0
        )

        unmatched_count = int(
            (
                migrations_df["migration_status"]
                == "unmatched_new_cluster"
            ).sum()
        )

    else:

        migration_rate = np.nan

        migrated_count = 0

        unmatched_count = 0

    event_counts = (
        events_df["event_type"].value_counts().to_dict()
        if not events_df.empty else {}
    )

    summary = {

        "snapshot_from": old_date,

        "snapshot_to": new_date,

        "old_cluster_count": len(old_clusters),

        "new_cluster_count": len(new_clusters),

        "common_customers": len(
            previous_record["rfm"].index.intersection(
                current_record["rfm"].index
            )
        ),

        "migrated_customers": migrated_count,

        "unmatched_new_cluster_customers": unmatched_count,

        "migration_evaluable_customers": (
            len(migrations_df) - unmatched_count
        ),

        "migration_rate": migration_rate,

        "alignment_coverage": (
            migrations_df["matched_old_cluster"].notna().sum()
            / len(migrations_df)
            if len(migrations_df) > 0 else np.nan
        ),

        "birth_events": event_counts.get("Birth", 0),

        "death_events": event_counts.get("Death", 0),

        "split_events": event_counts.get("Split", 0),

        "merge_events": event_counts.get("Merge", 0),

        "continuation_events": event_counts.get(
            "Continuation", 0
        ),

        "reorganization_events": event_counts.get(
            "Reorganization", 0
        )

    }

    return {

        "events": events_df,

        "migrations": migrations_df,

        "overlaps": overlap_df,

        "summary": summary,

        "alignment": alignment

    }


# ------------------------------------------------------------
# 10. RUN EVOLUTION OVER ALL SNAPSHOTS
# ------------------------------------------------------------

def run_segment_evolution(adaptive_history):

    if len(adaptive_history) < 2:

        print(
            "At least two valid snapshots are required."
        )

        return (
            [],
            pd.DataFrame(),
            pd.DataFrame(),
            pd.DataFrame()
        )

    evolution_history = []

    all_events = []

    all_migrations = []

    all_summaries = []

    for i in range(1, len(adaptive_history)):

        previous_record = adaptive_history[i - 1]

        current_record = adaptive_history[i]

        result = analyze_segment_transition(
            previous_record,
            current_record
        )

        evolution_history.append(result)

        if not result["events"].empty:
            all_events.append(result["events"])

        if not result["migrations"].empty:
            all_migrations.append(result["migrations"])

        all_summaries.append(
            result["summary"]
        )

        summary = result["summary"]

        migration_text = (
            f"{summary['migration_rate']:.2%}"
            if pd.notna(summary["migration_rate"])
            else "N/A"
        )

        print(
            f"Transition {i}/{len(adaptive_history)-1} | "
            f"{pd.Timestamp(summary['snapshot_from']).date()} -> "
            f"{pd.Timestamp(summary['snapshot_to']).date()} | "
            f"Migration: {migration_text} | "
            f"Split: {summary['split_events']} | "
            f"Merge: {summary['merge_events']} | "
            f"Birth: {summary['birth_events']} | "
            f"Death: {summary['death_events']}"
        )

    events_df = (
        pd.concat(all_events, ignore_index=True)
        if all_events else pd.DataFrame()
    )

    migrations_df = (
        pd.concat(all_migrations, ignore_index=True)
        if all_migrations else pd.DataFrame()
    )

    summaries_df = pd.DataFrame(
        all_summaries
    )

    return (
        evolution_history,
        events_df,
        migrations_df,
        summaries_df
    )


# ============================================================
# 11. EXECUTE
# ============================================================

(
    evolution_history,
    evolution_events,
    customer_migrations,
    evolution_summary
) = run_segment_evolution(
    adaptive_history
)


# ============================================================
# 12. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 75)
print("SEGMENT EVOLUTION SUMMARY")
print("=" * 75)

if not evolution_summary.empty:

    display(
        evolution_summary.round(4)
    )

print("\n" + "=" * 75)
print("EVOLUTION EVENT COUNTS")
print("=" * 75)

if not evolution_events.empty:

    display(
        evolution_events[
            "event_type"
        ].value_counts().rename_axis(
            "event_type"
        ).reset_index(
            name="count"
        )
    )

    print("\nDetailed Evolution Events:")

    display(
        evolution_events.head(20)
    )

print("\n" + "=" * 75)
print("CUSTOMER MIGRATION SAMPLE")
print("=" * 75)

if not customer_migrations.empty:

    display(
        customer_migrations.head(20)
    )

    evaluable_migrations = customer_migrations[
        customer_migrations["migrated"].notna()
    ]

    if not evaluable_migrations.empty:

        print(
            "\nAligned customer migration rate:",
            f"{evaluable_migrations['migrated'].astype(bool).mean():.2%}"
        )

    print(
        "Customers in unmatched new clusters:",
        int(
            (
                customer_migrations["migration_status"]
                == "unmatched_new_cluster"
            ).sum()
        )
    )

else:

    print(
        "No common customers available for migration analysis."
    )


Cell 12 — Temporal Stability Evaluation

Báo cáo ARI, NMI, migration, alignment coverage, tỷ lệ common cohort và rolling mean 3 kỳ; đồng thời kiểm tra khoảng cách snapshot.


In [ ]:
# ============================================================
# CELL 12 - TEMPORAL STABILITY EVALUATION
# ARI + NMI + customer migration + cohort-overlap diagnostics
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment


def get_common_customer_labels(previous_record, current_record):
    old_rfm = previous_record["rfm"]
    new_rfm = current_record["rfm"]
    old_labels = pd.Series(np.asarray(previous_record["labels"]), index=old_rfm.index)
    new_labels = pd.Series(np.asarray(current_record["labels"]), index=new_rfm.index)
    common_customers = old_labels.index.intersection(new_labels.index)
    return (
        old_labels.loc[common_customers].to_numpy(),
        new_labels.loc[common_customers].to_numpy(),
        common_customers
    )


def align_cluster_labels_hungarian(old_labels, new_labels):
    old_labels = np.asarray(old_labels)
    new_labels = np.asarray(new_labels)
    if len(old_labels) != len(new_labels):
        raise ValueError("Old and new labels must have equal lengths.")
    if len(old_labels) == 0:
        return {
            "mapping": {},
            "contingency": np.zeros((0, 0), dtype=int),
            "old_cluster_ids": np.array([]),
            "new_cluster_ids": np.array([]),
            "matched_overlap": 0,
            "matched_pairs": []
        }

    old_ids = np.unique(old_labels)
    new_ids = np.unique(new_labels)
    contingency = np.zeros((len(old_ids), len(new_ids)), dtype=int)
    for i, old_id in enumerate(old_ids):
        for j, new_id in enumerate(new_ids):
            contingency[i, j] = np.sum((old_labels == old_id) & (new_labels == new_id))

    row_ind, col_ind = linear_sum_assignment(-contingency)
    mapping = {}
    matched_pairs = []
    matched_overlap = 0
    for row, col in zip(row_ind, col_ind):
        overlap = int(contingency[row, col])
        if overlap <= 0:
            continue
        old_id = int(old_ids[row])
        new_id = int(new_ids[col])
        mapping[new_id] = old_id
        matched_overlap += overlap
        matched_pairs.append({
            "old_cluster": old_id,
            "new_cluster": new_id,
            "overlap": overlap
        })
    return {
        "mapping": mapping,
        "contingency": contingency,
        "old_cluster_ids": old_ids,
        "new_cluster_ids": new_ids,
        "matched_overlap": matched_overlap,
        "matched_pairs": matched_pairs
    }


def evaluate_temporal_stability(
    previous_record,
    current_record,
    min_common_customers=10
):
    old_y, new_y, common_customers = get_common_customer_labels(
        previous_record,
        current_record
    )
    n_common = int(len(common_customers))
    old_n = int(len(previous_record["rfm"]))
    new_n = int(len(current_record["rfm"]))
    smaller_n = max(1, min(old_n, new_n))
    date_from = pd.Timestamp(previous_record["snapshot_date"])
    date_to = pd.Timestamp(current_record["snapshot_date"])
    gap_days = int((date_to.normalize() - date_from.normalize()).days)

    result = {
        "snapshot_from": date_from,
        "snapshot_to": date_to,
        "snapshot_gap_days": gap_days,
        "previous_customers": old_n,
        "current_customers": new_n,
        "common_customers": n_common,
        "common_fraction_smaller_cohort": n_common / smaller_n,
        "ari": np.nan,
        "nmi": np.nan,
        "migration_rate": np.nan,
        "stable_customer_rate": np.nan,
        "stable_customer_rate_all_common": np.nan,
        "alignment_coverage": np.nan,
        "matched_customers": 0,
        "unmatched_customers": 0,
        "migrated_customers": 0,
        "stable_customers": 0,
        "status": None
    }

    if n_common < min_common_customers:
        result["status"] = "insufficient_overlap"
        return result

    result["ari"] = float(adjusted_rand_score(old_y, new_y))
    result["nmi"] = float(normalized_mutual_info_score(
        old_y, new_y, average_method="arithmetic"
    ))

    alignment = align_cluster_labels_hungarian(old_y, new_y)
    mapping = alignment["mapping"]
    aligned_new = np.array([mapping.get(int(label), -1) for label in new_y])
    matched_mask = aligned_new != -1
    matched_count = int(matched_mask.sum())
    unmatched_count = int((~matched_mask).sum())

    if matched_count > 0:
        migration_mask = old_y[matched_mask] != aligned_new[matched_mask]
        migrated_count = int(migration_mask.sum())
        stable_count = matched_count - migrated_count
        migration_rate = migrated_count / matched_count
        stable_rate = stable_count / matched_count
        stable_rate_all_common = stable_count / n_common
    else:
        migrated_count = 0
        stable_count = 0
        migration_rate = np.nan
        stable_rate = np.nan
        stable_rate_all_common = 0.0

    result.update({
        "migration_rate": float(migration_rate) if pd.notna(migration_rate) else np.nan,
        "stable_customer_rate": float(stable_rate) if pd.notna(stable_rate) else np.nan,
        "stable_customer_rate_all_common": float(stable_rate_all_common),
        "alignment_coverage": matched_count / n_common,
        "matched_customers": matched_count,
        "unmatched_customers": unmatched_count,
        "migrated_customers": migrated_count,
        "stable_customers": stable_count,
        "status": "valid" if gap_days == SNAPSHOT_STRIDE else "nonstandard_snapshot_gap"
    })
    return result


def run_temporal_stability_evaluation(adaptive_history, min_common_customers=10):
    stability_history = []
    if len(adaptive_history) < 2:
        print("At least two snapshots are required for temporal stability evaluation.")
        return stability_history, pd.DataFrame()

    for i in range(1, len(adaptive_history)):
        result = evaluate_temporal_stability(
            adaptive_history[i - 1],
            adaptive_history[i],
            min_common_customers=min_common_customers
        )
        stability_history.append(result)
        print(
            f"Transition {i}/{len(adaptive_history)-1} | "
            f"gap={result['snapshot_gap_days']}d | "
            f"common={result['common_customers']:,} "
            f"({result['common_fraction_smaller_cohort']:.1%}) | "
            f"ARI={result['ari'] if pd.notna(result['ari']) else 'N/A'} | "
            f"NMI={result['nmi'] if pd.notna(result['nmi']) else 'N/A'} | "
            f"migration={result['migration_rate'] if pd.notna(result['migration_rate']) else 'N/A'} | "
            f"coverage={result['alignment_coverage'] if pd.notna(result['alignment_coverage']) else 'N/A'} | "
            f"status={result['status']}"
        )

    stability_df = pd.DataFrame(stability_history)
    if not stability_df.empty:
        for metric in ["ari", "nmi", "migration_rate", "stable_customer_rate", "alignment_coverage"]:
            stability_df[f"{metric}_rolling_3"] = (
                stability_df[metric].rolling(window=3, min_periods=1).mean()
            )

    return stability_history, stability_df


# Execute
stability_history, stability_df = run_temporal_stability_evaluation(
    adaptive_history,
    min_common_customers=10
)

print("\n" + "=" * 75)
print("TEMPORAL STABILITY EVALUATION")
print("=" * 75)
if not stability_df.empty:
    display(stability_df.round(4))
    summary_columns = [
        "ari", "nmi", "migration_rate", "stable_customer_rate",
        "stable_customer_rate_all_common", "alignment_coverage",
        "common_fraction_smaller_cohort", "snapshot_gap_days"
    ]
    display(stability_df[summary_columns].mean(numeric_only=True).to_frame("mean").round(4))
    display(stability_df["status"].value_counts().to_frame("transition_count"))
else:
    print("No temporal stability results available.")

# ------------------------------------------------------------
# Separate lifecycle activity transitions (fixed 28-day horizon).
# This is intentionally distinct from cluster migration among active customers.
# ------------------------------------------------------------
def evaluate_activity_state_transitions(data, dates, activity_window_days=ACTIVITY_STATE_WINDOW_DAYS):
    rows = []
    previous_status = None
    previous_known_ids = set()
    for snapshot in dates:
        snapshot = pd.Timestamp(snapshot)
        snapshot_end = snapshot.normalize() + pd.Timedelta(days=1)
        historical = data.loc[data["InvoiceDate"] < snapshot_end, ["CustomerID"]]
        known_ids = set(historical["CustomerID"].astype(int).unique().tolist())
        active_rfm = calculate_rfm(data, snapshot, window_days=int(activity_window_days))
        active_ids = set(active_rfm.index.astype(int).tolist())
        status = pd.Series("dormant", index=pd.Index(sorted(known_ids), name="CustomerID"), dtype="object")
        if active_ids:
            status.loc[status.index.intersection(pd.Index(sorted(active_ids)))] = "active"

        if previous_status is not None:
            common_known = previous_status.index.intersection(status.index)
            old_state = previous_status.loc[common_known]
            new_state = status.loc[common_known]
            new_customer_count = int(len(known_ids - previous_known_ids))
            counts = {
                "active_to_active": int(((old_state == "active") & (new_state == "active")).sum()),
                "active_to_dormant": int(((old_state == "active") & (new_state == "dormant")).sum()),
                "dormant_to_active": int(((old_state == "dormant") & (new_state == "active")).sum()),
                "dormant_to_dormant": int(((old_state == "dormant") & (new_state == "dormant")).sum())
            }
            denominator = max(1, len(common_known))
            rows.append({
                "snapshot_from": pd.Timestamp(previous_snapshot),
                "snapshot_to": snapshot,
                "activity_window_days": int(activity_window_days),
                "common_known_customers": int(len(common_known)),
                "new_customers_seen_since_previous_snapshot": new_customer_count,
                **counts,
                "became_dormant_count": counts["active_to_dormant"],
                "reactivated_count": counts["dormant_to_active"],
                "active_retention_rate": counts["active_to_active"] / max(1, counts["active_to_active"] + counts["active_to_dormant"]),
                "reactivation_rate_among_dormant": counts["dormant_to_active"] / max(1, counts["dormant_to_active"] + counts["dormant_to_dormant"]),
                "active_fraction_of_known_customers": float(len(active_ids) / max(1, len(known_ids)))
            })
        previous_status = status
        previous_known_ids = known_ids
        previous_snapshot = snapshot
    return pd.DataFrame(rows)

activity_state_transition_df = evaluate_activity_state_transitions(
    df,
    snapshot_dates,
    activity_window_days=ACTIVITY_STATE_WINDOW_DAYS
)
print("Fixed-horizon customer activity transitions (separate from segment migration):")
display(activity_state_transition_df.round(4))
if not activity_state_transition_df.empty:
    activity_transition_summary = activity_state_transition_df[
        ["active_to_active", "active_to_dormant", "dormant_to_active", "dormant_to_dormant", "new_customers_seen_since_previous_snapshot"]
    ].agg(["mean", "std"]).T
    display(activity_transition_summary.round(2))
    fig, ax = plt.subplots(figsize=(12, 5))
    activity_state_transition_df.set_index("snapshot_to")[[
        "active_to_active", "active_to_dormant", "dormant_to_active", "dormant_to_dormant"
    ]].plot(kind="bar", stacked=True, ax=ax)
    ax.set_title(f"Customer activity-state transitions ({ACTIVITY_STATE_WINDOW_DAYS}-day fixed horizon)")
    ax.set_xlabel("Snapshot transition ending")
    ax.set_ylabel("Number of previously known customers")
    ax.legend(title="Transition", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()
    plt.close(fig)



# ------------------------------------------------------------
# Behavioural meaning of the discovered clusters (descriptive)
# ------------------------------------------------------------
# Profiles are computed from raw RFM values, not standardized features.
# Kruskal-Wallis and epsilon-squared are descriptive within each snapshot;
# snapshots are not treated as independent replications for pooled inference.
from scipy.stats import kruskal
cluster_business_profile_rows = []
cluster_separation_rows = []
for record in adaptive_history:
    snap = pd.Timestamp(record["snapshot_date"])
    rfm_frame = record["rfm"].copy()
    labels = pd.Series(np.asarray(record["labels"]), index=rfm_frame.index, name="cluster")
    combined = rfm_frame.join(labels)
    total_monetary = float(combined["Monetary"].sum())
    for cluster_id, group in combined.groupby("cluster"):
        cluster_business_profile_rows.append({
            "snapshot_date": snap,
            "window_days": int(record["window"]),
            "k": int(record["k"]),
            "cluster": int(cluster_id),
            "customer_count": int(len(group)),
            "customer_fraction": float(len(group) / len(combined)),
            "median_recency_days": float(group["Recency"].median()),
            "median_frequency_invoices": float(group["Frequency"].median()),
            "median_monetary": float(group["Monetary"].median()),
            "mean_monetary": float(group["Monetary"].mean()),
            "gross_monetary_total": float(group["Monetary"].sum()),
            "gross_monetary_share": float(group["Monetary"].sum() / total_monetary) if total_monetary > 0 else np.nan
        })
    for feature in ["Recency", "Frequency", "Monetary"]:
        arrays = [g[feature].to_numpy(dtype=float) for _, g in combined.groupby("cluster")]
        arrays = [a for a in arrays if len(a) >= 2]
        if len(arrays) >= 2:
            try:
                stat, p_value = kruskal(*arrays)
                n = int(sum(len(a) for a in arrays))
                k_groups = int(len(arrays))
                epsilon_sq = max(0.0, float((stat - k_groups + 1) / max(1, n - k_groups)))
                cluster_separation_rows.append({
                    "snapshot_date": snap,
                    "feature": feature,
                    "n_customers": n,
                    "n_clusters_tested": k_groups,
                    "kruskal_h": float(stat),
                    "p_value_descriptive": float(p_value),
                    "epsilon_squared": epsilon_sq
                })
            except ValueError:
                pass

cluster_business_profiles_df = pd.DataFrame(cluster_business_profile_rows)
cluster_profile_separation_df = pd.DataFrame(cluster_separation_rows)
print("Raw-RFM behavioural profile by cluster (descriptive; snapshot-local labels):")
display(cluster_business_profiles_df.head(30).round(4))
print("Within-snapshot feature separation diagnostics (not pooled across time):")
display(cluster_profile_separation_df.round(6))


# Consistency check with Cell 11.
if (
    "evolution_summary" in globals()
    and not evolution_summary.empty
    and not stability_df.empty
):
    check_df = evolution_summary[
        ["snapshot_from", "snapshot_to", "migration_rate"]
    ].merge(
        stability_df[["snapshot_from", "snapshot_to", "migration_rate"]],
        on=["snapshot_from", "snapshot_to"],
        how="inner",
        suffixes=("_cell11", "_cell12")
    )
    if not check_df.empty:
        check_df["absolute_difference"] = (
            check_df["migration_rate_cell11"] - check_df["migration_rate_cell12"]
        ).abs()
        display(check_df.round(6))
        max_difference = check_df["absolute_difference"].dropna().max()
        if pd.isna(max_difference) or max_difference < 1e-8:
            print("PASS: Cell 11 and Cell 12 migration rates are consistent.")
        else:
            print("WARNING: Cell 11 and Cell 12 migration rates differ.")


Cell 13 — Drift Detection bằng Page-Hinkley

Tín hiệu thực tế dùng Rolling RFM 56 ngày cố định để tránh lẫn biến động do Adaptive Window. Detector hai chiều có re-baselining sau cảnh báo; kiểm thử synthetic gồm no-drift, upward, downward và up-then-return.


In [ ]:
# ============================================================
# CELL 13 - REGIME-AWARE PAGE-HINKLEY DRIFT DETECTION
# Fixed reference window + two-sided PH + synthetic validation
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PH_DELTA = 0.90
PH_THRESHOLD = 6.0  # initial value; recalibrated from null streams below
PH_SYNTHETIC_WARMUP = 30
PH_SYNTHETIC_LENGTH = 200
PH_DETECTION_TOLERANCE = 20
PH_RANDOM_SEED = RANDOM_STATE
PH_REBASELINE_WINDOW = 12
PH_SCALE_FLOOR_MULTIPLIER = 1.5
PH_PREWHITEN_AR1 = True
PH_AR1_COEFFICIENT_CLIP = 0.80
PH_ALARM_GAP = 30
PH_MIN_BASELINE_STD = 1e-8
PH_SIGNAL_WINDOW = 56
PH_MONTE_CARLO_TRIALS = 100
PH_CALIBRATION_TRIALS = 100
PH_THRESHOLD_GRID = [5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
PH_TARGET_FALSE_ALARM_RATE = 0.05
PH_CALIBRATION_SEED_OFFSET = 1_000_000
PH_TEST_SEED_OFFSET = 2_000_000


class PageHinkleyDetector:
    """Two-sided Page-Hinkley detector with post-alarm regime re-baselining.

    The stream is standardized using the current reference regime. After an
    alarm, a short re-baselining buffer estimates the new regime mean. This
    lets the detector identify both upward and downward changes, including a
    return from an elevated regime to the original level.
    """
    def __init__(
        self,
        delta=PH_DELTA,
        threshold=PH_THRESHOLD,
        direction="both",
        baseline_mean=None,
        baseline_std=None,
        rebaseline_window=PH_REBASELINE_WINDOW,
        scale_floor_multiplier=PH_SCALE_FLOOR_MULTIPLIER
    ):
        if delta < 0:
            raise ValueError("delta must be non-negative.")
        if threshold <= 0:
            raise ValueError("threshold must be positive.")
        if direction not in ["up", "down", "both"]:
            raise ValueError("direction must be up, down or both.")
        if baseline_mean is None or baseline_std is None:
            raise ValueError("A baseline mean and standard deviation are required.")
        if baseline_std <= 0:
            raise ValueError("baseline_std must be positive.")
        if rebaseline_window < 2:
            raise ValueError("rebaseline_window must be at least 2.")
        if scale_floor_multiplier < 1:
            raise ValueError("scale_floor_multiplier must be at least 1.")

        self.delta = float(delta)
        self.threshold = float(threshold)
        self.direction = direction
        self.baseline_mean = float(baseline_mean)
        self.baseline_std = float(baseline_std)
        self.scale_floor = float(baseline_std) * float(scale_floor_multiplier)
        self.rebaseline_window = int(rebaseline_window)
        self.cumulative_up = 0.0
        self.cumulative_down = 0.0
        self.rebaseline_buffer = None
        self.total_alarms = 0

    def reset_statistics(self):
        self.cumulative_up = 0.0
        self.cumulative_down = 0.0

    def update(self, value):
        value = float(value)
        if not np.isfinite(value):
            raise ValueError("Input value must be finite.")

        # After an alarm, use the next observations to estimate the new regime.
        if self.rebaseline_buffer is not None:
            self.rebaseline_buffer.append(value)
            if len(self.rebaseline_buffer) >= self.rebaseline_window:
                buffer = np.asarray(self.rebaseline_buffer, dtype=float)
                self.baseline_mean = float(np.mean(buffer))
                estimated_std = float(np.std(buffer, ddof=1))
                self.baseline_std = max(estimated_std, self.scale_floor, PH_MIN_BASELINE_STD)
                self.rebaseline_buffer = None
                self.reset_statistics()
                rebaseline_complete = True
            else:
                rebaseline_complete = False
            return {
                "alarm": False,
                "direction": None,
                "statistic_up": float(self.cumulative_up),
                "statistic_down": float(self.cumulative_down),
                "rebaselining": True,
                "rebaseline_complete": rebaseline_complete,
                "reference_mean": self.baseline_mean,
                "reference_std": self.baseline_std
            }

        z = (value - self.baseline_mean) / max(self.baseline_std, PH_MIN_BASELINE_STD)
        self.cumulative_up = max(0.0, self.cumulative_up + z - self.delta)
        self.cumulative_down = max(0.0, self.cumulative_down - z - self.delta)

        upward_alarm = self.cumulative_up > self.threshold
        downward_alarm = self.cumulative_down > self.threshold
        if self.direction == "up":
            alarm = upward_alarm
        elif self.direction == "down":
            alarm = downward_alarm
        else:
            alarm = upward_alarm or downward_alarm

        if upward_alarm and downward_alarm:
            detected_direction = "both"
        elif upward_alarm:
            detected_direction = "up"
        elif downward_alarm:
            detected_direction = "down"
        else:
            detected_direction = None

        result = {
            "alarm": bool(alarm),
            "direction": detected_direction,
            "statistic_up": float(self.cumulative_up),
            "statistic_down": float(self.cumulative_down),
            "rebaselining": False,
            "rebaseline_complete": False,
            "reference_mean": self.baseline_mean,
            "reference_std": self.baseline_std
        }

        if alarm:
            self.total_alarms += 1
            self.reset_statistics()
            self.rebaseline_buffer = []

        return result


def run_page_hinkley(
    values,
    dates=None,
    warmup=5,
    delta=PH_DELTA,
    threshold=PH_THRESHOLD,
    direction="both",
    rebaseline_window=PH_REBASELINE_WINDOW,
    scale_floor_multiplier=PH_SCALE_FLOOR_MULTIPLIER,
    prewhiten=PH_PREWHITEN_AR1,
    ar1_coefficient_clip=PH_AR1_COEFFICIENT_CLIP
):
    """Run Page-Hinkley, optionally prewhitening AR(1) serial dependence.

    The AR(1) coefficient is estimated using warmup observations only. This
    reduces false alarms caused by positive autocorrelation under a stationary
    null. The raw signal and detector signal are both retained in the output.
    """
    raw_values = np.asarray(values, dtype=float)
    if raw_values.ndim != 1:
        raise ValueError("Input values must be one-dimensional.")
    if len(raw_values) <= warmup:
        raise ValueError("Series length must be greater than warmup.")
    if not np.all(np.isfinite(raw_values)):
        raise ValueError("Input contains NaN or infinite values.")
    if dates is not None and len(dates) != len(raw_values):
        raise ValueError("Length of dates must equal length of values.")
    if warmup < 2:
        raise ValueError("warmup must be at least 2.")
    if not 0 < ar1_coefficient_clip < 1:
        raise ValueError("ar1_coefficient_clip must be in (0, 1).")

    rho = 0.0
    detector_values = raw_values.copy()
    if prewhiten and warmup >= 5:
        baseline_for_rho = raw_values[:warmup]
        centered = baseline_for_rho - float(np.mean(baseline_for_rho))
        denominator = float(np.dot(centered[:-1], centered[:-1]))
        if denominator > 1e-12:
            rho = float(np.dot(centered[1:], centered[:-1]) / denominator)
            rho = float(np.clip(rho, -ar1_coefficient_clip, ar1_coefficient_clip))
            detector_values[1:] = raw_values[1:] - rho * raw_values[:-1]

    baseline = detector_values[:warmup]
    baseline_mean = float(np.mean(baseline))
    baseline_std = float(np.std(baseline, ddof=1))
    baseline_std = max(baseline_std, PH_MIN_BASELINE_STD)

    detector = PageHinkleyDetector(
        delta=delta,
        threshold=threshold,
        direction=direction,
        baseline_mean=baseline_mean,
        baseline_std=baseline_std,
        rebaseline_window=rebaseline_window,
        scale_floor_multiplier=scale_floor_multiplier
    )

    records = []
    for i, detector_value in enumerate(detector_values):
        if i < warmup:
            result = {
                "alarm": False,
                "direction": None,
                "statistic_up": 0.0,
                "statistic_down": 0.0,
                "rebaselining": False,
                "rebaseline_complete": False,
                "reference_mean": baseline_mean,
                "reference_std": baseline_std
            }
        else:
            result = detector.update(detector_value)
        records.append({
            "index": i,
            "snapshot_date": dates[i] if dates is not None else i,
            "value": float(raw_values[i]),
            "detector_value": float(detector_value),
            "alarm": bool(result["alarm"]),
            "direction": result["direction"],
            "statistic_up": float(result["statistic_up"]),
            "statistic_down": float(result["statistic_down"]),
            "rebaselining": bool(result["rebaselining"]),
            "rebaseline_complete": bool(result["rebaseline_complete"]),
            "reference_mean": float(result["reference_mean"]),
            "reference_std": float(result["reference_std"]),
            "prewhiten_rho": float(rho)
        })

    result_df = pd.DataFrame(records)
    result_df["cooldown"] = result_df["rebaselining"]
    result_df.attrs["baseline_mean"] = baseline_mean
    result_df.attrs["baseline_std"] = baseline_std
    result_df.attrs["warmup"] = warmup
    result_df.attrs["rebaseline_window"] = rebaseline_window
    result_df.attrs["prewhiten_rho"] = rho
    result_df.attrs["prewhiten_enabled"] = bool(prewhiten)
    return result_df


def group_alarm_events(alarm_indices, max_gap=PH_ALARM_GAP):
    alarm_indices = sorted(set(int(x) for x in alarm_indices))
    if not alarm_indices:
        return []
    groups = [[alarm_indices[0]]]
    for index in alarm_indices[1:]:
        if index - groups[-1][-1] <= max_gap:
            groups[-1].append(index)
        else:
            groups.append([index])
    return [group[0] for group in groups]


def extract_monetary_signal(data, dates, reference_window_days=PH_SIGNAL_WINDOW):
    """Build a comparable signal from a fixed RFM window, not the adaptive one."""
    records = []
    for snapshot in dates:
        rfm = calculate_rfm(data, snapshot, window_days=reference_window_days)
        if rfm is None or rfm.empty or "Monetary" not in rfm.columns:
            continue
        monetary = pd.to_numeric(rfm["Monetary"], errors="coerce")
        monetary = monetary[np.isfinite(monetary) & (monetary >= 0)]
        if monetary.empty:
            continue
        median_monetary = float(monetary.median())
        records.append({
            "snapshot_date": pd.Timestamp(snapshot),
            "reference_window_days": int(reference_window_days),
            "customer_count": int(len(rfm)),
            "median_monetary": median_monetary,
            "log_median_monetary": float(np.log1p(median_monetary))
        })
    return pd.DataFrame(records)


def evaluate_drift_events(
    detected_indices,
    true_drift_points,
    tolerance=PH_DETECTION_TOLERANCE,
    series_length=None,
    warmup=0
):
    detected_indices = sorted(set(int(x) for x in detected_indices))
    true_drift_points = sorted(set(int(x) for x in true_drift_points))
    matched_detections = set()
    matched_true_events = set()
    delays = []

    for event_index, true_point in enumerate(true_drift_points):
        candidates = [
            detection for detection in detected_indices
            if true_point <= detection <= true_point + tolerance
            and detection not in matched_detections
        ]
        if candidates:
            detection = min(candidates)
            matched_detections.add(detection)
            matched_true_events.add(event_index)
            delays.append(detection - true_point)

    tp = len(matched_true_events)
    fp = len(detected_indices) - len(matched_detections)
    fn = len(true_drift_points) - tp
    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    monitored_points = max(1, (series_length - warmup) if series_length is not None else 1)

    return {
        "true_drift_events": len(true_drift_points),
        "detected_events": len(detected_indices),
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "mean_detection_delay": float(np.mean(delays)) if delays else np.nan,
        "false_alarms_per_100_points": (fp / monitored_points) * 100.0
    }


def generate_synthetic_drift_data(
    length=PH_SYNTHETIC_LENGTH,
    warmup=PH_SYNTHETIC_WARMUP,
    drift_points=(80, 140),
    drift_levels=(1.2, 0.0),
    noise_std=0.20,
    random_state=PH_RANDOM_SEED,
    noise_autocorrelation=0.0,
    drift_mode="step",
    ramp_length=30
):
    """Generate synthetic mean-drift streams with optional AR(1) noise.

    `drift_levels` are absolute target means. `drift_mode='gradual'` ramps
    linearly from the prior regime to the target over `ramp_length` points.
    """
    if length <= warmup:
        raise ValueError("length must exceed warmup.")
    if len(drift_points) != len(drift_levels):
        raise ValueError("drift_points and drift_levels must have equal lengths.")
    if noise_std <= 0:
        raise ValueError("noise_std must be positive.")
    if not 0.0 <= noise_autocorrelation < 1.0:
        raise ValueError("noise_autocorrelation must be in [0, 1).")
    if drift_mode not in {"step", "gradual"}:
        raise ValueError("drift_mode must be 'step' or 'gradual'.")
    if ramp_length < 2:
        raise ValueError("ramp_length must be at least 2.")

    rng = np.random.default_rng(random_state)
    noise = rng.normal(0.0, 1.0, size=length)
    rho = float(noise_autocorrelation)
    if rho > 0:
        innovation_scale = np.sqrt(1.0 - rho ** 2)
        for index in range(1, length):
            noise[index] = rho * noise[index - 1] + innovation_scale * noise[index]
        noise = (noise - noise.mean()) / max(noise.std(ddof=1), 1e-12)
    values = noise * float(noise_std)

    signal = np.zeros(length, dtype=float)
    current_level = 0.0
    sorted_events = sorted(zip(drift_points, drift_levels), key=lambda pair: pair[0])
    true_drift_points = []
    for drift_point, target_level in sorted_events:
        drift_point = int(drift_point)
        target_level = float(target_level)
        if not (warmup <= drift_point < length):
            raise ValueError("Drift points must be within monitoring range.")
        true_drift_points.append(drift_point)
        if drift_mode == "step":
            signal[drift_point:] = target_level
        else:
            start_level = current_level
            ramp_end = min(length, drift_point + int(ramp_length))
            ramp_values = np.linspace(
                start_level, target_level,
                num=max(1, ramp_end - drift_point),
                endpoint=False
            )
            signal[drift_point:ramp_end] = ramp_values
            signal[ramp_end:] = target_level
        current_level = target_level

    values = values + signal
    true_labels = np.zeros(length, dtype=int)
    true_labels[true_drift_points] = 1
    return values, true_labels, true_drift_points


def wilson_interval(successes, trials, z=1.96):
    """Wilson score interval for a binomial proportion."""
    n = int(trials)
    if n <= 0:
        return (np.nan, np.nan)
    p = float(successes) / n
    denominator = 1.0 + (z ** 2) / n
    center = (p + (z ** 2) / (2.0 * n)) / denominator
    radius = (
        z * np.sqrt((p * (1.0 - p) / n) + (z ** 2) / (4.0 * n ** 2))
        / denominator
    )
    return max(0.0, center - radius), min(1.0, center + radius)


def bootstrap_mean_interval(values, n_boot=2000, random_state=RANDOM_STATE):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return (np.nan, np.nan)
    if len(values) == 1:
        return (float(values[0]), float(values[0]))
    rng = np.random.default_rng(random_state)
    samples = rng.choice(values, size=(n_boot, len(values)), replace=True)
    means = np.mean(samples, axis=1)
    return tuple(np.quantile(means, [0.025, 0.975]).astype(float))


def calibrate_page_hinkley_threshold(
    threshold_grid=PH_THRESHOLD_GRID,
    n_trials=PH_CALIBRATION_TRIALS,
    target_false_alarm_rate=PH_TARGET_FALSE_ALARM_RATE
):
    """Choose the smallest PH threshold meeting a predeclared null FAR target.

    Calibration uses independent stationary IID and AR(1) null streams. Drift
    detection performance is then reported on separate test seeds in
    run_drift_monte_carlo; no drift labels are used to tune the threshold.
    """
    rows = []
    for threshold in sorted(float(x) for x in threshold_grid):
        false_alarm_count = 0
        total = 0
        for null_type, rho in [("iid", 0.0), ("ar1", 0.35)]:
            for trial in range(int(n_trials)):
                seed = PH_RANDOM_SEED + PH_CALIBRATION_SEED_OFFSET + (0 if null_type == "iid" else 500_000) + trial
                rng = np.random.default_rng(seed + 77)
                noise_std = float(rng.uniform(0.16, 0.32))
                values, _, _ = generate_synthetic_drift_data(
                    length=PH_SYNTHETIC_LENGTH,
                    warmup=PH_SYNTHETIC_WARMUP,
                    drift_points=(),
                    drift_levels=(),
                    noise_std=noise_std,
                    random_state=seed,
                    noise_autocorrelation=rho,
                    drift_mode="step"
                )
                result = run_page_hinkley(
                    values=values,
                    warmup=PH_SYNTHETIC_WARMUP,
                    delta=PH_DELTA,
                    threshold=threshold,
                    direction="both",
                    rebaseline_window=PH_REBASELINE_WINDOW,
                    scale_floor_multiplier=PH_SCALE_FLOOR_MULTIPLIER
                )
                alarm_indices = result.loc[result["alarm"], "index"].tolist()
                events = group_alarm_events(alarm_indices, max_gap=PH_ALARM_GAP)
                false_alarm_count += int(len(events) > 0)
                total += 1
        far = false_alarm_count / max(total, 1)
        ci_low, ci_high = wilson_interval(false_alarm_count, total)
        rows.append({
            "threshold": threshold,
            "delta": float(PH_DELTA),
            "null_trials": int(total),
            "false_alarm_trials": int(false_alarm_count),
            "false_alarm_rate": float(far),
            "false_alarm_ci_low": ci_low,
            "false_alarm_ci_high": ci_high,
            "target_false_alarm_rate": float(target_false_alarm_rate)
        })

    table = pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True)
    eligible = table[table["false_alarm_rate"] <= target_false_alarm_rate]
    if not eligible.empty:
        selected_threshold = float(eligible.iloc[0]["threshold"])
        selection_status = "lowest_threshold_meeting_empirical_null_FAR_target"
    else:
        # Transparent fallback: choose the most conservative tested threshold
        # with the lowest observed null FAR; do not claim target attainment.
        best = table.sort_values(
            ["false_alarm_rate", "threshold"], ascending=[True, False]
        ).iloc[0]
        selected_threshold = float(best["threshold"])
        selection_status = "target_not_met_used_lowest_observed_null_FAR"
    table["selected"] = np.isclose(table["threshold"], selected_threshold)
    table["selection_status"] = selection_status
    return table, selected_threshold, selection_status


def run_drift_monte_carlo(
    n_trials=PH_MONTE_CARLO_TRIALS,
    length=PH_SYNTHETIC_LENGTH,
    warmup=PH_SYNTHETIC_WARMUP,
    delta=PH_DELTA,
    threshold=PH_THRESHOLD,
    seed_offset=PH_TEST_SEED_OFFSET
):
    """Held-out Monte Carlo evaluation across null, abrupt and gradual shifts."""
    scenarios = [
        {"scenario": "no_drift_iid", "points": (), "mode": "step", "rho": 0.0,
         "shift_range": (0.0, 0.0), "tolerance": PH_DETECTION_TOLERANCE},
        {"scenario": "no_drift_ar1", "points": (), "mode": "step", "rho": 0.35,
         "shift_range": (0.0, 0.0), "tolerance": PH_DETECTION_TOLERANCE},
        {"scenario": "upward_shift", "points": (80,), "mode": "step", "rho": 0.10,
         "shift_range": (0.55, 1.00), "tolerance": PH_DETECTION_TOLERANCE},
        {"scenario": "downward_shift", "points": (80,), "mode": "step", "rho": 0.10,
         "shift_range": (0.55, 1.00), "tolerance": PH_DETECTION_TOLERANCE},
        {"scenario": "weak_upward_shift", "points": (80,), "mode": "step", "rho": 0.10,
         "shift_range": (0.30, 0.55), "tolerance": PH_DETECTION_TOLERANCE},
        {"scenario": "gradual_upward_shift", "points": (80,), "mode": "gradual", "rho": 0.10,
         "shift_range": (0.60, 1.00), "tolerance": 35},
        {"scenario": "up_then_return", "points": (80, 140), "mode": "step", "rho": 0.10,
         "shift_range": (0.65, 1.00), "tolerance": PH_DETECTION_TOLERANCE}
    ]

    trial_results = []
    for scenario_index, scenario in enumerate(scenarios):
        for trial in range(int(n_trials)):
            seed = int(PH_RANDOM_SEED + seed_offset + scenario_index * 100_000 + trial)
            rng = np.random.default_rng(seed + 91)
            noise_std = float(rng.uniform(0.16, 0.32))
            if scenario["scenario"] == "up_then_return":
                shift = float(rng.uniform(*scenario["shift_range"]))
                levels = (shift, 0.0)
            elif scenario["scenario"] == "upward_shift" or scenario["scenario"] in {"weak_upward_shift", "gradual_upward_shift"}:
                shift = float(rng.uniform(*scenario["shift_range"]))
                levels = (shift,)
            elif scenario["scenario"] == "downward_shift":
                shift = float(rng.uniform(*scenario["shift_range"]))
                levels = (-shift,)
            else:
                shift = 0.0
                levels = ()

            values, _, true_points = generate_synthetic_drift_data(
                length=length,
                warmup=warmup,
                drift_points=scenario["points"],
                drift_levels=levels,
                noise_std=noise_std,
                random_state=seed,
                noise_autocorrelation=float(scenario["rho"]),
                drift_mode=scenario["mode"],
                ramp_length=30
            )
            detector_result = run_page_hinkley(
                values=values,
                warmup=warmup,
                delta=delta,
                threshold=threshold,
                direction="both",
                rebaseline_window=PH_REBASELINE_WINDOW,
                scale_floor_multiplier=PH_SCALE_FLOOR_MULTIPLIER
            )
            raw_detections = detector_result.loc[detector_result["alarm"], "index"].tolist()
            detections = group_alarm_events(raw_detections, max_gap=PH_ALARM_GAP)
            metrics = evaluate_drift_events(
                detected_indices=detections,
                true_drift_points=true_points,
                tolerance=int(scenario["tolerance"]),
                series_length=length,
                warmup=warmup
            )
            metrics.update({
                "scenario": scenario["scenario"],
                "trial": trial + 1,
                "noise_std": noise_std,
                "shift_magnitude": shift,
                "noise_autocorrelation": float(scenario["rho"]),
                "drift_mode": scenario["mode"],
                "raw_alarm_count": len(raw_detections),
                "seed": seed,
                "false_alarm_trial": int(metrics["false_positive"] > 0)
            })
            trial_results.append(metrics)
    return pd.DataFrame(trial_results)


# Calibrate on null streams only; the held-out Monte Carlo test uses separate seeds.
ph_threshold_calibration, calibrated_threshold, ph_calibration_status = (
    calibrate_page_hinkley_threshold()
)
PH_THRESHOLD = float(calibrated_threshold)
print("Page-Hinkley calibrated threshold:", PH_THRESHOLD)
print("Calibration status:", ph_calibration_status)
display(ph_threshold_calibration.round(4))


# ============================================================
# REAL-DATA MONITORING (fixed 56-day window for comparability)
# ============================================================
print("=" * 75)
print("REAL DATA DRIFT MONITORING")
print("=" * 75)
real_drift_data = extract_monetary_signal(
    df,
    snapshot_dates,
    reference_window_days=PH_SIGNAL_WINDOW
)
real_drift_result = pd.DataFrame()
real_alarm_events = []

if len(real_drift_data) > 5:
    real_warmup = min(max(5, len(real_drift_data) // 3), len(real_drift_data) - 1)
    # The real series is much shorter than the synthetic stream (usually ~24 points).
    # Use a short regime buffer so a second change can still be observed.
    real_rebaseline_window = max(2, min(
        PH_REBASELINE_WINDOW,
        int(np.ceil(len(real_drift_data) * 0.15)),
        max(2, len(real_drift_data) - real_warmup - 1)
    ))
    real_drift_result = run_page_hinkley(
        values=real_drift_data["log_median_monetary"].values,
        dates=real_drift_data["snapshot_date"].tolist(),
        warmup=real_warmup,
        delta=PH_DELTA,
        threshold=PH_THRESHOLD,
        direction="both",
        rebaseline_window=real_rebaseline_window,
        scale_floor_multiplier=PH_SCALE_FLOOR_MULTIPLIER
    )
    real_drift_data = real_drift_data.merge(
        real_drift_result[[
            "snapshot_date", "alarm", "direction", "statistic_up",
            "statistic_down", "rebaselining", "reference_mean", "reference_std",
            "detector_value", "prewhiten_rho"
        ]],
        on="snapshot_date",
        how="left",
        validate="one_to_one"
    )
    real_drift_data["alarm"] = real_drift_data["alarm"].fillna(False)
    raw_real_alarm_indices = real_drift_result.loc[real_drift_result["alarm"], "index"].tolist()
    real_alarm_events = group_alarm_events(raw_real_alarm_indices)
    print("Reference window:", PH_SIGNAL_WINDOW, "days")
    print("Real-series rebaseline window:", real_rebaseline_window, "snapshots")
    print("Snapshots monitored:", len(real_drift_data))
    print("Raw alarms:", len(raw_real_alarm_indices))
    print("Grouped drift events:", len(real_alarm_events))
else:
    real_rebaseline_window = None
    real_drift_data["alarm"] = False
    real_drift_data["direction"] = None
    real_drift_data["statistic_up"] = np.nan
    real_drift_data["statistic_down"] = np.nan
    real_drift_data["rebaselining"] = False
    real_drift_data["reference_mean"] = np.nan
    real_drift_data["reference_std"] = np.nan
    print("Not enough snapshots for meaningful real-data drift monitoring (need >5).")

real_drift_data["cooldown"] = real_drift_data.get("rebaselining", False)
display(real_drift_data.round(4))

if not real_drift_data.empty:
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(
        real_drift_data["snapshot_date"],
        real_drift_data["log_median_monetary"],
        marker="o",
        label=f"Log Median Monetary ({PH_SIGNAL_WINDOW}-day reference window)"
    )
    alarm_data = real_drift_data[real_drift_data["alarm"]]
    if not alarm_data.empty:
        ax.scatter(
            alarm_data["snapshot_date"],
            alarm_data["log_median_monetary"],
            marker="x",
            s=100,
            label="Detected drift"
        )
    ax.set_title("Page-Hinkley Drift Detection - Fixed-Window Real Signal")
    ax.set_xlabel("Snapshot Date")
    ax.set_ylabel("Log Median Monetary")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# ============================================================
# SYNTHETIC VALIDATION + MONTE CARLO
# ============================================================
synthetic_values, synthetic_labels, true_drift_points = generate_synthetic_drift_data(
    length=PH_SYNTHETIC_LENGTH,
    warmup=PH_SYNTHETIC_WARMUP,
    drift_points=(80, 140),
    drift_levels=(1.2, 0.0),
    noise_std=0.20,
    random_state=PH_RANDOM_SEED
)
synthetic_result = run_page_hinkley(
    values=synthetic_values,
    warmup=PH_SYNTHETIC_WARMUP,
    delta=PH_DELTA,
    threshold=PH_THRESHOLD,
    direction="both",
    rebaseline_window=PH_REBASELINE_WINDOW,
    scale_floor_multiplier=PH_SCALE_FLOOR_MULTIPLIER
)
raw_detected_indices = synthetic_result.loc[synthetic_result["alarm"], "index"].tolist()
detected_indices = group_alarm_events(raw_detected_indices, max_gap=PH_ALARM_GAP)
synthetic_metrics = evaluate_drift_events(
    detected_indices=detected_indices,
    true_drift_points=true_drift_points,
    tolerance=PH_DETECTION_TOLERANCE,
    series_length=PH_SYNTHETIC_LENGTH,
    warmup=PH_SYNTHETIC_WARMUP
)
print("\n" + "=" * 75)
print("SYNTHETIC DRIFT EXPERIMENT")
print("=" * 75)
print("True drift points:", true_drift_points)
print("Raw alarms:", raw_detected_indices)
print("Grouped detected events:", detected_indices)
display(pd.DataFrame([synthetic_metrics]).round(4))

monte_carlo_results = run_drift_monte_carlo(n_trials=PH_MONTE_CARLO_TRIALS, threshold=PH_THRESHOLD, seed_offset=PH_TEST_SEED_OFFSET)
print("\n" + "=" * 75)
print("MONTE CARLO DRIFT EVALUATION BY SCENARIO")
print("=" * 75)
display(monte_carlo_results.round(4))
monte_carlo_summary = monte_carlo_results.groupby("scenario").agg(
    trials=("trial", "count"),
    precision_mean=("precision", "mean"),
    precision_std=("precision", "std"),
    recall_mean=("recall", "mean"),
    recall_std=("recall", "std"),
    f1_mean=("f1_score", "mean"),
    f1_std=("f1_score", "std"),
    detection_delay_mean=("mean_detection_delay", "mean"),
    false_alarms_per_100_mean=("false_alarms_per_100_points", "mean"),
    false_alarm_trial_rate=("false_alarm_trial", "mean")
).reset_index()

# Uncertainty intervals across independent held-out Monte Carlo trials.
ci_records = []
for scenario_name, scenario_rows in monte_carlo_results.groupby("scenario"):
    f1_low, f1_high = bootstrap_mean_interval(
        scenario_rows["f1_score"].to_numpy(),
        random_state=PH_RANDOM_SEED + 301
    )
    precision_low, precision_high = bootstrap_mean_interval(
        scenario_rows["precision"].to_numpy(),
        random_state=PH_RANDOM_SEED + 302
    )
    recall_low, recall_high = bootstrap_mean_interval(
        scenario_rows["recall"].to_numpy(),
        random_state=PH_RANDOM_SEED + 303
    )
    false_count = int(scenario_rows["false_alarm_trial"].sum())
    far_low, far_high = wilson_interval(false_count, len(scenario_rows))
    ci_records.append({
        "scenario": scenario_name,
        "f1_ci_low": f1_low, "f1_ci_high": f1_high,
        "precision_ci_low": precision_low, "precision_ci_high": precision_high,
        "recall_ci_low": recall_low, "recall_ci_high": recall_high,
        "false_alarm_rate_ci_low": far_low,
        "false_alarm_rate_ci_high": far_high
    })
monte_carlo_summary = monte_carlo_summary.merge(
    pd.DataFrame(ci_records), on="scenario", how="left", validate="one_to_one"
)
display(monte_carlo_summary.round(4))

fig, ax = plt.subplots(figsize=(10, 5))
plot_data = monte_carlo_summary.set_index("scenario")[["f1_mean", "false_alarm_trial_rate"]]
plot_data.plot(kind="bar", ax=ax)
ax.set_title("Drift detector: event F1 and false-alarm trial rate")
ax.set_ylabel("Rate")
ax.set_ylim(0, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

positive_scenarios = monte_carlo_results[
    ~monte_carlo_results["scenario"].str.startswith("no_drift")
]
null_scenario = monte_carlo_results[
    monte_carlo_results["scenario"].str.startswith("no_drift")
]
drift_experiment_summary = {
    "real_snapshots": int(len(real_drift_data)),
    "real_reference_window_days": int(PH_SIGNAL_WINDOW),
    "ph_delta": float(PH_DELTA),
    "ph_calibrated_threshold": float(PH_THRESHOLD),
    "ph_calibration_status": ph_calibration_status,
    "ph_calibration_null_trials": int(ph_threshold_calibration["null_trials"].iloc[0]),
    "ph_calibration_target_far": float(PH_TARGET_FALSE_ALARM_RATE),
    "real_rebaseline_window": (int(real_rebaseline_window) if real_rebaseline_window is not None else None),
    "real_raw_alarm_count": int(real_drift_data["alarm"].sum()) if "alarm" in real_drift_data else 0,
    "real_grouped_drift_events": int(len(real_alarm_events)),
    "synthetic_precision": float(synthetic_metrics["precision"]),
    "synthetic_recall": float(synthetic_metrics["recall"]),
    "synthetic_f1": float(synthetic_metrics["f1_score"]),
    "synthetic_detection_delay": synthetic_metrics["mean_detection_delay"],
    "synthetic_false_alarms_per_100_points": float(synthetic_metrics["false_alarms_per_100_points"]),
    "monte_carlo_mean_precision": float(positive_scenarios["precision"].mean()),
    "monte_carlo_mean_recall": float(positive_scenarios["recall"].mean()),
    "monte_carlo_mean_f1": float(positive_scenarios["f1_score"].mean()),
    "monte_carlo_std_f1": float(positive_scenarios["f1_score"].std()),
    "monte_carlo_mean_detection_delay": float(positive_scenarios["mean_detection_delay"].mean()),
    "monte_carlo_mean_false_alarms_per_100_points": float(positive_scenarios["false_alarms_per_100_points"].mean()),
    "monte_carlo_null_false_alarm_trial_rate": float(null_scenario["false_alarm_trial"].mean()),
    "monte_carlo_null_false_alarm_ci_low": float(monte_carlo_summary.loc[
        monte_carlo_summary["scenario"].str.startswith("no_drift"),
        "false_alarm_rate_ci_low"
    ].min()),
    "monte_carlo_null_false_alarm_ci_high": float(monte_carlo_summary.loc[
        monte_carlo_summary["scenario"].str.startswith("no_drift"),
        "false_alarm_rate_ci_high"
    ].max())
}
print("\n" + "=" * 75)
print("DRIFT DETECTION EXPERIMENT COMPLETED")
print("=" * 75)
display(pd.DataFrame([drift_experiment_summary]).T.rename(columns={0: "value"}))


Cell 14 — Baseline: Static RFM và Fixed Rolling RFM

In [ ]:

# ============================================================
# CELL 14 - BASELINE BENCHMARK
# Improved Fair Comparison Implementation
#
# Experiments:
# 1. Lifetime RFM + Fixed K-Means
# 2. Rolling 28 Days + Fixed K-Means
# 3. Rolling 56 Days + Fixed K-Means
# 4. Rolling 91 Days + Fixed K-Means
#
# Additional:
# 5. Common-cohort evaluation across all methods
# 6. Adaptive model evaluation on the same common cohort
#
# Outputs:
# - baseline_history
# - baseline_results
# - baseline_summary
# - baseline_rfm_data
# - common_cohort_results
# - adaptive_common_cohort_results
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)


# ============================================================
# 0. CONFIGURATION
# ============================================================

BASELINE_FIXED_K = 4

BASELINE_WINDOWS = [28, 56, 91]

BASELINE_RANDOM_STATE = 42

BASELINE_N_INIT = 10

BASELINE_SILHOUETTE_SAMPLE = 10000

BASELINE_MIN_CUSTOMERS = 10

BASELINE_MIN_COMMON_COHORT = 10

BASELINE_MIN_CLUSTER_SIZE = 2


# ============================================================
# 1. VALIDATE INPUTS
# ============================================================

if "df" not in globals():
    raise NameError(
        "DataFrame 'df' not found. Run preprocessing first."
    )

if "snapshot_dates" not in globals():
    raise NameError(
        "'snapshot_dates' not found. Run Cell 5 first."
    )

if "adaptive_history" not in globals():
    raise NameError(
        "'adaptive_history' not found. Run Cell 10 first."
    )

if len(snapshot_dates) == 0:
    raise ValueError("snapshot_dates is empty.")

if not isinstance(df, pd.DataFrame) or df.empty:
    raise ValueError("df is empty or is not a DataFrame.")

required_columns = [
    "CustomerID",
    "Invoice",
    "InvoiceDate",
    "Quantity",
    "Price",
    "Amount"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns in df: {missing_columns}"
    )


# ============================================================
# 2. PREPARE BASELINE TRANSACTIONS
# ============================================================

baseline_transactions = df[
    required_columns
].copy()

baseline_transactions["InvoiceDate"] = pd.to_datetime(
    baseline_transactions["InvoiceDate"],
    errors="coerce"
)

baseline_transactions["Quantity"] = pd.to_numeric(
    baseline_transactions["Quantity"],
    errors="coerce"
)

baseline_transactions["Price"] = pd.to_numeric(
    baseline_transactions["Price"],
    errors="coerce"
)

baseline_transactions["Amount"] = pd.to_numeric(
    baseline_transactions["Amount"],
    errors="coerce"
)

baseline_transactions["Invoice"] = (
    baseline_transactions["Invoice"].astype(str).str.strip()
)

baseline_transactions = baseline_transactions.dropna(
    subset=required_columns
)

# Use the same preprocessing policy as the main pipeline.
baseline_transactions = baseline_transactions[
    (baseline_transactions["Quantity"] > 0)
    & (baseline_transactions["Price"] > 0)
    & (baseline_transactions["Amount"] > 0)
    & (
        ~baseline_transactions["Invoice"]
        .str.upper()
        .str.startswith("C")
    )
].copy()

baseline_transactions = baseline_transactions.sort_values(
    "InvoiceDate"
).reset_index(drop=True)

# Keep the same monetary definition as Cell 6.
baseline_transactions["MonetaryValue"] = (
    baseline_transactions["Amount"]
)

if baseline_transactions.empty:
    raise ValueError(
        "No valid transactions remain for baseline experiments."
    )

print("=" * 75)
print("BASELINE TRANSACTION DATA")
print("=" * 75)

print(
    "Rows:",
    f"{len(baseline_transactions):,}"
)

print(
    "Customers:",
    f"{baseline_transactions['CustomerID'].nunique():,}"
)

print(
    "Date range:",
    baseline_transactions["InvoiceDate"].min(),
    "to",
    baseline_transactions["InvoiceDate"].max()
)


# ============================================================
# 3. CALCULATE RFM
# ============================================================

def baseline_calculate_rfm(
    transactions,
    snapshot_date,
    window_days=None
):
    """
    Calculate RFM using the same end-of-day anchor as Cell 6.

    Snapshot convention:
        snapshot_end = snapshot date normalized + 1 day

    Inclusion:
        InvoiceDate < snapshot_end

    Rolling window:
        InvoiceDate >= snapshot_end - window_days

    Recency:
        snapshot_end - LastPurchase

    Frequency:
        Number of unique invoices.

    Monetary:
        Sum of positive purchase amounts.
    """

    snapshot_end = (
        pd.Timestamp(snapshot_date).normalize()
        + pd.Timedelta(days=1)
    )

    # No future transactions.
    data = transactions[
        transactions["InvoiceDate"] < snapshot_end
    ].copy()

    if window_days is not None:

        if int(window_days) <= 0:
            raise ValueError(
                "window_days must be positive."
            )

        start_date = (
            snapshot_end
            - pd.Timedelta(days=int(window_days))
        )

        data = data[
            data["InvoiceDate"] >= start_date
        ]

    if data.empty:

        return pd.DataFrame(
            columns=[
                "Recency",
                "Frequency",
                "Monetary"
            ]
        ).rename_axis("CustomerID")

    rfm = data.groupby(
        "CustomerID"
    ).agg(

        LastPurchase=("InvoiceDate", "max"),

        Frequency=("Invoice", "nunique"),

        Monetary=("MonetaryValue", "sum")

    )

    rfm["Recency"] = (
        snapshot_end - rfm["LastPurchase"]
    ).dt.total_seconds() / 86400.0

    rfm = rfm[
        [
            "Recency",
            "Frequency",
            "Monetary"
        ]
    ]

    rfm = rfm.replace(
        [np.inf, -np.inf],
        np.nan
    ).dropna()

    rfm = rfm[
        (rfm["Recency"] >= 0)
        & (rfm["Frequency"] > 0)
        & (rfm["Monetary"] > 0)
    ]

    return rfm


# ============================================================
# 4. TRANSFORM AND SCALE RFM
# ============================================================

def baseline_transform_rfm(rfm):
    """
    Use the same feature transformation policy as Cell 7.

    Recency, Frequency and Monetary:
        log1p transformation

    Outliers:
        1st and 99th percentile clipping

    Scaling:
        StandardScaler

    Scaler is fitted only on the current snapshot/cohort.
    """

    required = [
        "Recency",
        "Frequency",
        "Monetary"
    ]

    if not set(required).issubset(rfm.columns):
        raise ValueError(
            "RFM must contain Recency, Frequency and Monetary."
        )

    if rfm.empty:
        raise ValueError(
            "Cannot transform an empty RFM dataset."
        )

    transformed = rfm[required].copy()

    transformed["Recency"] = np.log1p(
        transformed["Recency"].clip(lower=0)
    )

    transformed["Frequency"] = np.log1p(
        transformed["Frequency"].clip(lower=0)
    )

    transformed["Monetary"] = np.log1p(
        transformed["Monetary"].clip(lower=0)
    )

    for col in required:

        lower = transformed[col].quantile(0.01)

        upper = transformed[col].quantile(0.99)

        transformed[col] = transformed[col].clip(
            lower,
            upper
        )

    scaler = StandardScaler()

    X = scaler.fit_transform(
        transformed
    )

    return X, scaler


# ============================================================
# 5. EVALUATE CLUSTERING
# ============================================================

def baseline_evaluate_clustering(
    X,
    labels
):
    """
    Calculate internal clustering metrics.
    """

    X = np.asarray(X)

    labels = np.asarray(labels)

    if len(X) != len(labels):
        raise ValueError(
            "X and labels must have equal lengths."
        )

    if len(X) < 3:

        return {
            "silhouette": np.nan,
            "davies_bouldin": np.nan,
            "calinski_harabasz": np.nan
        }

    unique_labels = np.unique(labels)

    n_clusters = len(unique_labels)

    if (
        n_clusters < 2
        or n_clusters >= len(X)
    ):

        return {
            "silhouette": np.nan,
            "davies_bouldin": np.nan,
            "calinski_harabasz": np.nan
        }

    sample_size = min(
        BASELINE_SILHOUETTE_SAMPLE,
        len(X)
    )

    try:

        silhouette = silhouette_score(

            X,

            labels,

            sample_size=sample_size,

            random_state=BASELINE_RANDOM_STATE

        )

    except Exception as error:

        print(
            "Silhouette calculation warning:",
            str(error)
        )

        silhouette = np.nan

    try:

        dbi = davies_bouldin_score(
            X,
            labels
        )

    except Exception:

        dbi = np.nan

    try:

        chi = calinski_harabasz_score(
            X,
            labels
        )

    except Exception:

        chi = np.nan

    return {

        "silhouette": float(silhouette),

        "davies_bouldin": float(dbi),

        "calinski_harabasz": float(chi)

    }


# ============================================================
# 6. RUN ONE FIXED-K BASELINE
# ============================================================

def baseline_run_fixed_k(
    transactions,
    snapshot_date,
    window_days=None,
    fixed_k=BASELINE_FIXED_K,
    method_name="Baseline"
):
    """
    Run a fixed-K baseline on its complete available cohort.
    """

    rfm = baseline_calculate_rfm(
        transactions=transactions,
        snapshot_date=snapshot_date,
        window_days=window_days
    )

    if len(rfm) < max(
        BASELINE_MIN_CUSTOMERS,
        fixed_k + 1
    ):

        return None

    X, scaler = baseline_transform_rfm(
        rfm
    )

    model = KMeans(

        n_clusters=fixed_k,

        random_state=BASELINE_RANDOM_STATE,

        n_init=BASELINE_N_INIT

    )

    labels = model.fit_predict(X)

    cluster_sizes = np.bincount(
        labels
    )

    metrics = baseline_evaluate_clustering(
        X,
        labels
    )

    result = {

        "method": method_name,

        "snapshot_date": pd.Timestamp(
            snapshot_date
        ),

        "window_days": (
            "Lifetime"
            if window_days is None
            else int(window_days)
        ),

        "n_customers": len(rfm),

        "n_clusters": len(
            np.unique(labels)
        ),

        "silhouette": metrics["silhouette"],

        "davies_bouldin": metrics["davies_bouldin"],

        "calinski_harabasz": metrics["calinski_harabasz"],

        "inertia": float(model.inertia_),

        "minimum_cluster_size": int(
            cluster_sizes.min()
        ),

        "minimum_cluster_fraction": float(
            cluster_sizes.min() / len(rfm)
        ),

        "rfm": rfm.copy(),

        "X": X,

        "labels": labels,

        "model": model,

        "scaler": scaler

    }

    return result


# ============================================================
# 7. COMMON-COHORT BENCHMARK
# ============================================================

def evaluate_common_cohort(
    snapshot_date,
    method_rfms,
    method_k_values,
    method_window_days=None
):
    """Compare all baseline methods on the same customers active in 28 days.

    Because the candidate windows are nested, customers active in the shortest
    candidate window are also active in every longer candidate window. This
    creates a fair behavioural-clustering cohort without inserting synthetic
    F=M=0 records. The 91-day population is retained as a coverage diagnostic.
    """
    if len(method_rfms) < 2:
        return pd.DataFrame(), pd.DataFrame()
    method_window_days = method_window_days or {}
    common_window_days = int(min(BASELINE_WINDOWS))
    reference_window_days = int(max(BASELINE_WINDOWS))
    common_active_rfm = baseline_calculate_rfm(
        baseline_transactions, snapshot_date, window_days=common_window_days
    )
    reference_rfm = baseline_calculate_rfm(
        baseline_transactions, snapshot_date, window_days=reference_window_days
    )
    common_ids = common_active_rfm.index.sort_values()
    if len(common_ids) < BASELINE_MIN_COMMON_COHORT:
        return pd.DataFrame(), pd.DataFrame()
    if not common_ids.isin(reference_rfm.index).all():
        raise RuntimeError("Common-active cohort is not contained in the longest-window reference population.")

    metric_records = []
    profile_records = []
    for method_name, native_rfm in method_rfms.items():
        missing_ids = common_ids.difference(native_rfm.index)
        if len(missing_ids):
            raise RuntimeError(
                f"{method_name} misses {len(missing_ids)} common-active customers; "
                "check rolling-window construction."
            )
        rfm = native_rfm.reindex(common_ids).copy()
        if rfm.isna().any().any() or (rfm[["Frequency", "Monetary"]] <= 0).any().any():
            raise RuntimeError(f"{method_name} common-active RFM contains invalid/inactive values.")

        k = int(method_k_values[method_name])
        if len(rfm) <= k:
            print(f"Skipping {method_name}: common-active cohort is smaller than K.")
            continue
        X, scaler = baseline_transform_rfm(rfm)
        model = KMeans(
            n_clusters=k,
            random_state=BASELINE_RANDOM_STATE,
            n_init=20,
            algorithm="lloyd"
        )
        labels = model.fit_predict(X)
        metrics = baseline_evaluate_clustering(X, labels)
        sizes = np.bincount(labels, minlength=k)
        method_window = method_window_days.get(method_name)
        metric_records.append({
            "snapshot_date": pd.Timestamp(snapshot_date),
            "method": method_name,
            "n_common_customers": int(len(common_ids)),
            "common_active_window_days": common_window_days,
            "reference_window_days": reference_window_days,
            "reference_91d_customers": int(len(reference_rfm)),
            "native_n_customers": int(len(native_rfm)),
            "method_window_days": method_window,
            "inactive_in_common_cohort": 0,
            "n_clusters": int(len(np.unique(labels))),
            "silhouette": metrics["silhouette"],
            "davies_bouldin": metrics["davies_bouldin"],
            "calinski_harabasz": metrics["calinski_harabasz"],
            "inertia": float(model.inertia_),
            "minimum_cluster_size": int(sizes.min()),
            "minimum_cluster_fraction": float(sizes.min() / len(rfm))
        })
        profile = rfm.copy()
        profile["Cluster"] = labels
        profile["CustomerID"] = profile.index
        profile["method"] = method_name
        profile["snapshot_date"] = pd.Timestamp(snapshot_date)
        profile["common_active_window_days"] = common_window_days
        profile_records.append(profile.reset_index(drop=True))

    metrics_df = pd.DataFrame(metric_records)
    profile_df = pd.concat(profile_records, ignore_index=True) if profile_records else pd.DataFrame()
    return metrics_df, profile_df


# ============================================================
# 8. RUN BASELINE EXPERIMENTS
# ============================================================

baseline_methods = [

    {
        "name": "Lifetime RFM",
        "window_days": None
    },

    {
        "name": "Rolling 28 Days",
        "window_days": 28
    },

    {
        "name": "Rolling 56 Days",
        "window_days": 56
    },

    {
        "name": "Rolling 91 Days",
        "window_days": 91
    }

]

baseline_history = []

common_cohort_records = []

common_cohort_profiles = []

adaptive_common_records = []

print("\n" + "=" * 75)
print("RUNNING BASELINE BENCHMARK")
print("=" * 75)

for snapshot_index, snapshot_date in enumerate(
    snapshot_dates
):

    snapshot_date = pd.Timestamp(
        snapshot_date
    )

    print(
        f"\nSnapshot {snapshot_index + 1}/{len(snapshot_dates)}:",
        snapshot_date.date()
    )

    # --------------------------------------------------------
    # Run full-cohort baseline methods
    # --------------------------------------------------------

    snapshot_method_rfms = {}

    snapshot_method_k = {}

    snapshot_method_windows = {}

    for method in baseline_methods:

        result = baseline_run_fixed_k(

            transactions=baseline_transactions,

            snapshot_date=snapshot_date,

            window_days=method["window_days"],

            fixed_k=BASELINE_FIXED_K,

            method_name=method["name"]

        )

        if result is None:

            print(
                f"  Skipped {method['name']}: "
                "insufficient customers."
            )

            continue

        baseline_history.append(
            result
        )

        snapshot_method_rfms[
            method["name"]
        ] = result["rfm"]

        snapshot_method_k[
            method["name"]
        ] = BASELINE_FIXED_K
        snapshot_method_windows[method["name"]] = method["window_days"]

        print(

            f"  {method['name']}: "

            f"customers={result['n_customers']:,}, "

            f"Silhouette={result['silhouette']:.4f}, "

            f"DBI={result['davies_bouldin']:.4f}"

        )

    # --------------------------------------------------------
    # Add adaptive model to common-cohort comparison
    # --------------------------------------------------------

    matching_adaptive = [

        item for item in adaptive_history

        if pd.Timestamp(
            item["snapshot_date"]
        ).normalize() == snapshot_date.normalize()

    ]

    if matching_adaptive:

        adaptive_record = matching_adaptive[0]

        adaptive_rfm = adaptive_record["rfm"]

        adaptive_k = int(
            adaptive_record["k"]
        )

        snapshot_method_rfms[
            "Adaptive Rolling Dynamic RFM"
        ] = adaptive_rfm

        snapshot_method_k[
            "Adaptive Rolling Dynamic RFM"
        ] = adaptive_k
        snapshot_method_windows["Adaptive Rolling Dynamic RFM"] = int(adaptive_record["window"])

    # --------------------------------------------------------
    # Evaluate all methods on the same cohort
    # --------------------------------------------------------

    if len(snapshot_method_rfms) >= 2:

        common_metrics, common_profiles = (
            evaluate_common_cohort(

                snapshot_date=snapshot_date,

                method_rfms=snapshot_method_rfms,

                method_k_values=snapshot_method_k,

                method_window_days=snapshot_method_windows

            )
        )

        if not common_metrics.empty:

            common_cohort_records.append(
                common_metrics
            )

            if not common_profiles.empty:

                common_cohort_profiles.append(
                    common_profiles
                )

            print(
                "  Common cohort:",
                f"{common_metrics['n_common_customers'].iloc[0]:,}",
                "customers"
            )

            print(
                "  Common-cohort methods evaluated:",
                len(common_metrics)
            )

        else:

            print(
                "  Common-cohort evaluation unavailable."
            )


# ============================================================
# 9. CREATE BASELINE RESULTS
# ============================================================

baseline_results = pd.DataFrame([

    {

        "method": item["method"],

        "snapshot_date": item["snapshot_date"],

        "window_days": item["window_days"],

        "n_customers": item["n_customers"],

        "n_clusters": item["n_clusters"],

        "silhouette": item["silhouette"],

        "davies_bouldin": item["davies_bouldin"],

        "calinski_harabasz": item["calinski_harabasz"],

        "inertia": item["inertia"],

        "minimum_cluster_size": item["minimum_cluster_size"],

        "minimum_cluster_fraction": item[
            "minimum_cluster_fraction"
        ]

    }

    for item in baseline_history

])

if baseline_results.empty:

    raise RuntimeError(
        "No baseline experiment produced a valid result."
    )

print("\n" + "=" * 75)
print("BASELINE RESULTS - FULL COHORT")
print("=" * 75)

display(
    baseline_results.round(4)
)


# ============================================================
# 10. AGGREGATE BASELINE RESULTS
# ============================================================

baseline_summary = (

    baseline_results

    .groupby("method")

    .agg(

        snapshots=("snapshot_date", "count"),

        mean_customers=("n_customers", "mean"),

        mean_silhouette=("silhouette", "mean"),

        std_silhouette=("silhouette", "std"),

        mean_davies_bouldin=("davies_bouldin", "mean"),

        mean_calinski_harabasz=("calinski_harabasz", "mean"),

        mean_inertia=("inertia", "mean"),

        mean_minimum_cluster_fraction=(
            "minimum_cluster_fraction",
            "mean"
        )

    )

    .reset_index()

)

print("\n" + "=" * 75)
print("BASELINE SUMMARY")
print("=" * 75)

display(
    baseline_summary.round(4)
)


# ============================================================
# 11. COMMON-COHORT RESULTS
# ============================================================

common_cohort_results = (

    pd.concat(
        common_cohort_records,
        ignore_index=True
    )

    if common_cohort_records

    else pd.DataFrame()

)

adaptive_common_cohort_results = (

    common_cohort_results[
        common_cohort_results["method"]
        == "Adaptive Rolling Dynamic RFM"
    ].copy()

    if not common_cohort_results.empty

    else pd.DataFrame()

)

common_cohort_profiles_df = (

    pd.concat(
        common_cohort_profiles,
        ignore_index=True
    )

    if common_cohort_profiles

    else pd.DataFrame()

)

print("\n" + "=" * 75)
print("COMMON-COHORT BENCHMARK")
print("=" * 75)

if not common_cohort_results.empty:

    display(
        common_cohort_results.round(4)
    )

else:

    print(
        "No valid common-cohort comparisons were produced."
    )

    print(
        "This can happen when the intersection of customers "
        "across all windows is too small."
    )


# ============================================================
# 12. BUILD BASELINE RFM DATA EXPORT
# ============================================================

baseline_rfm_records = []

for item in baseline_history:

    temp = item["rfm"].copy()

    temp["CustomerID"] = temp.index

    temp["snapshot_date"] = item["snapshot_date"]

    temp["method"] = item["method"]

    temp["window_days"] = item["window_days"]

    temp["Cluster"] = item["labels"]

    baseline_rfm_records.append(
        temp.reset_index(drop=True)
    )

baseline_rfm_data = (

    pd.concat(
        baseline_rfm_records,
        ignore_index=True
    )

    if baseline_rfm_records

    else pd.DataFrame()

)


# ============================================================
# 13. VISUALIZE FULL-COHORT BASELINE
# ============================================================

plt.figure(figsize=(12, 6))

for method_name in baseline_results["method"].unique():

    method_data = baseline_results[
        baseline_results["method"] == method_name
    ].sort_values("snapshot_date")

    plt.plot(

        method_data["snapshot_date"],

        method_data["silhouette"],

        marker="o",

        label=method_name

    )

plt.title(
    "Baseline Comparison - Silhouette (Full Cohort)"
)

plt.xlabel("Snapshot Date")

plt.ylabel("Silhouette Score")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()


# ============================================================
# 14. VISUALIZE COMMON-COHORT BASELINE
# ============================================================

if not common_cohort_results.empty:

    plt.figure(figsize=(12, 6))

    for method_name in common_cohort_results[
        "method"
    ].unique():

        method_data = common_cohort_results[
            common_cohort_results["method"]
            == method_name
        ].sort_values("snapshot_date")

        plt.plot(

            method_data["snapshot_date"],

            method_data["silhouette"],

            marker="o",

            label=method_name

        )

    plt.title(
        "Fair Comparison - Silhouette on Common Customer Cohort"
    )

    plt.xlabel("Snapshot Date")

    plt.ylabel("Silhouette Score")

    plt.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.grid(alpha=0.3)

    plt.xticks(rotation=45)

    plt.tight_layout()

    plt.show()


# ============================================================
# 15. COMPLETION SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("CELL 14 - BASELINE BENCHMARK COMPLETED")
print("=" * 75)

print(
    "Full-cohort baseline runs:",
    len(baseline_history)
)

print(
    "Baseline methods:",
    baseline_results["method"].nunique()
)

print(
    "Snapshots:",
    baseline_results["snapshot_date"].nunique()
)

print(
    "Common-cohort evaluation rows:",
    len(common_cohort_results)
)

print("\nAvailable outputs:")

print("- baseline_history")
print("- baseline_results")
print("- baseline_summary")
print("- baseline_rfm_data")
print("- common_cohort_results")
print("- adaptive_common_cohort_results")
print("- common_cohort_profiles_df")


Cell 15 — Ablation Study và Benchmark

In [ ]:

# ============================================================
# CELL 15 - ABLATION STUDY
# Adaptive Rolling Dynamic RFM
#
# E1: Lifetime RFM + Fixed K
# E2: Fixed Window 56 + Fixed K
# E3: Fixed Window 56 + Adaptive K
# E4: Adaptive Window + Fixed K
# E5: Adaptive Window + Adaptive K
#
# Improvements:
# 1. Reuse Cell 8 Adaptive K-Means algorithm.
# 2. Record failures explicitly.
# 3. Compare all five experiments on a common cohort.
# 4. Keep native-cohort and common-cohort results separate.
# 5. Report K selection and cluster-size diagnostics.
#
# Outputs:
# - ablation_history
# - ablation_results
# - ablation_summary
# - ablation_k_selection
# - ablation_failures
# - ablation_common_cohort_results
# - ablation_common_cohort_profiles
# - ablation_comparison
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)


# ============================================================
# 0. CONFIGURATION
# ============================================================

ABLATION_FIXED_K = 4

ABLATION_RANDOM_STATE = 42

ABLATION_N_INIT = 10

ABLATION_SILHOUETTE_SAMPLE = 10000

ABLATION_MIN_CUSTOMERS = 10

ABLATION_MIN_COMMON_COHORT = 10

# Must agree with the fixed K used in Cell 14.
if "BASELINE_FIXED_K" in globals():

    ABLATION_FIXED_K = int(BASELINE_FIXED_K)


# ============================================================
# 1. VALIDATE INPUTS
# ============================================================

required_objects = [
    "baseline_transactions",
    "baseline_calculate_rfm",
    "baseline_transform_rfm",
    "baseline_evaluate_clustering",
    "snapshot_dates",
    "adaptive_history",
    "evaluate_k_candidates"
]

for object_name in required_objects:

    if object_name not in globals():

        raise NameError(
            f"Missing '{object_name}'. "
            "Run Cells 1-14 before Cell 15."
        )

if not adaptive_history:

    raise ValueError(
        "adaptive_history is empty. Run Cell 10 first."
    )

if len(snapshot_dates) == 0:

    raise ValueError(
        "snapshot_dates is empty."
    )

if ABLATION_FIXED_K < 2:

    raise ValueError(
        "ABLATION_FIXED_K must be at least 2."
    )

print("=" * 80)
print("ABLATION STUDY - VALIDATION")
print("=" * 80)

print("Fixed K:", ABLATION_FIXED_K)

print("Snapshots:", len(snapshot_dates))

print("Adaptive snapshots:", len(adaptive_history))


# ============================================================
# 2. BUILD ADAPTIVE WINDOW LOOKUP
# ============================================================

adaptive_lookup = {}

for item in adaptive_history:

    snapshot_key = pd.Timestamp(
        item["snapshot_date"]
    ).normalize()

    adaptive_lookup[snapshot_key] = item


def ablation_get_adaptive_window(value):

    if isinstance(value, (int, float, np.integer, np.floating)):

        if not np.isfinite(value):
            raise ValueError(
                f"Invalid adaptive window: {value}"
            )

        return int(value)

    raise ValueError(
        f"Adaptive window must be numeric, got: {value}"
    )


print(
    "Adaptive window lookup:",
    len(adaptive_lookup),
    "snapshots"
)


# ============================================================
# 3. REUSE CELL 8 ADAPTIVE K-MEANS
# ============================================================

def ablation_select_adaptive_k(X):

    """
    Use the exact Adaptive K selection function from Cell 8.

    Cell 8 performs:
        - Candidate K evaluation
        - Minimum cluster-size filtering
        - Bounded Silhouette utility
        - Bounded Davies-Bouldin utility
        - CHI retained as a diagnostic, not a selection objective
        - Explicit cluster-size fallback status

    This avoids implementing a different K-selection
    algorithm inside the ablation study.
    """

    best_result, k_metrics = evaluate_k_candidates(X)

    if best_result is None:

        raise RuntimeError(
            "Cell 8 evaluate_k_candidates returned no valid K."
        )

    selected_k = int(
        best_result["best_k"]
    )

    return selected_k, k_metrics.copy(), best_result


# ============================================================
# 4. DEFINE ABLATION EXPERIMENTS
# ============================================================

ABLATION_EXPERIMENTS = [

    {
        "name": "E1_Lifetime_FixedK",
        "window_type": "lifetime",
        "adaptive_k": False
    },

    {
        "name": "E2_Rolling56_FixedK",
        "window_type": "fixed56",
        "adaptive_k": False
    },

    {
        "name": "E3_Rolling56_AdaptiveK",
        "window_type": "fixed56",
        "adaptive_k": True
    },

    {
        "name": "E4_AdaptiveWindow_FixedK",
        "window_type": "adaptive",
        "adaptive_k": False
    },

    {
        "name": "E5_AdaptiveWindow_AdaptiveK",
        "window_type": "adaptive",
        "adaptive_k": True
    }

]

EXPECTED_EXPERIMENTS = len(
    ABLATION_EXPERIMENTS
)


# ============================================================
# 5. HELPER: DETERMINE WINDOW
# ============================================================

def ablation_resolve_window(
    experiment,
    snapshot_date
):

    window_type = experiment["window_type"]

    if window_type == "lifetime":

        return None

    if window_type == "fixed56":

        return 56

    if window_type == "adaptive":

        snapshot_key = pd.Timestamp(
            snapshot_date
        ).normalize()

        if snapshot_key not in adaptive_lookup:

            raise KeyError(
                f"No adaptive result for {snapshot_key.date()}."
            )

        adaptive_record = adaptive_lookup[
            snapshot_key
        ]

        return ablation_get_adaptive_window(
            adaptive_record["window"]
        )

    raise ValueError(
        f"Unknown window type: {window_type}"
    )


# ============================================================
# 6. RUN ONE NATIVE-COHORT EXPERIMENT
# ============================================================

def ablation_run_experiment(
    experiment_name,
    snapshot_date,
    window_days,
    adaptive_k=False,
    fixed_k=ABLATION_FIXED_K
):

    # --------------------------------------------------------
    # Calculate RFM
    # --------------------------------------------------------

    rfm = baseline_calculate_rfm(

        transactions=baseline_transactions,

        snapshot_date=snapshot_date,

        window_days=window_days

    )

    if len(rfm) < max(
        ABLATION_MIN_CUSTOMERS,
        fixed_k + 1
    ):

        raise ValueError(
            f"Insufficient customers: {len(rfm)}."
        )

    # --------------------------------------------------------
    # Transform RFM
    # --------------------------------------------------------

    X, scaler = baseline_transform_rfm(
        rfm
    )

    # --------------------------------------------------------
    # Select K
    # --------------------------------------------------------

    k_table = None

    adaptive_k_details = None

    if adaptive_k:

        selected_k, k_table, adaptive_k_details = (
            ablation_select_adaptive_k(X)
        )

    else:

        selected_k = int(fixed_k)

    if len(rfm) <= selected_k:

        raise ValueError(
            f"Number of customers ({len(rfm)}) "
            f"must be greater than K ({selected_k})."
        )

    # --------------------------------------------------------
    # Final K-Means
    # --------------------------------------------------------

    model = KMeans(

        n_clusters=selected_k,

        random_state=ABLATION_RANDOM_STATE,

        n_init=ABLATION_N_INIT

    )

    labels = model.fit_predict(X)

    cluster_sizes = np.bincount(
        labels,
        minlength=selected_k
    )

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    metrics = baseline_evaluate_clustering(
        X,
        labels
    )

    result = {

        "experiment": experiment_name,

        "snapshot_date": pd.Timestamp(
            snapshot_date
        ),

        "window_days": (
            "Lifetime"
            if window_days is None
            else int(window_days)
        ),

        "k_selection": (
            "Adaptive"
            if adaptive_k
            else "Fixed"
        ),

        "selected_k": selected_k,

        "n_customers": len(rfm),

        "n_clusters": len(
            np.unique(labels)
        ),

        "min_cluster_size": int(
            cluster_sizes.min()
        ),

        "max_cluster_size": int(
            cluster_sizes.max()
        ),

        "minimum_cluster_fraction": float(
            cluster_sizes.min() / len(rfm)
        ),

        "silhouette": metrics["silhouette"],

        "davies_bouldin": metrics["davies_bouldin"],

        "calinski_harabasz": metrics["calinski_harabasz"],

        "inertia": float(model.inertia_),

        "rfm": rfm.copy(),

        "X": X,

        "labels": labels.copy(),

        "model": model,

        "scaler": scaler,

        "k_metrics": k_table

    }

    return result, k_table


# ============================================================
# 7. EXECUTE FIVE ABLATION EXPERIMENTS
# ============================================================

ablation_history = []

ablation_k_selection_records = []

ablation_failures = []

print("\n" + "=" * 80)
print("STARTING ABLATION EXPERIMENTS")
print("=" * 80)

for snapshot_index, snapshot_date in enumerate(
    snapshot_dates
):

    snapshot_date = pd.Timestamp(
        snapshot_date
    )

    print(
        f"\nSnapshot {snapshot_index + 1}/{len(snapshot_dates)}: "
        f"{snapshot_date.date()}"
    )

    for experiment in ABLATION_EXPERIMENTS:

        experiment_name = experiment["name"]

        try:

            # Resolve window.
            window_days = ablation_resolve_window(
                experiment,
                snapshot_date
            )

            # Run experiment.
            result, k_table = ablation_run_experiment(

                experiment_name=experiment_name,

                snapshot_date=snapshot_date,

                window_days=window_days,

                adaptive_k=experiment["adaptive_k"],

                fixed_k=ABLATION_FIXED_K

            )

            ablation_history.append(
                result
            )

            # Save candidate K evaluation table.
            if k_table is not None:

                k_table = k_table.copy()

                k_table["experiment"] = experiment_name

                k_table["snapshot_date"] = snapshot_date

                k_table["selected_k"] = result[
                    "selected_k"
                ]

                ablation_k_selection_records.append(
                    k_table.reset_index(names="candidate_k")
                )

            print(

                f"  {experiment_name}: "

                f"Window={result['window_days']}, "

                f"K={result['selected_k']}, "

                f"Customers={result['n_customers']:,}, "

                f"Silhouette={result['silhouette']:.4f}"

            )

        except Exception as error:

            failure_record = {

                "experiment": experiment_name,

                "snapshot_date": snapshot_date,

                "error_type": type(error).__name__,

                "error_message": str(error)

            }

            ablation_failures.append(
                failure_record
            )

            print(
                f"  FAILED {experiment_name}: "
                f"{type(error).__name__}: {error}"
            )


# ============================================================
# 8. BUILD NATIVE-COHORT RESULTS
# ============================================================

if not ablation_history:

    raise RuntimeError(
        "No ablation experiment completed successfully. "
        "Inspect ablation_failures."
    )

ablation_results = pd.DataFrame([

    {

        "experiment": item["experiment"],

        "snapshot_date": item["snapshot_date"],

        "window_days": item["window_days"],

        "k_selection": item["k_selection"],

        "selected_k": item["selected_k"],

        "n_customers": item["n_customers"],

        "n_clusters": item["n_clusters"],

        "min_cluster_size": item["min_cluster_size"],

        "max_cluster_size": item["max_cluster_size"],

        "minimum_cluster_fraction": item[
            "minimum_cluster_fraction"
        ],

        "silhouette": item["silhouette"],

        "davies_bouldin": item["davies_bouldin"],

        "calinski_harabasz": item["calinski_harabasz"],

        "inertia": item["inertia"]

    }

    for item in ablation_history

])

ablation_failures = pd.DataFrame(
    ablation_failures
)

print("\n" + "=" * 80)
print("NATIVE-COHORT ABLATION RESULTS")
print("=" * 80)

display(
    ablation_results.round(4)
)

if not ablation_failures.empty:

    print("\nExperiment failures:")

    display(
        ablation_failures
    )

else:

    print(
        "\nAll experiment/snapshot combinations completed successfully."
    )


# ============================================================
# 9. BUILD ABLATION SUMMARY
# ============================================================

ablation_summary = (

    ablation_results

    .groupby("experiment")

    .agg(

        snapshots=("snapshot_date", "nunique"),

        mean_k=("selected_k", "mean"),

        std_k=("selected_k", "std"),

        mean_customers=("n_customers", "mean"),

        mean_silhouette=("silhouette", "mean"),

        std_silhouette=("silhouette", "std"),

        mean_davies_bouldin=("davies_bouldin", "mean"),

        mean_calinski_harabasz=("calinski_harabasz", "mean"),

        mean_inertia=("inertia", "mean"),

        mean_minimum_cluster_fraction=(
            "minimum_cluster_fraction",
            "mean"
        )

    )

    .reset_index()

)

print("\n" + "=" * 80)
print("ABLATION SUMMARY - NATIVE COHORT")
print("=" * 80)

display(
    ablation_summary.round(4)
)


# ============================================================
# 10. BUILD ADAPTIVE K SELECTION TABLE
# ============================================================

ablation_k_selection = (

    pd.concat(
        ablation_k_selection_records,
        ignore_index=True
    )

    if ablation_k_selection_records

    else pd.DataFrame()

)

print("\nAdaptive K candidate records:")

if not ablation_k_selection.empty:

    display(
        ablation_k_selection.head(20).round(4)
    )

else:

    print(
        "No adaptive K candidate records were produced."
    )


# ============================================================
# 11. COMMON-COHORT EVALUATION
# ============================================================

def ablation_evaluate_common_cohort(
    snapshot_date,
    experiment_records
):
    """Refit E1-E5 on the same customers active in the shortest window.

    This avoids a zero-inflated reference cohort where inactive customers can
    form an artificial cluster and dominate the internal clustering scores.
    Dormancy is reported separately by the activity analysis in Cell 12.
    """
    if len(experiment_records) != EXPECTED_EXPERIMENTS:
        raise ValueError("Not all five experiments are available for this snapshot.")

    common_window_days = int(min(BASELINE_WINDOWS))
    reference_window_days = int(max(BASELINE_WINDOWS))
    common_active_rfm = baseline_calculate_rfm(
        baseline_transactions, snapshot_date, window_days=common_window_days
    )
    reference_rfm = baseline_calculate_rfm(
        baseline_transactions, snapshot_date, window_days=reference_window_days
    )
    common_customers = common_active_rfm.index.sort_values()
    if len(common_customers) < ABLATION_MIN_COMMON_COHORT:
        raise ValueError(f"Common-active cohort too small: {len(common_customers)} customers.")
    if not common_customers.isin(reference_rfm.index).all():
        raise RuntimeError("Common-active cohort is not contained in the 91-day reference cohort.")

    common_results = []
    common_profiles = []
    for item in experiment_records:
        experiment_name = item["experiment"]
        source_rfm = item["rfm"]
        missing_ids = common_customers.difference(source_rfm.index)
        if len(missing_ids):
            raise RuntimeError(
                f"{experiment_name} misses {len(missing_ids)} common-active customers."
            )
        rfm = source_rfm.reindex(common_customers).copy()
        if rfm.isna().any().any() or (rfm[["Frequency", "Monetary"]] <= 0).any().any():
            raise RuntimeError(f"{experiment_name} common-active RFM contains invalid/inactive values.")

        X, scaler = baseline_transform_rfm(rfm)
        if item["k_selection"] == "Adaptive":
            selected_k, k_table, best_details = ablation_select_adaptive_k(X)
        else:
            selected_k = ABLATION_FIXED_K
            k_table = None
        if len(rfm) <= selected_k:
            raise ValueError(
                f"{experiment_name}: common-active cohort ({len(rfm)}) is too small for K={selected_k}."
            )
        model = KMeans(
            n_clusters=selected_k,
            random_state=ABLATION_RANDOM_STATE,
            n_init=20,
            algorithm="lloyd"
        )
        labels = model.fit_predict(X)
        metrics = baseline_evaluate_clustering(X, labels)
        cluster_sizes = np.bincount(labels, minlength=selected_k)

        common_results.append({
            "snapshot_date": pd.Timestamp(snapshot_date),
            "experiment": experiment_name,
            "n_common_customers": int(len(common_customers)),
            "common_active_window_days": common_window_days,
            "reference_window_days": reference_window_days,
            "reference_91d_customers": int(len(reference_rfm)),
            "native_n_customers": int(len(source_rfm)),
            "inactive_in_common_cohort": 0,
            "selected_k": int(selected_k),
            "n_clusters": int(len(np.unique(labels))),
            "silhouette": metrics["silhouette"],
            "davies_bouldin": metrics["davies_bouldin"],
            "calinski_harabasz": metrics["calinski_harabasz"],
            "inertia": float(model.inertia_),
            "min_cluster_size": int(cluster_sizes.min()),
            "max_cluster_size": int(cluster_sizes.max()),
            "minimum_cluster_fraction": float(cluster_sizes.min() / len(rfm))
        })
        profile = rfm.copy()
        profile["Cluster"] = labels
        profile["CustomerID"] = profile.index
        profile["experiment"] = experiment_name
        profile["snapshot_date"] = pd.Timestamp(snapshot_date)
        profile["common_active_window_days"] = common_window_days
        common_profiles.append(profile.reset_index(drop=True))

    return pd.DataFrame(common_results), pd.concat(common_profiles, ignore_index=True)


# ============================================================
# 12. RUN COMMON-COHORT COMPARISON
# ============================================================

ablation_common_results_records = []

ablation_common_profiles_records = []

successful_snapshot_keys = set(
    ablation_results["snapshot_date"].dt.normalize()
)

for snapshot_date in snapshot_dates:

    snapshot_date = pd.Timestamp(
        snapshot_date
    )

    snapshot_records = [

        item for item in ablation_history

        if pd.Timestamp(
            item["snapshot_date"]
        ).normalize() == snapshot_date.normalize()

    ]

    if len(snapshot_records) != EXPECTED_EXPERIMENTS:

        print(
            f"Common cohort skipped for "
            f"{snapshot_date.date()}: "
            f"{len(snapshot_records)}/{EXPECTED_EXPERIMENTS} experiments."
        )

        continue

    try:

        common_results, common_profiles = (
            ablation_evaluate_common_cohort(

                snapshot_date=snapshot_date,

                experiment_records=snapshot_records

            )
        )

        ablation_common_results_records.append(
            common_results
        )

        ablation_common_profiles_records.append(
            common_profiles
        )

        print(

            f"Common cohort {snapshot_date.date()}: "

            f"{common_results['n_common_customers'].iloc[0]:,} customers"

        )

    except Exception as error:

        ablation_failures = pd.concat(

            [

                ablation_failures,

                pd.DataFrame([{

                    "experiment": "COMMON_COHORT",

                    "snapshot_date": snapshot_date,

                    "error_type": type(error).__name__,

                    "error_message": str(error)

                }])

            ],

            ignore_index=True

        )

        print(
            f"Common-cohort evaluation failed "
            f"for {snapshot_date.date()}: {error}"
        )


ablation_common_cohort_results = (

    pd.concat(
        ablation_common_results_records,
        ignore_index=True
    )

    if ablation_common_results_records

    else pd.DataFrame()

)

ablation_common_cohort_profiles = (

    pd.concat(
        ablation_common_profiles_records,
        ignore_index=True
    )

    if ablation_common_profiles_records

    else pd.DataFrame()

)

# This is the preferred comparison table for paired
# cross-experiment interpretation.
ablation_comparison = ablation_common_cohort_results.copy()


# ============================================================
# 13. COMMON-COHORT SUMMARY
# ============================================================

if not ablation_common_cohort_results.empty:

    ablation_common_cohort_summary = (

        ablation_common_cohort_results

        .groupby("experiment")

        .agg(

            snapshots=("snapshot_date", "nunique"),

            mean_common_customers=("n_common_customers", "mean"),

            mean_k=("selected_k", "mean"),

            mean_silhouette=("silhouette", "mean"),

            std_silhouette=("silhouette", "std"),

            mean_davies_bouldin=("davies_bouldin", "mean"),

            mean_calinski_harabasz=("calinski_harabasz", "mean")

        )

        .reset_index()

    )

    print("\n" + "=" * 80)
    print("ABLATION SUMMARY - COMMON COHORT")
    print("=" * 80)

    display(
        ablation_common_cohort_summary.round(4)
    )

else:

    ablation_common_cohort_summary = pd.DataFrame()

    print(
        "\nNo common-cohort ablation results available."
    )


# ============================================================
# 14. PAIRED SNAPSHOT COMPARISON
# ============================================================

if not ablation_common_cohort_results.empty:

    complete_snapshot_counts = (

        ablation_common_cohort_results

        .groupby("snapshot_date")["experiment"]

        .nunique()

    )

    complete_ablation_snapshots = (
        complete_snapshot_counts[
            complete_snapshot_counts == EXPECTED_EXPERIMENTS
        ].index
    )

    ablation_comparison = (
        ablation_common_cohort_results[
            ablation_common_cohort_results["snapshot_date"].isin(
                complete_ablation_snapshots
            )
        ].copy()
    )

else:

    complete_ablation_snapshots = []

    ablation_comparison = pd.DataFrame()


print(
    "\nComplete common-cohort snapshots:",
    len(complete_ablation_snapshots)
)

if not ablation_failures.empty:

    print(
        "Total recorded failures:",
        len(ablation_failures)
    )


# ============================================================
# 15. VISUALIZE SILHOUETTE - NATIVE COHORT
# ============================================================

plt.figure(figsize=(13, 6))

for experiment_name in ablation_results[
    "experiment"
].unique():

    subset = ablation_results[
        ablation_results["experiment"] == experiment_name
    ].sort_values("snapshot_date")

    plt.plot(

        subset["snapshot_date"],

        subset["silhouette"],

        marker="o",

        label=experiment_name

    )

plt.title(
    "Ablation Study - Silhouette (Native Cohorts)"
)

plt.xlabel("Snapshot Date")

plt.ylabel("Silhouette Score")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()


# ============================================================
# 16. VISUALIZE SILHOUETTE - COMMON COHORT
# ============================================================

if not ablation_comparison.empty:

    plt.figure(figsize=(13, 6))

    for experiment_name in ablation_comparison[
        "experiment"
    ].unique():

        subset = ablation_comparison[
            ablation_comparison["experiment"] == experiment_name
        ].sort_values("snapshot_date")

        plt.plot(

            subset["snapshot_date"],

            subset["silhouette"],

            marker="o",

            label=experiment_name

        )

    plt.title(
        "Ablation Study - Silhouette (Common Customer Cohort)"
    )

    plt.xlabel("Snapshot Date")

    plt.ylabel("Silhouette Score")

    plt.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.grid(alpha=0.3)

    plt.xticks(rotation=45)

    plt.tight_layout()

    plt.show()


# ============================================================
# 17. VISUALIZE SELECTED K
# ============================================================

plt.figure(figsize=(13, 6))

for experiment_name in ablation_results[
    "experiment"
].unique():

    subset = ablation_results[
        ablation_results["experiment"] == experiment_name
    ].sort_values("snapshot_date")

    plt.plot(

        subset["snapshot_date"],

        subset["selected_k"],

        marker="o",

        label=experiment_name

    )

plt.title(
    "Ablation Study - Selected K (Native Cohorts)"
)

plt.xlabel("Snapshot Date")

plt.ylabel("Selected K")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()


# ============================================================
# 18. VISUALIZE DAVIES-BOULDIN INDEX
# ============================================================

plt.figure(figsize=(13, 6))

for experiment_name in ablation_results[
    "experiment"
].unique():

    subset = ablation_results[
        ablation_results["experiment"] == experiment_name
    ].sort_values("snapshot_date")

    plt.plot(

        subset["snapshot_date"],

        subset["davies_bouldin"],

        marker="o",

        label=experiment_name

    )

plt.title(
    "Ablation Study - Davies-Bouldin Index"
)

plt.xlabel("Snapshot Date")

plt.ylabel("Davies-Bouldin Index")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"

)

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

plt.tight_layout()

plt.show()


# ============================================================
# 19. FINAL COMPLETION SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("CELL 15 - ABLATION STUDY COMPLETED")
print("=" * 80)

print(
    "Successful native experiment runs:",
    len(ablation_history)
)

print(
    "Expected native experiment runs:",
    len(snapshot_dates) * EXPECTED_EXPERIMENTS
)

print(
    "Native experiment failures:",
    len(ablation_failures)
)

print(
    "Experiments represented:",
    ablation_results["experiment"].nunique()
)

print(
    "Common-cohort experiment rows:",
    len(ablation_common_cohort_results)
)

print(
    "Complete common-cohort snapshots:",
    len(complete_ablation_snapshots)
)

print("\nAvailable outputs:")

print("- ablation_history")
print("- ablation_results")
print("- ablation_summary")
print("- ablation_k_selection")
print("- ablation_failures")
print("- ablation_common_cohort_results")
print("- ablation_common_cohort_profiles")
print("- ablation_common_cohort_summary")
print("- ablation_comparison")

print("\nRecommended interpretation:")
print(
    "Use ablation_comparison for comparisons across E1-E5 "
    "because all experiments use the same customer cohort."
)


# Cell 16 — Visualization, Statistical Evaluation, Export và Reproducibility

Tổng hợp kết quả Adaptive RFM, Baseline, Ablation, Evolution, Stability và Drift; thực hiện kiểm định thống kê; xuất bảng/biểu đồ/ZIP; tạo reproducibility manifest và requirements.txt khóa phiên bản thư viện của lần chạy.

In [ ]:

# ============================================================
# CELL 16 - FINAL VISUALIZATION, STATISTICAL EVALUATION & EXPORT
# Adaptive Rolling Dynamic RFM
#
# Outputs:
# 1. Adaptive model summary
# 2. Baseline and ablation comparison
# 3. Common-cohort comparison
# 4. Segment evolution and customer migration
# 5. Temporal stability
# 6. Drift detection
# 7. Paired Wilcoxon tests + Holm correction
# 8. CSV, PNG and ZIP exports
# ============================================================

import os
import json
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon, rankdata


# ============================================================
# 0. CONFIGURATION
# ============================================================

OUTPUT_DIR = os.environ.get(
    "AR_DRFM_OUTPUT_DIR",
    "/content/adaptive_rolling_dynamic_rfm_results"
)

PLOT_DIR = os.path.join(
    OUTPUT_DIR,
    "figures"
)

TABLE_DIR = os.path.join(
    OUTPUT_DIR,
    "tables"
)

os.makedirs(
    PLOT_DIR,
    exist_ok=True
)

os.makedirs(
    TABLE_DIR,
    exist_ok=True
)

STATISTICAL_MIN_PAIRS = 8
STATISTICAL_BLOCK_LENGTH = 3
STATISTICAL_BOOTSTRAP_RESAMPLES = 3000

STATISTICAL_ALPHA = 0.05

print("=" * 80)
print("FINAL ANALYSIS")
print("=" * 80)


# ============================================================
# 1. VALIDATE MAIN OUTPUTS
# ============================================================

required_objects = [
    "adaptive_history",
    "baseline_results",
    "baseline_summary",
    "ablation_results",
    "ablation_summary",
    "evolution_events",
    "customer_migrations",
    "stability_df",
    "real_drift_data",
    "drift_experiment_summary"
]

missing_objects = [

    name for name in required_objects

    if name not in globals()

]

if missing_objects:

    raise NameError(
        "Missing required outputs: "
        + ", ".join(missing_objects)
    )

if not adaptive_history:

    raise ValueError(
        "adaptive_history is empty."
    )

if baseline_results.empty:

    raise ValueError(
        "baseline_results is empty."
    )

if ablation_results.empty:

    raise ValueError(
        "ablation_results is empty."
    )

print("Adaptive snapshots:", len(adaptive_history))

print("Baseline rows:", len(baseline_results))

print("Ablation rows:", len(ablation_results))


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def to_dataframe(value):

    if isinstance(value, pd.DataFrame):

        return value.copy()

    if isinstance(value, list):

        return pd.DataFrame(value)

    if isinstance(value, dict):

        return pd.DataFrame([value])

    return pd.DataFrame()


def safe_numeric_mean(series):

    values = pd.to_numeric(
        series,
        errors="coerce"
    )

    return values.mean()


def save_figure(filename):

    path = os.path.join(
        PLOT_DIR,
        filename
    )

    plt.tight_layout()

    plt.savefig(
        path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    plt.close()

    return path


# ============================================================
# 3. ADAPTIVE MODEL SNAPSHOT SUMMARY
# ============================================================

adaptive_records = []

adaptive_rfm_records = []

for item in adaptive_history:

    snapshot_date = pd.Timestamp(
        item["snapshot_date"]
    )

    rfm = item["rfm"].copy()

    X = np.asarray(
        item["X"]
    )

    labels = np.asarray(
        item["labels"]
    )

    if len(rfm) != len(labels):

        raise ValueError(
            f"RFM and labels mismatch at {snapshot_date}."
        )

    n_clusters = len(
        np.unique(labels)
    )

    # Get K from stored result or fitted model.
    if "k" in item:

        selected_k = int(item["k"])

    elif item.get("model") is not None:

        selected_k = int(
            item["model"].n_clusters
        )

    else:

        selected_k = n_clusters

    # Recalculate metrics consistently.
    metrics = baseline_evaluate_clustering(
        X,
        labels
    )

    cluster_sizes = pd.Series(
        labels
    ).value_counts()

    adaptive_records.append({

        "snapshot_date": snapshot_date,

        "window_days": item["window"],

        "selected_k": selected_k,

        "n_customers": len(rfm),

        "n_reference_cohort_customers": len(rfm),

        "n_native_customers": int(item.get("n_native_customers", len(rfm))),

        "reference_window_days": int(item.get("reference_window_days", REFERENCE_COHORT_WINDOW)),

        "n_inactive_in_window": int(item.get("n_inactive_in_window", 0)),

        "native_silhouette": item.get("native_silhouette", np.nan),

        "native_davies_bouldin": item.get("native_dbi", np.nan),

        "native_calinski_harabasz": item.get("native_chi", np.nan),

        "n_clusters": n_clusters,

        # Native-cohort quality: all customers used to fit this snapshot's model.
        "silhouette": metrics["silhouette"],

        "davies_bouldin": metrics["davies_bouldin"],

        "calinski_harabasz": metrics["calinski_harabasz"],

        # Common-cohort metrics used by Adaptive Window Selection.
        "selection_silhouette": item.get("silhouette", np.nan),

        "selection_dbi": item.get("dbi", np.nan),

        "selection_quality": item.get("quality", np.nan),

        "selection_eval_customers": item.get("n_eval_customers", np.nan),

        "temporal_ari": item.get("temporal_ari", np.nan),

        "ari_positive_utility": item.get("ari_positive_utility", np.nan),

        "stability_reliability": item.get("stability_reliability", np.nan),

        "stability_smoothed": item.get("stability_smoothed", np.nan),

        "effective_quality_weight": item.get("effective_quality_weight", np.nan),

        "effective_stability_weight": item.get("effective_stability_weight", np.nan),

        "best_candidate_window": item.get("best_candidate_window", np.nan),

        "score_improvement": item.get("score_improvement", np.nan),

        "switch_reason": item.get("switch_reason", None),

        "pending_confirmation_count": item.get("pending_confirmation_count", 0),

        "minimum_cluster_size": int(
            cluster_sizes.min()
        ),

        "maximum_cluster_size": int(
            cluster_sizes.max()
        ),

        "minimum_cluster_fraction": float(
            cluster_sizes.min() / len(rfm)
        ),

        "inertia": (
            float(item["model"].inertia_)
            if item.get("model") is not None
            else np.nan
        )

    })

    profile = rfm.copy()

    profile["CustomerID"] = profile.index

    profile["Cluster"] = labels

    profile["snapshot_date"] = snapshot_date

    profile["window_days"] = item["window"]

    profile["selected_k"] = selected_k

    adaptive_rfm_records.append(
        profile.reset_index(drop=True)
    )


adaptive_model_summary = pd.DataFrame(
    adaptive_records
)

# Preserve every candidate window score so the adaptive decision is auditable.
adaptive_window_evaluation_records = []
for item in adaptive_history:
    candidate_table = item.get("window_table", pd.DataFrame())
    if candidate_table is None or candidate_table.empty:
        continue
    for _, candidate_row in candidate_table.iterrows():
        candidate_record = candidate_row.to_dict()
        candidate_record["snapshot_date"] = pd.Timestamp(item["snapshot_date"])
        candidate_record["selected_window"] = int(item["window"])
        adaptive_window_evaluation_records.append(candidate_record)

adaptive_window_evaluations = pd.DataFrame(adaptive_window_evaluation_records)

# Sensitivity table is generated by Cell 10 from the cached per-window scores.
# It is a policy replay; it does not re-fit K-Means models.
if "adaptive_hysteresis_sensitivity" not in globals():
    adaptive_hysteresis_sensitivity = pd.DataFrame()
if "adaptive_parameter_sensitivity" not in globals():
    adaptive_parameter_sensitivity = pd.DataFrame()

adaptive_segment_data = pd.concat(
    adaptive_rfm_records,
    ignore_index=True
)

print("\nADAPTIVE MODEL SUMMARY")

display(
    adaptive_model_summary.round(4)
)


# ============================================================
# 4. OVERALL ADAPTIVE SUMMARY
# ============================================================

adaptive_overall_summary = pd.DataFrame([{

    "snapshots": adaptive_model_summary[
        "snapshot_date"
    ].nunique(),

    "mean_reference_cohort_customers": adaptive_model_summary[
        "n_reference_cohort_customers"
    ].mean(),

    "mean_native_window_customers": adaptive_model_summary[
        "n_native_customers"
    ].mean(),

    "mean_selected_k": adaptive_model_summary[
        "selected_k"
    ].mean(),

    "mean_silhouette": adaptive_model_summary[
        "silhouette"
    ].mean(),

    "std_silhouette": adaptive_model_summary[
        "silhouette"
    ].std(),

    "mean_davies_bouldin": adaptive_model_summary[
        "davies_bouldin"
    ].mean(),

    "mean_calinski_harabasz": adaptive_model_summary[
        "calinski_harabasz"
    ].mean(),

    "mean_minimum_cluster_fraction": (
        adaptive_model_summary[
            "minimum_cluster_fraction"
        ].mean()
    )

}])

print("\nADAPTIVE OVERALL SUMMARY")

display(
    adaptive_overall_summary.round(4)
)


# ============================================================
# 5. COMBINE NATIVE-COHORT COMPARISON
# ============================================================

adaptive_native_comparison = adaptive_model_summary[
    [
        "snapshot_date",
        "window_days",
        "n_native_customers",
        "selected_k",
        "native_silhouette",
        "native_davies_bouldin",
        "native_calinski_harabasz"
    ]
].rename(columns={
    "n_native_customers": "n_customers",
    "native_silhouette": "silhouette",
    "native_davies_bouldin": "davies_bouldin",
    "native_calinski_harabasz": "calinski_harabasz"
}).copy()

baseline_native_comparison = baseline_results.copy()

baseline_native_comparison["selected_k"] = (
    baseline_native_comparison["n_clusters"]
)

baseline_native_comparison["window_days"] = (
    baseline_native_comparison["window_days"]
)

baseline_native_comparison = baseline_native_comparison[
    [
        "snapshot_date",
        "window_days",
        "n_customers",
        "selected_k",
        "silhouette",
        "davies_bouldin",
        "calinski_harabasz",
        "method"
    ]
]

final_native_comparison = pd.concat(
    [
        adaptive_native_comparison,
        baseline_native_comparison
    ],
    ignore_index=True
)

final_native_summary = (

    final_native_comparison

    .groupby("method")

    .agg(

        snapshots=("snapshot_date", "nunique"),

        mean_customers=("n_customers", "mean"),

        mean_k=("selected_k", "mean"),

        mean_silhouette=("silhouette", "mean"),

        std_silhouette=("silhouette", "std"),

        mean_davies_bouldin=("davies_bouldin", "mean"),

        mean_calinski_harabasz=("calinski_harabasz", "mean")

    )

    .reset_index()

)

print("\nNATIVE-COHORT METHOD COMPARISON")

display(
    final_native_summary.round(4)
)


# ============================================================
# 6. COMMON-COHORT COMPARISON
# ============================================================

common_cohort_tables = []

if (
    "common_cohort_results" in globals()
    and not common_cohort_results.empty
):

    temp = common_cohort_results.copy()

    temp["comparison_group"] = (
        "Adaptive vs Baselines"
    )

    common_cohort_tables.append(
        temp
    )


if (
    "ablation_common_cohort_results" in globals()
    and not ablation_common_cohort_results.empty
):

    temp = ablation_common_cohort_results.copy()

    temp["comparison_group"] = (
        "Ablation E1-E5"
    )

    common_cohort_tables.append(
        temp
    )


final_common_cohort = (

    pd.concat(
        common_cohort_tables,
        ignore_index=True
    )

    if common_cohort_tables

    else pd.DataFrame()

)

if not final_common_cohort.empty:

    print("\nCOMMON-COHORT COMPARISON")

    display(
        final_common_cohort.round(4)
    )

else:

    print(
        "\nNo common-cohort comparison available."
    )


# ============================================================
# 7. PAIRED WILCOXON TEST
# ============================================================

def moving_block_bootstrap_ci(
    values,
    block_length=STATISTICAL_BLOCK_LENGTH,
    n_boot=STATISTICAL_BOOTSTRAP_RESAMPLES,
    random_state=RANDOM_STATE
):
    """Circular moving-block bootstrap CIs for mean and median differences.

    Resampling contiguous snapshot blocks preserves short-range temporal
    dependence better than an IID bootstrap over individual dates.
    Returns (mean_low, mean_high, median_low, median_high).
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    n = len(values)
    if n == 0:
        return (np.nan, np.nan, np.nan, np.nan)
    if n == 1:
        value = float(values[0])
        return (value, value, value, value)
    block_length = max(1, min(int(block_length), n))
    n_blocks = int(np.ceil(n / block_length))
    rng = np.random.default_rng(random_state)
    means = np.empty(int(n_boot), dtype=float)
    medians = np.empty(int(n_boot), dtype=float)
    offsets = np.arange(block_length)
    for b in range(int(n_boot)):
        starts = rng.integers(0, n, size=n_blocks)
        sample = np.concatenate([
            values[(start + offsets) % n] for start in starts
        ])[:n]
        means[b] = float(np.mean(sample))
        medians[b] = float(np.median(sample))
    mean_low, mean_high = np.quantile(means, [0.025, 0.975])
    median_low, median_high = np.quantile(medians, [0.025, 0.975])
    return (
        float(mean_low), float(mean_high),
        float(median_low), float(median_high)
    )


def paired_wilcoxon_comparison(
    data,
    method_column,
    metric_column,
    reference_method,
    comparison_group
):

    """
    Paired comparison by snapshot date.

    Positive median_difference means:
        comparison method > reference method.

    For DBI:
        Lower is better, so interpret direction accordingly.

    A minimum of STATISTICAL_MIN_PAIRS valid snapshot pairs
    is required.
    """

    results = []

    if data.empty:

        return pd.DataFrame()

    required = [
        "snapshot_date",
        method_column,
        metric_column
    ]

    if not set(required).issubset(data.columns):

        return pd.DataFrame()

    pivot = data.pivot_table(

        index="snapshot_date",

        columns=method_column,

        values=metric_column,

        aggfunc="mean"

    )

    if reference_method not in pivot.columns:

        return pd.DataFrame()

    for method_name in pivot.columns:

        if method_name == reference_method:

            continue

        pair = pivot[
            [
                reference_method,
                method_name
            ]
        ].dropna()

        n_pairs = len(pair)

        if n_pairs < STATISTICAL_MIN_PAIRS:

            results.append({

                "comparison_group": comparison_group,

                "metric": metric_column,

                "reference_method": reference_method,

                "comparison_method": method_name,

                "n_pairs": n_pairs,

                "median_difference": np.nan,

                "mean_difference": np.nan,

                "mean_difference_ci_low": np.nan,

                "mean_difference_ci_high": np.nan,

                "median_difference_ci_low": np.nan,

                "median_difference_ci_high": np.nan,

                "paired_rank_biserial": np.nan,

                "bootstrap_block_length": np.nan,

                "wilcoxon_statistic": np.nan,

                "p_value": np.nan,

                "test_status": "insufficient_pairs"

            })

            continue

        reference_values = pair[
            reference_method
        ].to_numpy(dtype=float)

        comparison_values = pair[
            method_name
        ].to_numpy(dtype=float)

        differences = (
            comparison_values - reference_values
        )

        if np.allclose(
            differences,
            0
        ):

            statistic = 0.0

            p_value = 1.0

            test_status = "all_differences_zero"

        else:

            try:

                test = wilcoxon(

                    comparison_values,

                    reference_values,

                    alternative="two-sided",

                    zero_method="wilcox",

                    method="auto"

                )

                statistic = float(
                    test.statistic
                )

                p_value = float(
                    test.pvalue
                )

                test_status = "ok"

            except Exception as error:

                statistic = np.nan

                p_value = np.nan

                test_status = (
                    "test_error: "
                    + str(error)
                )

        (ci_low, ci_high, median_ci_low, median_ci_high) = moving_block_bootstrap_ci(
            differences,
            block_length=min(STATISTICAL_BLOCK_LENGTH, n_pairs),
            n_boot=STATISTICAL_BOOTSTRAP_RESAMPLES,
            random_state=RANDOM_STATE + len(results) + n_pairs
        )

        nonzero = differences[~np.isclose(differences, 0.0)]
        if len(nonzero):
            signed_ranks = rankdata(np.abs(nonzero), method="average")
            positive_rank_sum = float(signed_ranks[nonzero > 0].sum())
            negative_rank_sum = float(signed_ranks[nonzero < 0].sum())
            rank_total = positive_rank_sum + negative_rank_sum
            paired_rank_biserial = (positive_rank_sum - negative_rank_sum) / rank_total if rank_total else 0.0
        else:
            paired_rank_biserial = 0.0

        results.append({

            "comparison_group": comparison_group,

            "metric": metric_column,

            "reference_method": reference_method,

            "comparison_method": method_name,

            "n_pairs": n_pairs,

            "median_difference": float(
                np.median(differences)
            ),

            "mean_difference": float(
                np.mean(differences)
            ),

            "mean_difference_ci_low": ci_low,

            "mean_difference_ci_high": ci_high,

            "median_difference_ci_low": median_ci_low,

            "median_difference_ci_high": median_ci_high,

            "paired_rank_biserial": float(paired_rank_biserial),

            "bootstrap_block_length": int(min(STATISTICAL_BLOCK_LENGTH, n_pairs)),

            "wilcoxon_statistic": statistic,

            "p_value": p_value,

            "test_status": test_status

        })

    return pd.DataFrame(results)


# ============================================================
# 8. HOLM MULTIPLE-TEST CORRECTION
# ============================================================

def apply_holm_correction(
    results,
    alpha=STATISTICAL_ALPHA
):

    """
    Holm-Bonferroni correction for multiple comparisons.
    """

    results = results.copy()

    results["p_adjusted_holm"] = np.nan

    results["significant_after_holm"] = False

    valid_indices = results.index[
        results["p_value"].notna()
    ].tolist()

    if not valid_indices:

        return results

    sorted_indices = sorted(

        valid_indices,

        key=lambda idx: results.loc[
            idx,
            "p_value"
        ]

    )

    m = len(sorted_indices)

    adjusted_values = []

    running_max = 0.0

    for rank, idx in enumerate(
        sorted_indices
    ):

        p_value = float(
            results.loc[idx, "p_value"]
        )

        adjusted = min(
            1.0,
            (m - rank) * p_value
        )

        running_max = max(
            running_max,
            adjusted
        )

        adjusted_values.append(
            (idx, running_max)
        )

    for idx, adjusted in adjusted_values:

        results.loc[
            idx,
            "p_adjusted_holm"
        ] = adjusted

        results.loc[
            idx,
            "significant_after_holm"
        ] = adjusted < alpha

    return results


# ============================================================
# 9. RUN STATISTICAL TESTS
# ============================================================

statistical_results = []

# ------------------------------------------------------------
# A. Adaptive model vs baseline methods
# ------------------------------------------------------------

if not final_common_cohort.empty:

    adaptive_baseline_data = final_common_cohort[
        final_common_cohort["comparison_group"]
        == "Adaptive vs Baselines"
    ].copy()

    if not adaptive_baseline_data.empty:

        for metric in [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]:

            result = paired_wilcoxon_comparison(

                data=adaptive_baseline_data,

                method_column="method",

                metric_column=metric,

                reference_method="Adaptive Rolling Dynamic RFM",

                comparison_group="Adaptive vs Baselines"

            )

            if not result.empty:

                statistical_results.append(
                    result
                )


# ------------------------------------------------------------
# B. E5 vs E1-E4
# ------------------------------------------------------------

if not final_common_cohort.empty:

    ablation_stat_data = final_common_cohort[
        final_common_cohort["comparison_group"]
        == "Ablation E1-E5"
    ].copy()

    if not ablation_stat_data.empty:

        for metric in [
            "silhouette",
            "davies_bouldin",
            "calinski_harabasz"
        ]:

            result = paired_wilcoxon_comparison(

                data=ablation_stat_data,

                method_column="experiment",

                metric_column=metric,

                reference_method="E5_AdaptiveWindow_AdaptiveK",

                comparison_group="E5 vs E1-E4"

            )

            if not result.empty:

                statistical_results.append(
                    result
                )


statistical_results = (

    pd.concat(
        statistical_results,
        ignore_index=True
    )

    if statistical_results

    else pd.DataFrame()

)

if not statistical_results.empty:

    statistical_results = apply_holm_correction(
        statistical_results
    )

print("\nPAIRED WILCOXON RESULTS")

if not statistical_results.empty:

    display(
        statistical_results.round(5)
    )
    print(
        "Interpretation note: these are full-series exploratory comparisons across "
        "all snapshots, not the untouched final test. Final test-period paired "
        "comparisons are reported separately in Cell 17. Wilcoxon p-values are "
        "Holm-adjusted; moving-block bootstrap CIs account for temporal dependence. "
        "Treat inference as exploratory because there are few temporal blocks."
    )

else:

    print(
        "No valid paired statistical tests were produced."
    )


# ============================================================
# 10. SEGMENT PROFILE
# ============================================================

segment_profile = (

    adaptive_segment_data

    .groupby(
        [
            "snapshot_date",
            "Cluster"
        ]
    )

    .agg(

        customer_count=("CustomerID", "nunique"),

        mean_recency=("Recency", "mean"),

        median_recency=("Recency", "median"),

        mean_frequency=("Frequency", "mean"),

        median_frequency=("Frequency", "median"),

        mean_monetary=("Monetary", "mean"),

        median_monetary=("Monetary", "median")

    )

    .reset_index()

)

print("\nSEGMENT PROFILE")

display(
    segment_profile.head(20).round(3)
)


# ============================================================
# 11. EVOLUTION AND MIGRATION TABLES
# ============================================================

evolution_events_df = to_dataframe(
    evolution_events
)

customer_migrations_df = to_dataframe(
    customer_migrations
)

stability_results_df = to_dataframe(
    stability_df
)

evolution_summary_df = to_dataframe(
    evolution_summary
    if "evolution_summary" in globals()
    else None
)

print("\nEVOLUTION EVENTS")

if not evolution_events_df.empty:

    display(
        evolution_events_df.head(20)
    )

else:

    print(
        "No evolution events available."
    )


print("\nCUSTOMER MIGRATIONS")

if not customer_migrations_df.empty:

    display(
        customer_migrations_df.head(20)
    )

else:

    print(
        "No customer migration records available."
    )


print("\nTEMPORAL STABILITY")

if not stability_results_df.empty:

    display(
        stability_results_df.head(20)
    )

else:

    print(
        "No temporal stability records available."
    )


# ============================================================
# 12. MIGRATION SUMMARY
# ============================================================

migration_summary = pd.DataFrame()

if not customer_migrations_df.empty:

    migration_group_column = next(

        (

            col for col in [
                "migrated",
                "status",
                "migration_type",
                "transition"
            ]

            if col in customer_migrations_df.columns

        ),

        None

    )

    if migration_group_column is not None:

        migration_summary = (

            customer_migrations_df

            .groupby(migration_group_column)

            .size()

            .reset_index(name="customer_count")

        )

    elif "migrated" in customer_migrations_df.columns:

        migration_summary = (

            customer_migrations_df[
                "migrated"
            ]

            .value_counts(dropna=False)

            .rename_axis("migrated")

            .reset_index(name="customer_count")

        )


# ============================================================
# 13. DRIFT DETECTION SUMMARY
# ============================================================

real_drift_data_df = to_dataframe(
    real_drift_data
)

drift_summary_df = to_dataframe(
    drift_experiment_summary
)

monte_carlo_summary_df = to_dataframe(
    monte_carlo_summary
    if "monte_carlo_summary" in globals()
    else None
)

synthetic_metrics_df = to_dataframe(
    synthetic_metrics
    if "synthetic_metrics" in globals()
    else None
)

real_alarm_events_df = to_dataframe(
    real_alarm_events
    if "real_alarm_events" in globals()
    else None
)

print("\nDRIFT EXPERIMENT SUMMARY")

display(
    drift_summary_df
)

if not monte_carlo_summary_df.empty:

    print("\nMONTE CARLO DRIFT SUMMARY")

    display(
        monte_carlo_summary_df
    )


# ============================================================
# 14. PLOT 1 - ADAPTIVE SILHOUETTE
# ============================================================

plt.figure(figsize=(12, 5))

plt.plot(

    adaptive_model_summary["snapshot_date"],

    adaptive_model_summary["silhouette"],

    marker="o"

)

plt.title(
    "Adaptive Rolling Dynamic RFM - Silhouette Over Time"
)

plt.xlabel("Snapshot Date")

plt.ylabel("Silhouette Score")

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

save_figure(
    "01_adaptive_silhouette.png"
)


# ============================================================
# 15. PLOT 2 - ADAPTIVE WINDOW
# ============================================================

plt.figure(figsize=(12, 5))

plt.plot(

    adaptive_model_summary["snapshot_date"],

    adaptive_model_summary["window_days"],

    marker="o"

)

plt.title(
    "Adaptive Rolling Window Selection"
)

plt.xlabel("Snapshot Date")

plt.ylabel("Selected Window (Days)")

plt.yticks(
    sorted(
        adaptive_model_summary[
            "window_days"
        ].dropna().unique()
    )
)

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

save_figure(
    "02_adaptive_window.png"
)


# ============================================================
# 16. PLOT 3 - ADAPTIVE K
# ============================================================

plt.figure(figsize=(12, 5))

plt.plot(

    adaptive_model_summary["snapshot_date"],

    adaptive_model_summary["selected_k"],

    marker="o"

)

plt.title(
    "Adaptive K Selection Over Time"
)

plt.xlabel("Snapshot Date")

plt.ylabel("Selected K")

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

save_figure(
    "03_adaptive_k.png"
)


# ============================================================
# 17. PLOT 4 - BASELINE VS ADAPTIVE
# ============================================================

plt.figure(figsize=(13, 6))

for method_name in final_native_comparison[
    "method"
].unique():

    subset = final_native_comparison[
        final_native_comparison["method"]
        == method_name
    ].sort_values("snapshot_date")

    plt.plot(

        subset["snapshot_date"],

        subset["silhouette"],

        marker="o",

        label=method_name

    )

plt.title(
    "Adaptive Model vs Baselines - Native Cohorts"
)

plt.xlabel("Snapshot Date")

plt.ylabel("Silhouette Score")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

save_figure(
    "04_adaptive_vs_baselines.png"
)


# ============================================================
# 18. PLOT 5 - ABLATION COMPARISON
# ============================================================

plt.figure(figsize=(13, 6))

for experiment_name in ablation_results[
    "experiment"
].unique():

    subset = ablation_results[
        ablation_results["experiment"]
        == experiment_name
    ].sort_values("snapshot_date")

    plt.plot(

        subset["snapshot_date"],

        subset["silhouette"],

        marker="o",

        label=experiment_name

    )

plt.title(
    "Ablation Study - Silhouette"
)

plt.xlabel("Snapshot Date")

plt.ylabel("Silhouette Score")

plt.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

save_figure(
    "05_ablation_silhouette.png"
)


# ============================================================
# 19. PLOT 6 - COMMON COHORT COMPARISON
# ============================================================

if not final_common_cohort.empty:

    plt.figure(figsize=(13, 6))

    for method_name in final_common_cohort[
        "method"
    ].dropna().unique():

        subset = final_common_cohort[
            final_common_cohort["method"]
            == method_name
        ].sort_values("snapshot_date")

        if "silhouette" not in subset.columns:
            continue

        plt.plot(

            subset["snapshot_date"],

            subset["silhouette"],

            marker="o",

            label=method_name

        )

    if "experiment" in final_common_cohort.columns:

        for experiment_name in final_common_cohort[
            "experiment"
        ].dropna().unique():

            subset = final_common_cohort[
                final_common_cohort["experiment"]
                == experiment_name
            ].sort_values("snapshot_date")

            if "silhouette" not in subset.columns:
                continue

            plt.plot(

                subset["snapshot_date"],

                subset["silhouette"],

                marker="o",

                label=experiment_name

            )

    plt.title(
        "Common-Cohort Comparison"
    )

    plt.xlabel("Snapshot Date")

    plt.ylabel("Silhouette Score")

    plt.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left"
    )

    plt.grid(alpha=0.3)

    plt.xticks(rotation=45)

    save_figure(
        "06_common_cohort_comparison.png"
    )


# ============================================================
# 20. PLOT 7 - SEGMENT SIZE OVER TIME
# ============================================================

segment_size_data = (

    adaptive_segment_data

    .groupby(
        [
            "snapshot_date",
            "Cluster"
        ]
    )["CustomerID"]

    .nunique()

    .reset_index(name="customer_count")

)

plt.figure(figsize=(13, 6))

for cluster_id in sorted(
    segment_size_data["Cluster"].unique()
):

    subset = segment_size_data[
        segment_size_data["Cluster"] == cluster_id
    ].sort_values("snapshot_date")

    plt.plot(

        subset["snapshot_date"],

        subset["customer_count"],

        marker="o",

        label=f"Cluster {cluster_id}"

    )

plt.title(
    "Customer Segment Size Over Time"
)

plt.xlabel("Snapshot Date")

plt.ylabel("Number of Customers")

plt.legend()

plt.grid(alpha=0.3)

plt.xticks(rotation=45)

save_figure(
    "07_segment_size_over_time.png"
)


# ============================================================
# 21. PLOT 8 - EVOLUTION EVENT COUNTS
# ============================================================

if (
    not evolution_events_df.empty
    and "event_type" in evolution_events_df.columns
):

    event_counts = (
        evolution_events_df["event_type"]
        .value_counts()
    )

    plt.figure(figsize=(10, 5))

    plt.bar(

        event_counts.index.astype(str),

        event_counts.values

    )

    plt.title(
        "Segment Evolution Event Counts"
    )

    plt.xlabel("Event Type")

    plt.ylabel("Number of Events")

    plt.xticks(rotation=30)

    save_figure(
        "08_evolution_events.png"
    )


# ============================================================
# 22. PLOT 9 - CUSTOMER MIGRATION
# ============================================================

if (
    not customer_migrations_df.empty
    and "migrated" in customer_migrations_df.columns
):

    migration_counts = (

        customer_migrations_df["migrated"]

        .value_counts(dropna=False)

    )

    plt.figure(figsize=(8, 5))

    plt.bar(

        migration_counts.index.astype(str),

        migration_counts.values

    )

    plt.title(
        "Customer Migration Status"
    )

    plt.xlabel("Migration Status")

    plt.ylabel("Customer Count")

    save_figure(
        "09_customer_migration.png"
    )

elif (
    not customer_migrations_df.empty
    and migration_summary.shape[0] > 0
):

    plt.figure(figsize=(9, 5))

    plt.bar(

        migration_summary.iloc[:, 0].astype(str),

        migration_summary["customer_count"]

    )

    plt.title(
        "Customer Migration Summary"
    )

    plt.xticks(rotation=30)

    save_figure(
        "09_customer_migration.png"
    )


# ============================================================
# 23. PLOT 10 - TEMPORAL STABILITY
# ============================================================

if (
    not stability_results_df.empty
    and "ari" in stability_results_df.columns
):

    plt.figure(figsize=(12, 5))

    plt.plot(

        range(len(stability_results_df)),

        stability_results_df["ari"],

        marker="o"

    )

    plt.title(
        "Temporal Clustering Stability - ARI"
    )

    plt.xlabel("Snapshot Transition")

    plt.ylabel("Adjusted Rand Index")

    plt.ylim(-1.05, 1.05)

    plt.grid(alpha=0.3)

    save_figure(
        "10_temporal_stability_ari.png"
    )


# ============================================================
# 24. PLOT 11 - REAL DRIFT SIGNAL
# ============================================================

if (
    not real_drift_data_df.empty
    and "log_median_monetary" in real_drift_data_df.columns
):

    plt.figure(figsize=(12, 5))

    plt.plot(

        real_drift_data_df["snapshot_date"],

        real_drift_data_df["log_median_monetary"],

        marker="o",

        label="Log median monetary"

    )

    if "alarm" in real_drift_data_df.columns:

        alarms = real_drift_data_df[
            real_drift_data_df["alarm"] == True
        ]

        if not alarms.empty:

            plt.scatter(

                alarms["snapshot_date"],

                alarms["log_median_monetary"],

                marker="x",

                s=100,

                label="Drift alarm"

            )

    plt.title(
        "Real Data - Monetary Drift Monitoring"
    )

    plt.xlabel("Snapshot Date")

    plt.ylabel("Log Median Monetary")

    plt.legend()

    plt.grid(alpha=0.3)

    plt.xticks(rotation=45)

    save_figure(
        "11_real_drift_monitoring.png"
    )


# ============================================================
# 25. PLOT 12 - SYNTHETIC DRIFT DETECTION
# ============================================================

if (
    "synthetic_result" in globals()
    and isinstance(synthetic_result, pd.DataFrame)
    and not synthetic_result.empty
):

    plt.figure(figsize=(12, 5))

    plt.plot(

        synthetic_result["index"],

        synthetic_result["value"],

        label="Synthetic signal"

    )

    if "alarm" in synthetic_result.columns:

        alarms = synthetic_result[
            synthetic_result["alarm"] == True
        ]

        if not alarms.empty:

            plt.scatter(

                alarms["index"],

                alarms["value"],

                marker="x",

                s=70,

                label="Detected alarm"

            )

    plt.title(
        "Synthetic Drift Detection"
    )

    plt.xlabel("Time Index")

    plt.ylabel("Signal Value")

    plt.legend()

    plt.grid(alpha=0.3)

    save_figure(
        "12_synthetic_drift_detection.png"
    )


# ============================================================
# 26. BUILD FINAL EXPERIMENT SUMMARY
# ============================================================

final_experiment_summary = {

    "dataset_rows": int(
        len(baseline_transactions)
    ),

    "dataset_customers": int(
        baseline_transactions["CustomerID"].nunique()
    ),

    "dataset_invoices": int(
        baseline_transactions["Invoice"].nunique()
    ),

    "snapshot_count": int(
        adaptive_model_summary["snapshot_date"].nunique()
    ),

    "adaptive_mean_silhouette": float(
        adaptive_model_summary["silhouette"].mean()
    ),

    "adaptive_mean_davies_bouldin": float(
        adaptive_model_summary["davies_bouldin"].mean()
    ),

    "adaptive_mean_calinski_harabasz": float(
        adaptive_model_summary["calinski_harabasz"].mean()
    ),

    "adaptive_window_switch_count": int(adaptive_model_summary["window_days"].ne(adaptive_model_summary["window_days"].shift()).sum() - 1) if len(adaptive_model_summary) > 1 else 0,
    "adaptive_mean_window_days": float(
        adaptive_model_summary["window_days"].mean()
    ),

    "adaptive_mean_k": float(
        adaptive_model_summary["selected_k"].mean()
    ),

    "adaptive_mean_selection_silhouette": float(
        adaptive_model_summary["selection_silhouette"].mean()
    ),

    "adaptive_window_candidate_evaluations": int(
        len(adaptive_window_evaluations)
    ),

    "mean_temporal_ari": float(
        stability_results_df["ari"].mean()
    ) if "ari" in stability_results_df.columns else np.nan,

    "baseline_method_count": int(
        baseline_results["method"].nunique()
    ),

    "ablation_experiment_count": int(
        ablation_results["experiment"].nunique()
    ),

    "common_cohort_rows": int(
        len(final_common_cohort)
    ),

    "evolution_event_count": int(
        len(evolution_events_df)
    ),

    "customer_migration_records": int(
        len(customer_migrations_df)
    ),

    "temporal_stability_records": int(
        len(stability_results_df)
    ),

    "real_drift_alarm_count": int(

        real_drift_data_df["alarm"].sum()

        if (
            not real_drift_data_df.empty
            and "alarm" in real_drift_data_df.columns
        )

        else 0

    ),

    "statistical_test_count": int(
        len(statistical_results)
    )

}


final_experiment_summary_df = pd.DataFrame(
    [final_experiment_summary]
)

print("\nFINAL EXPERIMENT SUMMARY")

display(
    final_experiment_summary_df.T.rename(
        columns={0: "Value"}
    )
)


# ============================================================
# 27. EXPORT TABLES
# ============================================================

tables_to_export = {

    "adaptive_model_summary": adaptive_model_summary,

    "adaptive_window_evaluations": adaptive_window_evaluations,

    "adaptive_hysteresis_sensitivity": adaptive_hysteresis_sensitivity,

    "adaptive_parameter_sensitivity": adaptive_parameter_sensitivity,

    "adaptive_overall_summary": adaptive_overall_summary,

    "adaptive_segment_data": adaptive_segment_data,

    "baseline_results": baseline_results,

    "baseline_summary": baseline_summary,

    "baseline_rfm_data": (
        baseline_rfm_data
        if "baseline_rfm_data" in globals()
        else pd.DataFrame()
    ),

    "final_native_comparison": final_native_comparison,

    "final_native_summary": final_native_summary,

    "final_common_cohort": final_common_cohort,

    "ablation_results": ablation_results,

    "ablation_summary": ablation_summary,

    "ablation_k_selection": (
        ablation_k_selection
        if "ablation_k_selection" in globals()
        else pd.DataFrame()
    ),

    "ablation_failures": (
        ablation_failures
        if isinstance(ablation_failures, pd.DataFrame)
        else pd.DataFrame()
    ),

    "ablation_common_cohort_results": (
        ablation_common_cohort_results
        if "ablation_common_cohort_results" in globals()
        else pd.DataFrame()
    ),

    "ablation_common_cohort_summary": (
        ablation_common_cohort_summary
        if "ablation_common_cohort_summary" in globals()
        else pd.DataFrame()
    ),

    "segment_profile": segment_profile,

    "evolution_events": evolution_events_df,

    "evolution_summary": evolution_summary_df,

    "customer_migrations": customer_migrations_df,

    "migration_summary": migration_summary,

    "stability_results": stability_results_df,

    "real_drift_data": real_drift_data_df,

    "drift_experiment_summary": drift_summary_df,

    "monte_carlo_summary": monte_carlo_summary_df,
    "monte_carlo_drift_trials": monte_carlo_results,

    "synthetic_metrics": synthetic_metrics_df,

    "real_alarm_events": real_alarm_events_df,

    "statistical_results": statistical_results,

    "ph_threshold_calibration": ph_threshold_calibration,

    "final_experiment_summary": final_experiment_summary_df

}

exported_files = []

for table_name, table in tables_to_export.items():

    if not isinstance(table, pd.DataFrame):

        continue

    if table.empty:

        continue

    output_path = os.path.join(

        TABLE_DIR,

        f"{table_name}.csv"

    )

    table.to_csv(

        output_path,

        index=False,

        encoding="utf-8-sig"

    )

    exported_files.append(
        output_path
    )

    print(
        "Exported:",
        table_name
    )


# ============================================================
# 28. EXPORT EXPERIMENT METADATA
# ============================================================

metadata = {

    "project": "Adaptive Rolling Dynamic RFM",

    "dataset": "UCI Online Retail II",

    "baseline_fixed_k": int(
        BASELINE_FIXED_K
    ) if "BASELINE_FIXED_K" in globals() else 4,

    "baseline_windows": [28, 56, 91],

    "ablation_experiments": [
        "E1_Lifetime_FixedK",
        "E2_Rolling56_FixedK",
        "E3_Rolling56_AdaptiveK",
        "E4_AdaptiveWindow_FixedK",
        "E5_AdaptiveWindow_AdaptiveK"
    ],

    "statistical_test": "Paired Wilcoxon signed-rank test",

    "multiple_test_correction": "Holm-Bonferroni",

    "statistical_alpha": STATISTICAL_ALPHA,

    "snapshot_count": len(adaptive_history),

    "notes": [

        "RFM Frequency uses unique invoices.",

        "Monetary uses positive purchase amounts.",

        "Cancelled invoices and non-positive purchases are excluded.",

        "Adaptive Window Selection uses a fixed longest-window reference cohort with observed Recency and window-specific Frequency/Monetary.",
        "Inactive customers in a shorter rolling window remain in the reference cohort with Frequency=0 and Monetary=0.",
        "Adaptive Window candidates are fit and evaluated on the same reference-cohort customer IDs.",
        "Page-Hinkley threshold is calibrated on independent stationary IID and AR(1) null streams; reported Monte Carlo testing uses separate seeds.",
        "Moving-block bootstrap confidence intervals complement paired Wilcoxon tests because snapshot metrics are temporally dependent.",

        "Native-cohort clustering metrics are reported separately from common-cohort selection metrics.",

        "K-Means cluster labels are arbitrary across snapshots.",

        "ARI and NMI are permutation-invariant stability measures.",

        "Drift alarms are statistical signals, not proof of behavioral drift."

    ]

}

metadata_path = os.path.join(
    OUTPUT_DIR,
    "experiment_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        metadata,
        file,
        indent=4,
        ensure_ascii=False
    )

exported_files.append(
    metadata_path
)


# ============================================================
# 28B. REPRODUCIBILITY MANIFEST
# ============================================================
import hashlib
import platform
import sys
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def sha256_dataframe(frame):
    """Hash values, column names and dtypes of the preprocessed frame."""
    digest = hashlib.sha256()
    digest.update(json.dumps(
        {"columns": [str(c) for c in frame.columns],
         "dtypes": [str(t) for t in frame.dtypes]},
        ensure_ascii=False,
        sort_keys=True
    ).encode("utf-8"))
    hashed_values = pd.util.hash_pandas_object(frame, index=True).values
    digest.update(hashed_values.tobytes())
    return digest.hexdigest()


source_dataset_path = globals().get("dataset_path", None)
dataset_file_metadata = {
    "path_or_name": str(source_dataset_path) if source_dataset_path is not None else None,
    "exists": False,
    "size_bytes": None,
    "sha256": None
}
if source_dataset_path is not None:
    try:
        from pathlib import Path
        source_path_obj = Path(source_dataset_path)
        if source_path_obj.is_file():
            dataset_file_metadata.update({
                "exists": True,
                "size_bytes": int(source_path_obj.stat().st_size),
                "sha256": sha256_file(source_path_obj)
            })
    except Exception as hash_error:
        dataset_file_metadata["hash_error"] = str(hash_error)

package_versions = {}
for package_name in ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "seaborn", "openpyxl", "joblib", "threadpoolctl"]:
    try:
        package_versions[package_name] = importlib_metadata.version(package_name)
    except importlib_metadata.PackageNotFoundError:
        package_versions[package_name] = None

# Hash the 18 code inputs executed before this Cell 16 in a clean Run All session.
# Restricting the scope avoids accidentally including unrelated commands from
# an old interactive Colab session in the execution fingerprint.
EXPECTED_NOTEBOOK_CODE_CELLS = 18  # code cells executed up to and including this cell
input_history = globals().get("In", None)
if isinstance(input_history, list) and len(input_history) - 1 >= EXPECTED_NOTEBOOK_CODE_CELLS:
    recent_inputs = [str(item) for item in input_history[-EXPECTED_NOTEBOOK_CODE_CELLS:]]
    code_payload = "\n\n".join(recent_inputs)
    executed_code_sha256 = hashlib.sha256(code_payload.encode("utf-8")).hexdigest()
    code_hash_scope = "last_18_inputs_through_cell16; use a fresh runtime and Run All"
else:
    executed_code_sha256 = None
    code_hash_scope = "unavailable: fewer than 18 executed notebook inputs"

run_completed_at = datetime.now(timezone.utc).isoformat()
run_started_at = globals().get("RUN_STARTED_AT_UTC", None)

reproducibility_manifest = {
    "project": "Adaptive Rolling Dynamic RFM",
    "manifest_version": "1.3",
    "run_started_at_utc": run_started_at,
    "run_completed_at_utc": run_completed_at,
    "python_version": sys.version,
    "platform": platform.platform(),
    "machine": platform.machine(),
    "cpu_count": os.cpu_count(),
    "working_directory": os.getcwd(),
    "threadpool_configuration": [
        {
            "internal_api": str(item.get("internal_api")),
            "num_threads": int(item.get("num_threads", 1)),
            "prefix": str(item.get("prefix")),
            "version": str(item.get("version"))
        }
        for item in threadpool_info()
    ],
    "package_versions": package_versions,
    "random_state": int(RANDOM_STATE),
    "dataset": {
        "name": "UCI Online Retail II",
        "uploaded_file": dataset_file_metadata,
        "preprocessed_rows": int(len(df)),
        "preprocessed_customers": int(df["CustomerID"].nunique()),
        "preprocessed_sha256": sha256_dataframe(df)
    },
    "experiment_configuration": {
        "windows_days": [int(x) for x in WINDOWS],
        "snapshot_stride_days": int(SNAPSHOT_STRIDE),
        "k_min": int(K_MIN),
        "k_max": int(K_MAX),
        "min_cluster_fraction": float(MIN_CLUSTER_FRACTION),
        "max_silhouette_sample": int(MAX_SILHOUETTE_SAMPLE),
        "quality_weight": float(QUALITY_WEIGHT),
        "stability_weight": float(STABILITY_WEIGHT),
        "coverage_weight": float(COVERAGE_WEIGHT),
        "selected_adaptive_policy": globals().get("SELECTED_ADAPTIVE_POLICY", {}),
        "hysteresis_margin": float(HYSTERESIS_MARGIN),
        "hysteresis_confirmations": int(HYSTERESIS_CONFIRMATIONS),
        "ari_stability_transform": "clip(ARI, 0, 1); ARI<=0 receives zero stability credit",
        "stability_overlap_reliability": "min(common_fraction / full_overlap_fraction, 1)",
        "stability_full_overlap_fraction": float(STABILITY_FULL_OVERLAP_FRACTION),
        "stability_min_overlap_fraction": float(STABILITY_MIN_OVERLAP_FRACTION),
        "reference_cohort_window_days": int(REFERENCE_COHORT_WINDOW),
        "adaptive_window_evaluation_cohort": "intersection of customers active in all candidate windows; identical IDs for each window quality comparison; inactive zero-filled rows excluded from primary clustering quality",
        "adaptive_final_clustering_population": "native-active customers in selected window; dormant customers excluded from behavioural clusters and reported separately",
        "fixed_activity_state_window_days": int(ACTIVITY_STATE_WINDOW_DAYS),
        "adaptive_quality_components": ["common_active_silhouette", "common_active_davies_bouldin"],
        "adaptive_selection_components": ["common_active_behavioural_quality", "positive_overlap_aware_temporal_stability", "reference_population_activity_coverage"],
        "adaptive_policy_selection_period": "validation snapshots only; one-standard-error rule; selected policy rerun chronologically with fixed hyperparameters before final test reporting",
        "window_quality_weights": {
            "silhouette": float(WINDOW_SILHOUETTE_WEIGHT),
            "davies_bouldin": float(WINDOW_DBI_WEIGHT)
        },
        "adaptive_stability_smoothing_alpha": 0.70,
        "drift_reference_window_days": int(PH_SIGNAL_WINDOW),
        "ph_delta": float(PH_DELTA),
        "ph_threshold": float(PH_THRESHOLD),
        "ph_threshold_grid": [float(x) for x in PH_THRESHOLD_GRID],
        "ph_calibration_trials_per_null_type": int(PH_CALIBRATION_TRIALS),
        "ph_target_false_alarm_rate": float(PH_TARGET_FALSE_ALARM_RATE),
        "ph_prewhiten_ar1": bool(PH_PREWHITEN_AR1),
        "ph_ar1_coefficient_clip": float(PH_AR1_COEFFICIENT_CLIP),
        "ph_calibration_status": str(ph_calibration_status),
        "ph_warmup": int(PH_SYNTHETIC_WARMUP),
        "ph_rebaseline_window_synthetic": int(PH_REBASELINE_WINDOW),
        "ph_rebaseline_window_real": (int(real_rebaseline_window) if real_rebaseline_window is not None else None),
        "ph_scale_floor_multiplier": float(PH_SCALE_FLOOR_MULTIPLIER),
        "ph_monte_carlo_trials_per_scenario": int(PH_MONTE_CARLO_TRIALS),
        "statistical_min_pairs": int(STATISTICAL_MIN_PAIRS),
        "statistical_block_length": int(STATISTICAL_BLOCK_LENGTH),
        "statistical_bootstrap_resamples": int(STATISTICAL_BOOTSTRAP_RESAMPLES)
    },
    "execution": {
        "adaptive_snapshots_completed": int(len(adaptive_history)),
        "adaptive_window_switches": int(adaptive_model_summary["window_days"].ne(adaptive_model_summary["window_days"].shift()).sum() - 1) if len(adaptive_model_summary) > 1 else 0,
        "adaptive_hysteresis_sensitivity_rows": int(len(adaptive_hysteresis_sensitivity)),
        "baseline_result_rows": int(len(baseline_results)),
        "ablation_result_rows": int(len(ablation_results)),
        "drift_monte_carlo_rows": int(len(monte_carlo_results)),
        "executed_input_history_sha256": executed_code_sha256,
        "executed_code_hash_scope": code_hash_scope,
        "expected_notebook_code_cells_at_cell16": int(EXPECTED_NOTEBOOK_CODE_CELLS),
        "adaptive_parameter_sensitivity_rows": int(len(adaptive_parameter_sensitivity))
    },
    "output_files": []
}

# Export exact package pins used by this run. This makes the Python package
# environment reproducible rather than only recording its versions in JSON.
requirements_path = os.path.join(OUTPUT_DIR, "requirements.txt")
requirements_lines = [
    f"{package_name}=={version}"
    for package_name, version in sorted(package_versions.items())
    if version is not None
]
with open(requirements_path, "w", encoding="utf-8") as requirements_handle:
    requirements_handle.write("\n".join(requirements_lines) + "\n")
exported_files.append(requirements_path)

# Full environment snapshot supplements the compact direct-dependency pins.
full_requirements_path = os.path.join(OUTPUT_DIR, "requirements_full.txt")
try:
    import subprocess
    freeze_text = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
        stderr=subprocess.STDOUT
    )
    with open(full_requirements_path, "w", encoding="utf-8") as freeze_handle:
        freeze_handle.write(freeze_text)
    exported_files.append(full_requirements_path)
except Exception as freeze_error:
    with open(full_requirements_path, "w", encoding="utf-8") as freeze_handle:
        freeze_handle.write(f"pip freeze unavailable: {freeze_error}\n")
    exported_files.append(full_requirements_path)

# Hash all files already exported (tables, figures and metadata).
for output_file in exported_files:
    if os.path.isfile(output_file):
        reproducibility_manifest["output_files"].append({
            "relative_path": os.path.relpath(output_file, OUTPUT_DIR),
            "size_bytes": int(os.path.getsize(output_file)),
            "sha256": sha256_file(output_file)
        })

manifest_path = os.path.join(OUTPUT_DIR, "reproducibility_manifest.json")
with open(manifest_path, "w", encoding="utf-8") as manifest_handle:
    json.dump(reproducibility_manifest, manifest_handle, indent=4, ensure_ascii=False)
exported_files.append(manifest_path)
print("Reproducibility manifest:", manifest_path)
print("Preprocessed data SHA-256:", reproducibility_manifest["dataset"]["preprocessed_sha256"])


# ============================================================
# 29. ZIP ALL RESULTS
# ============================================================

zip_path = str(Path(OUTPUT_DIR).with_suffix(".zip"))

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as zip_file:

    for root, dirs, files in os.walk(
        OUTPUT_DIR
    ):

        for filename in files:

            full_path = os.path.join(
                root,
                filename
            )

            relative_path = os.path.relpath(
                full_path,
                OUTPUT_DIR
            )

            zip_file.write(

                full_path,

                arcname=os.path.join(
                    "adaptive_rolling_dynamic_rfm_results",
                    relative_path
                )

            )


# ============================================================
# 30. FINAL REPORT
# ============================================================

print("\n" + "=" * 80)
print("ADAPTIVE ROLLING DYNAMIC RFM - FINAL REPORT")
print("=" * 80)

print(
    "Dataset rows:",
    f"{final_experiment_summary['dataset_rows']:,}"
)

print(
    "Customers:",
    f"{final_experiment_summary['dataset_customers']:,}"
)

print(
    "Invoices:",
    f"{final_experiment_summary['dataset_invoices']:,}"
)

print(
    "Snapshots:",
    final_experiment_summary["snapshot_count"]
)

print(
    "Adaptive mean Silhouette:",
    round(
        final_experiment_summary[
            "adaptive_mean_silhouette"
        ],
        4
    )
)

print(
    "Evolution events:",
    final_experiment_summary["evolution_event_count"]
)

print(
    "Migration records:",
    final_experiment_summary["customer_migration_records"]
)

print(
    "Drift alarms:",
    final_experiment_summary["real_drift_alarm_count"]
)

print(
    "Statistical tests:",
    final_experiment_summary["statistical_test_count"]
)

print(
    "Exported tables:",
    len(exported_files)
)

print(
    "\nResult directory:",
    OUTPUT_DIR
)

print(
    "\nZIP file:",
    zip_path
)

print("\nCELL 16 COMPLETED.")


Cell 17 — Temporal holdout summary và Cluster Imbalance Analysis

Phần này đánh giá 4 snapshot Test theo thời gian, báo cáo cluster-size distribution, Imbalance Ratio và normalized entropy. Đây là cluster imbalance sau clustering; không phải class imbalance của dataset gốc.

In [ ]:
# ============================================================
# TEMPORAL TEST REPORT + CLUSTER IMBALANCE
# ============================================================
from pathlib import Path
RESEARCH_VALIDATION_DIR = Path(OUTPUT_DIR) / "research_validation"
RESEARCH_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

history_by_date = {pd.Timestamp(r["snapshot_date"]): r for r in adaptive_history}
train_history = [history_by_date[pd.Timestamp(d)] for d in train_snapshot_dates if pd.Timestamp(d) in history_by_date]
validation_history = [history_by_date[pd.Timestamp(d)] for d in validation_snapshot_dates if pd.Timestamp(d) in history_by_date]
test_history = [history_by_date[pd.Timestamp(d)] for d in test_snapshot_dates if pd.Timestamp(d) in history_by_date]
if len(test_history) != len(test_snapshot_dates):
    raise RuntimeError(f"Expected {len(test_snapshot_dates)} test snapshots, found {len(test_history)}.")

# The adaptive policy parameters are frozen before final test evaluation.
# Test snapshots are reported separately and are not included in Cell 10 sensitivity tuning.
temporal_partition_summary = pd.DataFrame([
    {"partition": "train", "n_snapshots": len(train_history), "start": train_snapshot_dates[0], "end": train_snapshot_dates[-1]},
    {"partition": "validation", "n_snapshots": len(validation_history), "start": validation_snapshot_dates[0], "end": validation_snapshot_dates[-1]},
    {"partition": "test", "n_snapshots": len(test_history), "start": test_snapshot_dates[0], "end": test_snapshot_dates[-1]}
])
print("Temporal partition execution summary:")
display(temporal_partition_summary)

cluster_imbalance_rows = []
for record in adaptive_history:
    labels = np.asarray(record["labels"])
    counts = pd.Series(labels).value_counts().sort_index()
    sizes = counts.to_numpy(dtype=float)
    proportions = sizes / sizes.sum() if sizes.sum() else np.array([])
    nonzero = sizes[sizes > 0]
    if len(proportions) > 1:
        entropy = float(-(proportions * np.log(proportions)).sum() / np.log(len(proportions)))
    else:
        entropy = 0.0
    imbalance_ratio = float(nonzero.max() / nonzero.min()) if len(nonzero) else np.nan
    cluster_imbalance_rows.append({
        "snapshot_date": pd.Timestamp(record["snapshot_date"]),
        "period": "test" if pd.Timestamp(record["snapshot_date"]) in set(pd.to_datetime(test_snapshot_dates)) else ("validation" if pd.Timestamp(record["snapshot_date"]) in set(pd.to_datetime(validation_snapshot_dates)) else "train"),
        "window_days": int(record["window"]),
        "k": int(record["k"]),
        "n_customers": int(len(labels)),
        "n_reference_91d_customers": int(record.get("n_reference_cohort_customers", len(labels))),
        "dormant_in_selected_window": int(record.get("n_inactive_in_window", 0)),
        "active_coverage_of_91d_reference": float(record.get("activity_coverage_fraction", np.nan)),
        "largest_cluster_fraction": float(proportions.max()) if len(proportions) else np.nan,
        "smallest_cluster_fraction": float(proportions.min()) if len(proportions) else np.nan,
        "cluster_imbalance_ratio": imbalance_ratio,
        "normalized_cluster_entropy": entropy,
        "min_cluster_size": int(nonzero.min()) if len(nonzero) else 0,
        "max_cluster_size": int(nonzero.max()) if len(nonzero) else 0
    })
    for cluster_id, count in counts.items():
        cluster_imbalance_rows[-1][f"cluster_{int(cluster_id)}_count"] = int(count)
        cluster_imbalance_rows[-1][f"cluster_{int(cluster_id)}_fraction"] = float(count / counts.sum())
cluster_imbalance_df = pd.DataFrame(cluster_imbalance_rows)
print("Cluster imbalance by snapshot:")
display(cluster_imbalance_df.round(4))
print("Test-period cluster imbalance summary:")
display(cluster_imbalance_df.query("period == 'test'").select_dtypes(include=[np.number]).agg(["mean", "std"]).T.round(4))

# Per-snapshot cluster proportions as a heatmap (cluster IDs are local labels; do not infer semantic identity from numeric IDs).
cluster_fraction_cols = [c for c in cluster_imbalance_df.columns if c.endswith("_fraction") and c.startswith("cluster_")]
if cluster_fraction_cols:
    heat = cluster_imbalance_df.set_index("snapshot_date")[cluster_fraction_cols].fillna(0)
    fig, ax = plt.subplots(figsize=(12, max(4, 0.35 * len(heat))))
    sns.heatmap(heat, cmap="Blues", annot=False, ax=ax)
    ax.set_title("Cluster-size fractions across snapshots (labels are snapshot-local)")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Snapshot")
    plt.tight_layout()
    fig.savefig(RESEARCH_VALIDATION_DIR / "cluster_imbalance_heatmap.png", dpi=220, bbox_inches="tight")
    plt.show(); plt.close(fig)

# Holdout metric summary: internal clustering metrics, not classification accuracy.
test_adaptive_metrics = pd.DataFrame([{
    "snapshot_date": r["snapshot_date"], "window_days": r["window"], "selected_k": r["k"],
    "quality_cohort": "common_active_across_candidate_windows",
    "silhouette": r["silhouette"], "davies_bouldin": r["dbi"], "calinski_harabasz_diagnostic": r["chi"],
    "temporal_ari": r["temporal_ari"], "stability": r["stability"], "runtime_seconds": r["runtime_seconds"],
    "n_active_customers_for_evolution": r["n_customers"],
    "n_common_active_quality_customers": r.get("n_common_active_customers", r.get("n_eval_customers")),
    "n_reference_91d_customers": r.get("n_reference_cohort_customers"),
    "dormant_in_selected_window": r.get("n_inactive_in_window"),
    "activity_coverage_of_91d_reference": r.get("activity_coverage_fraction")
} for r in test_history])
print("Final temporal test metrics (no class labels; Accuracy is not applicable):")
display(test_adaptive_metrics.round(4))


# Test-only comparisons for the fixed-window baselines and the five ablation configurations.
_test_dates_set = set(pd.to_datetime(test_snapshot_dates))
# Primary baseline comparison: identical common-active customer IDs for all methods.
test_baseline_native_results = baseline_results[
    pd.to_datetime(baseline_results["snapshot_date"]).isin(_test_dates_set)
].copy()
test_baseline_results = common_cohort_results[
    pd.to_datetime(common_cohort_results["snapshot_date"]).isin(_test_dates_set)
].copy()
if test_baseline_results.empty:
    raise RuntimeError("No test-period common-active baseline results were produced.")
test_baseline_summary = test_baseline_results.groupby("method").agg(
    test_snapshots=("snapshot_date", "nunique"),
    silhouette_mean=("silhouette", "mean"), silhouette_sd=("silhouette", "std"),
    dbi_mean=("davies_bouldin", "mean"), dbi_sd=("davies_bouldin", "std"),
    n_common_customers_mean=("n_common_customers", "mean"),
    n_native_customers_mean=("native_n_customers", "mean")
).reset_index()
print("Primary baseline comparison: same common-active customer IDs on final test snapshots:")
display(test_baseline_summary.round(4))
print("Native-population baseline results are retained separately as descriptive diagnostics.")

test_ablation_results = ablation_common_cohort_results[
    pd.to_datetime(ablation_common_cohort_results["snapshot_date"]).isin(_test_dates_set)
].copy()
test_ablation_summary = test_ablation_results.groupby("experiment").agg(
    test_snapshots=("snapshot_date", "nunique"),
    silhouette_mean=("silhouette", "mean"), silhouette_sd=("silhouette", "std"),
    dbi_mean=("davies_bouldin", "mean"), dbi_sd=("davies_bouldin", "std"),
    selected_k_mean=("selected_k", "mean")
).reset_index()
print("Ablation comparison on untouched final test snapshots:")
display(test_ablation_summary.round(4))

# Primary paired test-period comparison: E5 (full adaptive model) vs E3
# (Rolling-56 + Adaptive-K). With only a few temporal test blocks, report the
# number of paired snapshots and effect direction; p-values are exploratory.
test_ablation_pivot = test_ablation_results.pivot_table(
    index="snapshot_date", columns="experiment", values="silhouette", aggfunc="mean"
)
if {"E3_Rolling56_AdaptiveK", "E5_AdaptiveWindow_AdaptiveK"}.issubset(test_ablation_pivot.columns):
    paired = test_ablation_pivot[["E3_Rolling56_AdaptiveK", "E5_AdaptiveWindow_AdaptiveK"]].dropna()
    paired_diff = paired["E5_AdaptiveWindow_AdaptiveK"] - paired["E3_Rolling56_AdaptiveK"]
    if len(paired_diff) >= 5 and not np.allclose(paired_diff.to_numpy(), 0):
        from scipy.stats import wilcoxon
        _w = wilcoxon(paired_diff.to_numpy(), alternative="two-sided", method="auto")
        test_ablation_primary_comparison = pd.DataFrame([{
            "comparison": "E5 minus E3 Silhouette",
            "n_paired_test_snapshots": int(len(paired_diff)),
            "mean_difference": float(paired_diff.mean()),
            "median_difference": float(paired_diff.median()),
            "wilcoxon_statistic": float(_w.statistic),
            "p_value_exploratory": float(_w.pvalue),
            "interpretation": "Exploratory only; test period contains a small number of temporal blocks."
        }])
    else:
        test_ablation_primary_comparison = pd.DataFrame([{
            "comparison": "E5 minus E3 Silhouette",
            "n_paired_test_snapshots": int(len(paired_diff)),
            "mean_difference": float(paired_diff.mean()) if len(paired_diff) else np.nan,
            "median_difference": float(paired_diff.median()) if len(paired_diff) else np.nan,
            "wilcoxon_statistic": np.nan,
            "p_value_exploratory": np.nan,
            "interpretation": "Insufficient non-zero paired test snapshots for Wilcoxon."
        }])
else:
    test_ablation_primary_comparison = pd.DataFrame()
print("Primary test-only paired ablation comparison:")
display(test_ablation_primary_comparison)

# Test-only paired statistics for all primary baseline and ablation comparisons.
# Six temporal test snapshots are a small inferential sample; results remain
# exploratory and are not used for hyperparameter selection.
def _test_only_pairwise_stats(data, method_column, metric_column, reference_method, comparison_group):
    if data.empty or not {"snapshot_date", method_column, metric_column}.issubset(data.columns):
        return []
    pivot = data.pivot_table(index="snapshot_date", columns=method_column, values=metric_column, aggfunc="mean")
    if reference_method not in pivot.columns:
        return []
    rows = []
    for comparison_method in pivot.columns:
        if comparison_method == reference_method:
            continue
        pair = pivot[[reference_method, comparison_method]].dropna()
        n_pairs = len(pair)
        if n_pairs < 5:
            rows.append({
                "comparison_group": comparison_group, "metric": metric_column,
                "reference_method": reference_method, "comparison_method": comparison_method,
                "n_pairs": n_pairs, "mean_difference": np.nan, "median_difference": np.nan,
                "mean_difference_ci_low": np.nan, "mean_difference_ci_high": np.nan,
                "median_difference_ci_low": np.nan, "median_difference_ci_high": np.nan,
                "paired_rank_biserial": np.nan, "p_value": np.nan,
                "test_status": "insufficient_test_snapshots"
            })
            continue
        diff = pair[comparison_method].to_numpy(dtype=float) - pair[reference_method].to_numpy(dtype=float)
        if np.allclose(diff, 0.0):
            statistic, p_value, status = 0.0, 1.0, "all_differences_zero"
        else:
            test = wilcoxon(
                pair[comparison_method].to_numpy(dtype=float),
                pair[reference_method].to_numpy(dtype=float),
                alternative="two-sided", zero_method="wilcox", method="auto"
            )
            statistic, p_value, status = float(test.statistic), float(test.pvalue), "exploratory_test_only"
        ci = moving_block_bootstrap_ci(
            diff,
            block_length=min(STATISTICAL_BLOCK_LENGTH, n_pairs),
            n_boot=STATISTICAL_BOOTSTRAP_RESAMPLES,
            random_state=RANDOM_STATE + n_pairs + len(rows)
        )
        nonzero = diff[~np.isclose(diff, 0.0)]
        if len(nonzero):
            ranks = rankdata(np.abs(nonzero), method="average")
            positive = float(ranks[nonzero > 0].sum())
            negative = float(ranks[nonzero < 0].sum())
            rank_biserial = (positive - negative) / max(positive + negative, 1e-12)
        else:
            rank_biserial = 0.0
        rows.append({
            "comparison_group": comparison_group,
            "metric": metric_column,
            "reference_method": reference_method,
            "comparison_method": comparison_method,
            "n_pairs": int(n_pairs),
            "mean_difference": float(np.mean(diff)),
            "median_difference": float(np.median(diff)),
            "mean_difference_ci_low": ci[0],
            "mean_difference_ci_high": ci[1],
            "median_difference_ci_low": ci[2],
            "median_difference_ci_high": ci[3],
            "paired_rank_biserial": float(rank_biserial),
            "p_value": float(p_value),
            "test_status": status
        })
    return rows

test_only_stat_rows = []
for metric in ["silhouette", "davies_bouldin", "calinski_harabasz"]:
    test_only_stat_rows.extend(_test_only_pairwise_stats(
        test_baseline_results, "method", metric,
        "Adaptive Rolling Dynamic RFM", "Test: Adaptive vs Baselines"
    ))
    test_only_stat_rows.extend(_test_only_pairwise_stats(
        test_ablation_results, "experiment", metric,
        "E5_AdaptiveWindow_AdaptiveK", "Test: E5 vs E1-E4"
    ))
test_only_statistical_results = pd.DataFrame(test_only_stat_rows)
if not test_only_statistical_results.empty:
    test_only_statistical_results = apply_holm_correction(test_only_statistical_results)
print("Test-only paired statistical comparisons (exploratory; six temporal blocks):")
display(test_only_statistical_results.round(5))



Cell 18 — Benchmark thuật toán clustering cổ điển trên Test

So sánh K-Means, Gaussian Mixture Model (GMM) và Agglomerative Clustering trên cùng vector RFM, cùng snapshot Test và cùng K (K được chọn bởi policy AR-DRFM tại snapshot đó). Giới hạn số khách hàng benchmark Agglomerative để kiểm soát bộ nhớ bậc hai. Không xem ARI giữa các thuật toán là accuracy.

In [ ]:
# ============================================================
# CLASSICAL CLUSTERING ALGORITHM BENCHMARK ON THE TEST PERIOD
# ============================================================
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering

CLASSICAL_BENCHMARK_MAX_CUSTOMERS = 2000
classical_benchmark_rows = []
benchmark_label_cache = {}

for test_pos, record in enumerate(test_history):
    X_full = np.asarray(record["X"], dtype=float)
    labels_adaptive = np.asarray(record["labels"], dtype=int)
    n_full = len(X_full)
    if n_full < 10:
        continue
    sample_n = min(CLASSICAL_BENCHMARK_MAX_CUSTOMERS, n_full)
    rng = np.random.default_rng(RANDOM_STATE + test_pos)
    sample_idx = np.sort(rng.choice(n_full, size=sample_n, replace=False)) if sample_n < n_full else np.arange(n_full)
    X_eval = X_full[sample_idx]
    adaptive_eval_labels = labels_adaptive[sample_idx]
    snapshot_key = pd.Timestamp(record["snapshot_date"])
    benchmark_label_cache[snapshot_key] = {"Adaptive K-Means": adaptive_eval_labels.copy()}
    k_same = int(record["k"])
    if k_same < 2 or k_same >= sample_n:
        continue

    def _record_cluster_metrics(snapshot, algorithm, labels, fit_seconds, k_value, ari_vs_adaptive=np.nan):
        labels = np.asarray(labels)
        benchmark_label_cache.setdefault(pd.Timestamp(snapshot), {})[algorithm] = labels.copy()
        unique = np.unique(labels)
        if len(unique) < 2 or len(unique) >= len(labels):
            return None
        counts = np.bincount(labels)
        sil = float(silhouette_score(X_eval, labels, sample_size=min(1500, len(X_eval)), random_state=RANDOM_STATE))
        dbi = float(davies_bouldin_score(X_eval, labels))
        chi = float(calinski_harabasz_score(X_eval, labels))
        return {
            "snapshot_date": pd.Timestamp(snapshot), "algorithm": algorithm, "k": int(k_value),
            "n_eval_customers": int(len(labels)), "silhouette": sil, "davies_bouldin": dbi,
            "calinski_harabasz_diagnostic": chi, "minimum_cluster_fraction": float(counts.min()/counts.sum()),
            "cluster_imbalance_ratio": float(counts.max()/counts.min()),
            "ari_vs_adaptive_kmeans": float(adjusted_rand_score(adaptive_eval_labels, labels)) if np.isfinite(ari_vs_adaptive) else np.nan,
            "fit_seconds": float(fit_seconds)
        }

    # K-Means fixed to the adaptive model's K and same selected-window representation.
    start = time.perf_counter()
    km = KMeans(n_clusters=k_same, random_state=RANDOM_STATE, n_init=20, algorithm="lloyd")
    km_labels = km.fit_predict(X_eval)
    elapsed = time.perf_counter() - start
    row = _record_cluster_metrics(record["snapshot_date"], "K-Means", km_labels, elapsed, k_same, 1.0)
    if row: classical_benchmark_rows.append(row)

    # Gaussian mixture with diagonal covariance for stable low-dimensional RFM features.
    try:
        start = time.perf_counter()
        gmm = GaussianMixture(n_components=k_same, covariance_type="diag", random_state=RANDOM_STATE, n_init=3, reg_covar=1e-5)
        gmm_labels = gmm.fit_predict(X_eval)
        elapsed = time.perf_counter() - start
        row = _record_cluster_metrics(record["snapshot_date"], "Gaussian Mixture Model", gmm_labels, elapsed, k_same, 1.0)
        if row: classical_benchmark_rows.append(row)
    except Exception as exc:
        classical_benchmark_rows.append({"snapshot_date": pd.Timestamp(record["snapshot_date"]), "algorithm": "Gaussian Mixture Model", "k": k_same, "error": str(exc)[:250]})

    # Ward linkage requires Euclidean geometry; same X, same K and same sample.
    try:
        start = time.perf_counter()
        agg = AgglomerativeClustering(n_clusters=k_same, linkage="ward")
        agg_labels = agg.fit_predict(X_eval)
        elapsed = time.perf_counter() - start
        row = _record_cluster_metrics(record["snapshot_date"], "Agglomerative (Ward)", agg_labels, elapsed, k_same, 1.0)
        if row: classical_benchmark_rows.append(row)
    except Exception as exc:
        classical_benchmark_rows.append({"snapshot_date": pd.Timestamp(record["snapshot_date"]), "algorithm": "Agglomerative (Ward)", "k": k_same, "error": str(exc)[:250]})

classical_benchmark_df = pd.DataFrame(classical_benchmark_rows)
if classical_benchmark_df.empty:
    raise RuntimeError("No classical clustering benchmark results were produced.")
classical_benchmark_summary = classical_benchmark_df.groupby("algorithm").agg(
    snapshots=("snapshot_date", "nunique"),
    silhouette_mean=("silhouette", "mean"), silhouette_sd=("silhouette", "std"),
    dbi_mean=("davies_bouldin", "mean"), dbi_sd=("davies_bouldin", "std"),
    chi_mean=("calinski_harabasz_diagnostic", "mean"),
    runtime_mean_seconds=("fit_seconds", "mean"), runtime_p95_seconds=("fit_seconds", lambda x: x.quantile(0.95)),
    imbalance_ratio_mean=("cluster_imbalance_ratio", "mean")
).reset_index()
print("Classical clustering benchmark on final test snapshots:")
display(classical_benchmark_summary.round(4))


# Pairwise partition agreement is reported alongside internal metrics.
from itertools import combinations
pairwise_ari_rows = []
for snapshot, algorithm_labels in benchmark_label_cache.items():
    for algo_a, algo_b in combinations(sorted(algorithm_labels), 2):
        labels_a = algorithm_labels[algo_a]
        labels_b = algorithm_labels[algo_b]
        if len(labels_a) == len(labels_b) and len(labels_a):
            pairwise_ari_rows.append({
                "snapshot_date": snapshot,
                "algorithm_a": algo_a,
                "algorithm_b": algo_b,
                "pairwise_ari": float(adjusted_rand_score(labels_a, labels_b)),
                "n_eval_customers": int(len(labels_a))
            })
classical_pairwise_ari_df = pd.DataFrame(pairwise_ari_rows)
print("Pairwise ARI agreement between classical algorithms and the adaptive partition:")
display(classical_pairwise_ari_df.round(4))
if not classical_pairwise_ari_df.empty:
    classical_pairwise_ari_summary = classical_pairwise_ari_df.groupby(
        ["algorithm_a", "algorithm_b"]
    ).agg(
        snapshots=("snapshot_date", "nunique"),
        pairwise_ari_mean=("pairwise_ari", "mean"),
        pairwise_ari_sd=("pairwise_ari", "std")
    ).reset_index()
    display(classical_pairwise_ari_summary.round(4))
else:
    classical_pairwise_ari_summary = pd.DataFrame()


Cell 19 — Frozen-centroid temporal holdout

Thí nghiệm bổ sung tách biệt với clustering động: fit scaler và K-Means chỉ trên snapshot Train cuối cùng, khóa preprocessing bounds và centroid, sau đó gán khách hàng thuộc các snapshot Test. Do Online Retail II không có ground-truth customer-segment labels, không báo Accuracy/Precision/Recall; thay vào đó báo cáo internal metrics, new-customer coverage, novelty-distance rate và ARI so với clustering refit (chỉ là mức tương đồng, không phải độ chính xác).

In [ ]:
# ============================================================
# FROZEN-CENTROID TEMPORAL HOLDOUT (TRAIN -> FUTURE TEST)
# ============================================================
# This is a separate inductive experiment: fit preprocessing and centroids
# using the final TRAIN snapshot only, then assign future active customers.
# Dormant customers are not represented by synthetic zero-valued RFM rows.
train_record = train_history[-1]
train_rfm_frozen = train_record["rfm"].copy()  # native-active cohort in selected train window
train_window_frozen = int(train_record["window"])
train_k_frozen = int(train_record["k"])
train_model_customer_ids = set(train_rfm_frozen.index.tolist())
train_cutoff = pd.Timestamp(train_record["snapshot_date"]).normalize() + pd.Timedelta(days=1)
train_known_customer_ids = set(
    df.loc[df["InvoiceDate"] < train_cutoff, "CustomerID"].astype(int).unique().tolist()
)

RFM_COLUMNS = ["Recency", "Frequency", "Monetary"]
def _log_rfm_frame(rfm_frame):
    out = rfm_frame[RFM_COLUMNS].copy().astype(float)
    for col in RFM_COLUMNS:
        out[col] = np.log1p(out[col].clip(lower=0))
    return out

if len(train_rfm_frozen) <= train_k_frozen:
    raise RuntimeError("The final training active cohort is too small for the frozen-centroid model.")
if (train_rfm_frozen[["Frequency", "Monetary"]] <= 0).any().any():
    raise RuntimeError("Frozen-centroid training cohort must contain active customers only.")

train_logged = _log_rfm_frame(train_rfm_frozen)
train_clip_bounds = {
    col: (
        float(train_logged[col].quantile(0.01)),
        float(train_logged[col].quantile(0.99))
    )
    for col in RFM_COLUMNS
}
train_clipped = train_logged.copy()
for col, (lo, hi) in train_clip_bounds.items():
    train_clipped[col] = train_clipped[col].clip(lo, hi)
frozen_scaler = StandardScaler().fit(train_clipped)
X_train_frozen = frozen_scaler.transform(train_clipped)
frozen_kmeans = KMeans(
    n_clusters=train_k_frozen,
    random_state=RANDOM_STATE,
    n_init=KMEANS_N_INIT,
    algorithm="lloyd"
).fit(X_train_frozen)
train_labels_frozen = frozen_kmeans.labels_
train_distances = np.linalg.norm(
    X_train_frozen - frozen_kmeans.cluster_centers_[train_labels_frozen], axis=1
)
novelty_distance_threshold = float(np.quantile(train_distances, 0.95))

def _transform_with_frozen_train_preprocessor(rfm_frame):
    logged = _log_rfm_frame(rfm_frame)
    for col, (lo, hi) in train_clip_bounds.items():
        logged[col] = logged[col].clip(lo, hi)
    return frozen_scaler.transform(logged)

frozen_holdout_rows = []
for record in test_history:
    snap = pd.Timestamp(record["snapshot_date"])
    # The same fixed TRAIN-selected window is used in every future test period.
    native_rfm_test = calculate_rfm(df, snap, window_days=train_window_frozen)
    if native_rfm_test.empty or len(native_rfm_test) <= train_k_frozen:
        print(f"Skipping frozen-centroid test at {snap.date()}: insufficient active test customers.")
        continue
    if (native_rfm_test[["Frequency", "Monetary"]] <= 0).any().any():
        raise RuntimeError("Frozen-centroid test cohort contains inactive/non-positive F/M rows.")

    X_future = _transform_with_frozen_train_preprocessor(native_rfm_test)
    frozen_labels = frozen_kmeans.predict(X_future)
    distances = np.linalg.norm(
        X_future - frozen_kmeans.cluster_centers_[frozen_labels], axis=1
    )
    refit_model = KMeans(
        n_clusters=train_k_frozen,
        random_state=RANDOM_STATE,
        n_init=KMEANS_N_INIT,
        algorithm="lloyd"
    )
    refit_labels = refit_model.fit_predict(X_future)
    frozen_unique = np.unique(frozen_labels)
    frozen_sil = (
        float(silhouette_score(
            X_future, frozen_labels,
            sample_size=min(1500, len(X_future)),
            random_state=RANDOM_STATE
        ))
        if 2 <= len(frozen_unique) < len(X_future) else np.nan
    )
    frozen_dbi = (
        float(davies_bouldin_score(X_future, frozen_labels))
        if 2 <= len(frozen_unique) < len(X_future) else np.nan
    )
    # Truly new = never observed before the train cutoff. Also report active
    # customers absent from the frozen centroid fit cohort as a separate concept.
    test_active_ids = set(native_rfm_test.index.astype(int).tolist())
    new_customer_ids = test_active_ids - train_known_customer_ids
    not_in_fit_ids = test_active_ids - train_model_customer_ids
    reference_rfm_test = calculate_rfm(df, snap, window_days=REFERENCE_COHORT_WINDOW)
    dormant_in_reference = int(max(0, len(reference_rfm_test) - len(native_rfm_test)))

    frozen_holdout_rows.append({
        "snapshot_date": snap,
        "train_cutoff": train_cutoff,
        "train_window_days": train_window_frozen,
        "frozen_train_k": train_k_frozen,
        "test_active_customers": int(len(native_rfm_test)),
        "test_reference_91d_customers": int(len(reference_rfm_test)),
        "dormant_in_91d_reference": dormant_in_reference,
        "new_customer_count_vs_all_train_history": int(len(new_customer_ids)),
        "new_customer_fraction_vs_all_train_history": float(len(new_customer_ids) / len(native_rfm_test)),
        "active_customers_not_in_frozen_fit_cohort": int(len(not_in_fit_ids)),
        "not_in_frozen_fit_cohort_fraction": float(len(not_in_fit_ids) / len(native_rfm_test)),
        "frozen_centroid_silhouette": frozen_sil,
        "frozen_centroid_dbi": frozen_dbi,
        "ari_frozen_vs_test_refit": float(adjusted_rand_score(frozen_labels, refit_labels)),
        "mean_nearest_centroid_distance": float(np.mean(distances)),
        "p95_nearest_centroid_distance": float(np.quantile(distances, 0.95)),
        "novelty_fraction_above_train_p95": float(np.mean(distances > novelty_distance_threshold)),
        "train_p95_distance_threshold": novelty_distance_threshold
    })
frozen_centroid_holdout_df = pd.DataFrame(frozen_holdout_rows)
if frozen_centroid_holdout_df.empty:
    raise RuntimeError("No frozen-centroid temporal holdout results were produced.")
print("Frozen-centroid future assignment results (active customers only):")
display(frozen_centroid_holdout_df.round(4))
print("Interpretation: ARI frozen-vs-refit measures segmentation agreement, not classification accuracy.")
print("New customers are defined relative to all customer IDs observed before the final TRAIN cutoff.")


Cell 20 — System performance, tham số và export gói nghiên cứu

Báo cáo latency, P95, throughput và memory ở mức quan sát được; lập bảng nguồn gốc cấu hình (tham số cố định theo thiết kế, tham số chọn bằng validation, hoặc tham số kế thừa từ phương pháp); xuất các bảng bổ sung và manifest cho lần chạy thật. Các giá trị cấu hình đề xuất không được mô tả là trích nguyên văn từ nghiên cứu nếu không có nguồn xác thực.

In [ ]:
# ============================================================
# PERFORMANCE + PARAMETER PROVENANCE + RESEARCH PACKAGE EXPORT
# ============================================================
import json, hashlib, sys, platform, zipfile, os
from datetime import datetime, timezone

# 1) Runtime and throughput from actual adaptive snapshots
performance_rows = []
for record in adaptive_history:
    snap_end = pd.Timestamp(record["snapshot_date"]).normalize() + pd.Timedelta(days=1)
    window_days = int(record["window"])
    start = snap_end - pd.Timedelta(days=window_days)
    n_selected_window_transactions = int(((df["InvoiceDate"] >= start) & (df["InvoiceDate"] < snap_end)).sum())
    # Candidate windows are each materialized for the adaptive decision; this
    # approximate scan count sums their row counts (overlap is counted again,
    # consistent with repeated window-specific filtering/aggregation work).
    n_candidate_window_transactions = int(sum(
        ((df["InvoiceDate"] >= snap_end - pd.Timedelta(days=int(w))) & (df["InvoiceDate"] < snap_end)).sum()
        for w in WINDOWS
    ))
    runtime = float(record["runtime_seconds"])
    performance_rows.append({
        "snapshot_date": pd.Timestamp(record["snapshot_date"]),
        "period": "test" if pd.Timestamp(record["snapshot_date"]) in set(pd.to_datetime(test_snapshot_dates)) else ("validation" if pd.Timestamp(record["snapshot_date"]) in set(pd.to_datetime(validation_snapshot_dates)) else "train"),
        "window_days": window_days,
        "transactions_in_selected_window": n_selected_window_transactions,
        "candidate_window_transaction_rows_scanned_approx": n_candidate_window_transactions,
        "adaptive_pipeline_latency_seconds": runtime,
        "candidate_rows_per_second_approx": n_candidate_window_transactions / runtime if runtime > 0 else np.nan,
        "customers_in_selected_active_cohort": int(record["n_customers"]),
        "common_active_quality_customers": int(record.get("n_common_active_customers", record.get("n_eval_customers", 0))),
        "dormant_in_selected_window": int(record.get("n_inactive_in_window", 0)),
        "k_evaluated": int(record["k"])
    })
performance_df = pd.DataFrame(performance_rows)
performance_summary = performance_df.groupby("period").agg(
    snapshots=("snapshot_date", "nunique"),
    mean_latency_seconds=("adaptive_pipeline_latency_seconds", "mean"),
    sd_latency_seconds=("adaptive_pipeline_latency_seconds", "std"),
    p95_latency_seconds=("adaptive_pipeline_latency_seconds", lambda x: x.quantile(0.95)),
    mean_candidate_rows_per_second_approx=("candidate_rows_per_second_approx", "mean"),
    mean_customers_evaluated=("customers_in_selected_active_cohort", "mean")
).reset_index()
print("Adaptive runtime by temporal partition:")
display(performance_summary.round(4))

# Current/peak process memory: explicitly label the scope; ru_maxrss is process-level high-water mark.
try:
    import resource
    peak_rss_raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak_rss_mb = float(peak_rss_raw / 1024.0)  # Linux reports KiB (Colab)
except Exception:
    peak_rss_mb = np.nan
try:
    import psutil
    current_rss_mb = float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
except Exception:
    current_rss_mb = np.nan
memory_summary = {"process_peak_rss_mb_linux_ru_maxrss": peak_rss_mb, "process_current_rss_mb_psutil": current_rss_mb}

# 2) Parameter provenance: clearly distinguish design choices from validation-selected parameters.
parameter_provenance_df = pd.DataFrame([
    {"parameter": "WINDOWS", "value": str(WINDOWS), "origin": "Research design choice", "rationale": "Candidate horizons of 4, 8 and 13 weeks; sensitivity should be interpreted on the validation period."},
    {"parameter": "SNAPSHOT_STRIDE", "value": SNAPSHOT_STRIDE, "origin": "Research design choice", "rationale": "Four-week update cadence; all retained snapshot gaps must be checked."},
    {"parameter": "K_MIN", "value": K_MIN, "origin": "Algorithmic constraint", "rationale": "At least two clusters are required for internal clustering metrics."},
    {"parameter": "K_MAX", "value": K_MAX, "origin": "Selected on validation", "rationale": "Chosen using K_MAX sensitivity and one-standard-error rule; no final test observations used."},
    {"parameter": "MIN_CLUSTER_FRACTION", "value": MIN_CLUSTER_FRACTION, "origin": "Researcher-defined constraint", "rationale": "Prevents accepting extremely small partitions; report sensitivity/constraint violations."},
    {"parameter": "QUALITY / STABILITY / COVERAGE WEIGHTS", "value": f"{QUALITY_WEIGHT:.2f}/{STABILITY_WEIGHT:.2f}/{COVERAGE_WEIGHT:.2f}", "origin": "Selected on validation by one-SE rule", "rationale": "Balances common-active behavioural quality, overlap-aware temporal stability and coverage of the 91-day reference population; sensitivity replay is validation-only."},
    {"parameter": "WINDOW_SILHOUETTE_WEIGHT / WINDOW_DBI_WEIGHT", "value": f"{WINDOW_SILHOUETTE_WEIGHT:.2f}/{WINDOW_DBI_WEIGHT:.2f}", "origin": "Researcher-defined utility", "rationale": "Bounded utility combination; not claimed as copied from a paper."},
    {"parameter": "HYSTERESIS_MARGIN", "value": HYSTERESIS_MARGIN, "origin": "Selected on validation by one-SE policy", "rationale": "Chosen together with the three objective weights using validation-only replay."},
    {"parameter": "HYSTERESIS_CONFIRMATIONS", "value": HYSTERESIS_CONFIRMATIONS, "origin": "Selected on validation by one-SE policy", "rationale": "Number of consecutive challenger wins required before switching; test period is excluded from selection."},
    {"parameter": "BASELINE_FIXED_K", "value": globals().get("BASELINE_FIXED_K", 4), "origin": "Baseline configuration", "rationale": "Fixed-K comparator; report separately from adaptive K selection."},
    {"parameter": "RANDOM_STATE", "value": RANDOM_STATE, "origin": "Reproducibility setting", "rationale": "Fixed seed for repeatable sampling and model initialization."}
])
print("Parameter provenance:")
display(parameter_provenance_df)

# 3) Export all added research artifacts and integrity manifest.
RESEARCH_VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
# Include EDA figures generated before the main Cell 16 output directory was finalized.
if Path(EDA_DIR).exists():
    import shutil
    target_eda_dir = RESEARCH_VALIDATION_DIR / "eda_figures"
    if target_eda_dir.exists():
        shutil.rmtree(target_eda_dir)
    shutil.copytree(EDA_DIR, target_eda_dir)
export_frames = {
    "raw_missingness": raw_missingness_df,
    "coercion_audit": coercion_audit_df,
    "preprocessing_audit": preprocessing_audit_df,
    "eda_distribution_summary": eda_distribution_summary,
    "eda_rfm_summary": rfm_eda_summary,
    "monthly_eda": monthly_eda,
    "country_summary": country_summary,
    "temporal_partition_summary": temporal_partition_summary,
    "cluster_imbalance": cluster_imbalance_df,
    "activity_state_transitions": activity_state_transition_df,
    "cluster_business_profiles": cluster_business_profiles_df,
    "cluster_profile_separation": cluster_profile_separation_df,
    "duplicate_sensitivity": duplicate_sensitivity_df,
    "test_adaptive_metrics": test_adaptive_metrics,
    "test_baseline_results_common_active": test_baseline_results,
    "test_baseline_native_results": test_baseline_native_results,
    "test_baseline_summary": test_baseline_summary,
    "test_ablation_results": test_ablation_results,
    "test_ablation_summary": test_ablation_summary,
    "test_ablation_primary_comparison": test_ablation_primary_comparison,
    "test_only_statistical_results": test_only_statistical_results,
    "classical_benchmark_detail": classical_benchmark_df,
    "classical_benchmark_summary": classical_benchmark_summary,
    "classical_pairwise_ari": classical_pairwise_ari_df,
    "frozen_centroid_holdout": frozen_centroid_holdout_df,
    "performance_detail": performance_df,
    "performance_summary": performance_summary,
    "parameter_provenance": parameter_provenance_df,
    "kmax_validation_candidates": kmax_candidate_metrics,
    "kmax_validation_summary": kmax_sensitivity_summary,
    "adaptive_policy_sensitivity": adaptive_parameter_sensitivity,
    "adaptive_hysteresis_sensitivity": adaptive_hysteresis_sensitivity
}
for name, frame in export_frames.items():
    if isinstance(frame, pd.DataFrame):
        frame.to_csv(RESEARCH_VALIDATION_DIR / f"{name}.csv", index=False)

# Compute exact hashes for the cleaned data and all supplementary output files.
def _sha256_path(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _sha256_df(frame):
    return hashlib.sha256(pd.util.hash_pandas_object(frame, index=True).values.tobytes()).hexdigest()

supplementary_files = []
for path in sorted(RESEARCH_VALIDATION_DIR.rglob("*")):
    if path.is_file():
        supplementary_files.append({
            "relative_path": str(path.relative_to(OUTPUT_DIR)),
            "size_bytes": int(path.stat().st_size),
            "sha256": _sha256_path(path)
        })

ipython_inputs = globals().get("In", None)
if isinstance(ipython_inputs, list) and len(ipython_inputs) > 1:
    executed_inputs_payload = "\n\n".join(str(x) for x in ipython_inputs[1:])
    executed_inputs_sha256 = hashlib.sha256(executed_inputs_payload.encode("utf-8")).hexdigest()
else:
    executed_inputs_sha256 = None

manifest_extension = {
    "project": "Adaptive Rolling Dynamic RFM (AR-DRFM)",
    "manifest_version": "2.0-research-extension",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python_version": sys.version,
    "platform": platform.platform(),
    "random_state": int(RANDOM_STATE),
    "executed_notebook_inputs_sha256": executed_inputs_sha256,
    "dataset": {
        "name": "UCI Online Retail II",
        "raw_rows": int(len(raw_df)),
        "clean_rows": int(len(df)),
        "customers": int(df["CustomerID"].nunique()),
        "invoices": int(df["Invoice"].nunique()),
        "clean_dataframe_sha256": _sha256_df(df)
    },
    "temporal_split": {
        "policy": TEMPORAL_SPLIT_POLICY,
        "train_dates": [str(pd.Timestamp(x).date()) for x in train_snapshot_dates],
        "validation_dates": [str(pd.Timestamp(x).date()) for x in validation_snapshot_dates],
        "test_dates": [str(pd.Timestamp(x).date()) for x in test_snapshot_dates]
    },
    "modeling": {
        "classical_algorithms": ["KMeans", "GaussianMixture", "AgglomerativeClustering-Ward"],
        "deep_learning_used": False,
        "k_max_initial": int(KMAX_DEFAULT_BEFORE_VALIDATION),
        "k_max_validation_selected": int(K_MAX_RECOMMENDED),
        "k_max_selection_rule": "one-standard-error rule on validation snapshots",
        "frozen_centroid_holdout_is_separate_experiment": True,
        "no_ground_truth_customer_segment_labels": True,
        "accuracy_precision_recall_for_clustering_not_reported": True
    },
    "performance_memory": memory_summary,
    "synthetic_data_note": "Synthetic Page-Hinkley scenarios are detector stress-tests only; they are not reported as Online Retail II empirical results.",
    "evaluation_integrity_note": "The main clustering task is unsupervised. No class labels or classification Accuracy are claimed. Test snapshots are not used for parameter tuning; frozen-centroid holdout is a separate inductive assignment experiment.",
    "supplementary_output_files": supplementary_files
}
manifest_extension_path = RESEARCH_VALIDATION_DIR / "research_extension_manifest.json"
manifest_extension_path.write_text(json.dumps(manifest_extension, indent=2, ensure_ascii=False), encoding="utf-8")

# Final package includes original Cell 16 exports plus new validation/EDA outputs.
package_path = Path(OUTPUT_DIR).parent / "AR_DRFM_research_complete_package.zip"
source_dir = Path(OUTPUT_DIR)
with zipfile.ZipFile(package_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in source_dir.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(source_dir)))
print("\nResearch extension export completed:", package_path)
print("Manifest:", manifest_extension_path)
print("Supplementary files:", len(supplementary_files))
print("Memory observation:", memory_summary)


## Research integrity audit — active cohorts, leakage, and reproducibility

This final audit checks structural invariants after a clean Run All. The main behavioural clustering score is computed on the intersection of customers active in all candidate windows. Customers dormant in a selected window are not assigned artificial zero-valued RFM vectors; they are reported separately, and activity-state transitions use a fixed horizon. These design choices separate behavioural segmentation from customer dormancy.

The notebook remains an unsupervised clustering study. It does not report classification accuracy or pretend that customer-segment ground-truth labels exist. Synthetic Page-Hinkley streams are detector stress tests only.


In [ ]:
# ============================================================
# CELL 21 — FINAL RESEARCH INTEGRITY AUDIT
# ============================================================
from itertools import combinations
import hashlib, json, zipfile
from pathlib import Path

integrity_checks = []
def _add_check(name, passed, detail):
    integrity_checks.append({"check": name, "passed": bool(passed), "detail": str(detail)})

# 1) Chronological split must be complete, ordered and disjoint.
all_split_dates = list(train_snapshot_dates) + list(validation_snapshot_dates) + list(test_snapshot_dates)
expected_dates = list(snapshot_dates)
_add_check("temporal_split_covers_all_snapshots", all(pd.Timestamp(a) == pd.Timestamp(b) for a, b in zip(all_split_dates, expected_dates)) and len(all_split_dates) == len(expected_dates), f"train/validation/test={len(train_snapshot_dates)}/{len(validation_snapshot_dates)}/{len(test_snapshot_dates)}; total={len(all_split_dates)}")
_add_check("temporal_partitions_nonempty", bool(train_snapshot_dates and validation_snapshot_dates and test_snapshot_dates), "All three chronological partitions must be non-empty.")
_add_check("temporal_partition_order", bool(pd.Timestamp(train_snapshot_dates[-1]) < pd.Timestamp(validation_snapshot_dates[0]) < pd.Timestamp(test_snapshot_dates[0])), "Train precedes validation and validation precedes test.")

# 2) Every adaptive clustering record must contain active-only native RFM.
active_record_checks = []
for rec in adaptive_history:
    rfm = rec["rfm"]
    labels = np.asarray(rec["labels"])
    X = np.asarray(rec["X"])
    ok = (
        rfm.index.is_unique
        and len(rfm) == len(labels) == len(X)
        and not rfm.isna().any().any()
        and (rfm[["Frequency", "Monetary"]] > 0).all().all()
    )
    active_record_checks.append(bool(ok))
_add_check("adaptive_records_are_active_only", all(active_record_checks), f"valid_records={sum(active_record_checks)}/{len(active_record_checks)}; no zero-filled inactive F/M rows in the behavioural clustering population.")

# 3) The primary baseline and ablation comparisons must use one cohort size per snapshot.
base_cohort_sizes = common_cohort_results.groupby("snapshot_date")["n_common_customers"].nunique()
_add_check("baseline_common_active_cohort_equal_within_snapshot", bool((base_cohort_sizes <= 1).all()), f"snapshots_checked={len(base_cohort_sizes)}")
abl_cohort_sizes = ablation_common_cohort_results.groupby("snapshot_date")["n_common_customers"].nunique()
_add_check("ablation_common_active_cohort_equal_within_snapshot", bool((abl_cohort_sizes <= 1).all()), f"snapshots_checked={len(abl_cohort_sizes)}")

# 4) Final test period must have all expected adaptive results; all primary test
# comparisons use common-active cohorts (not method-specific native populations).
_add_check("all_test_snapshots_evaluated", len(test_history) == len(test_snapshot_dates) and len(test_adaptive_metrics) == len(test_snapshot_dates), f"test_results={len(test_history)}/{len(test_snapshot_dates)}")
_add_check("test_baseline_common_active_nonempty", not test_baseline_results.empty and test_baseline_results["n_common_customers"].nunique() >= 1, f"rows={len(test_baseline_results)}")
_add_check("test_ablation_common_active_nonempty", not test_ablation_results.empty, f"rows={len(test_ablation_results)}")

# 5) Frozen-centroid model must be fitted only on the train cutoff and preprocessing
# parameters must come from train RFM; test data are transformed, never used to fit scaler.
_add_check("frozen_centroid_cutoff_before_test", bool(train_cutoff < pd.Timestamp(test_snapshot_dates[0]).normalize() + pd.Timedelta(days=1)), f"train_cutoff={train_cutoff}; first_test={pd.Timestamp(test_snapshot_dates[0])}")
_add_check("frozen_centroid_train_scaler_fitted", hasattr(frozen_scaler, "mean_") and len(train_clip_bounds) == 3, "Scaler and clipping bounds were fitted on the final train snapshot only.")
_add_check("frozen_centroid_holdout_nonempty", not frozen_centroid_holdout_df.empty, f"rows={len(frozen_centroid_holdout_df)}")

# 6) Reproducibility: hash all code inputs executed in this fresh sequential run.
input_history = globals().get("In", None)
full_input_hash = None
if isinstance(input_history, list) and len(input_history) - 1 >= 23:
    payload = "\n\n".join(str(x) for x in input_history[-23:])
    full_input_hash = hashlib.sha256(payload.encode("utf-8")).hexdigest()
_add_check("all_23_code_inputs_available_for_fingerprint", full_input_hash is not None, "Run all cells from a fresh runtime to compute the complete code-input fingerprint.")

_add_check("adaptive_policy_weights_sum_to_one", bool(np.isclose(QUALITY_WEIGHT + STABILITY_WEIGHT + COVERAGE_WEIGHT, 1.0)), f"Q={QUALITY_WEIGHT:.3f}, S={STABILITY_WEIGHT:.3f}, C={COVERAGE_WEIGHT:.3f}")
_add_check("validation_policy_selection_recorded", bool(isinstance(SELECTED_ADAPTIVE_POLICY, dict) and SELECTED_ADAPTIVE_POLICY.get("selection_rule")), str(SELECTED_ADAPTIVE_POLICY))
_add_check("validation_policy_sensitivity_complete", bool((adaptive_parameter_sensitivity["snapshots"] == len(validation_snapshot_dates)).any()), f"policies={len(adaptive_parameter_sensitivity)}; validation_snapshots={len(validation_snapshot_dates)}")
_add_check("cluster_business_profiles_available", bool(not cluster_business_profiles_df.empty), f"rows={len(cluster_business_profiles_df)}")
_add_check("duplicate_sensitivity_audit_available", bool(not duplicate_sensitivity_df.empty), f"rows={len(duplicate_sensitivity_df)}")
_add_check("test_only_statistical_results_available", bool(not test_only_statistical_results.empty), f"rows={len(test_only_statistical_results)}; interpret with small test-block count")

integrity_df = pd.DataFrame(integrity_checks)
print("FINAL RESEARCH INTEGRITY AUDIT")
display(integrity_df)
failed_checks = integrity_df.loc[~integrity_df["passed"]]
if not failed_checks.empty:
    raise RuntimeError("Research integrity audit failed:\n" + failed_checks.to_string(index=False))

# Update both manifests with the complete code fingerprint, including this cell.
for manifest_file in [
    Path(OUTPUT_DIR) / "reproducibility_manifest.json",
    Path(RESEARCH_VALIDATION_DIR) / "research_extension_manifest.json"
]:
    if manifest_file.exists():
        manifest = json.loads(manifest_file.read_text(encoding="utf-8"))
        manifest["expected_notebook_code_cells"] = 23
        manifest["all_code_cells_fingerprint_scope"] = "all 23 code cells executed in one fresh sequential Run All session"
        manifest["all_code_cells_sha256"] = full_input_hash
        manifest["final_integrity_audit"] = {
            "passed": True,
            "checks": int(len(integrity_df)),
            "completed_at_utc": datetime.now(timezone.utc).isoformat()
        }
        manifest_file.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

# Rebuild packages after writing the final manifests.
package_targets = [
    Path(OUTPUT_DIR).with_suffix(".zip"),
    Path(OUTPUT_DIR).parent / "AR_DRFM_research_complete_package.zip"
]
for package_target in package_targets:
    package_target.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(package_target, "w", zipfile.ZIP_DEFLATED) as zf:
        for file_path in Path(OUTPUT_DIR).rglob("*"):
            if file_path.is_file():
                zf.write(file_path, arcname=str(Path("adaptive_rolling_dynamic_rfm_results") / file_path.relative_to(OUTPUT_DIR)))
print("Final integrity audit passed. Updated manifests and result packages.")
